<a href="https://colab.research.google.com/github/emisoft-designs/emeraldeo/blob/knowledgebase/Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import warnings

# Suppress DeprecationWarning from jupyter_client related to utcnow()
warnings.filterwarnings( "ignore")

print("Deprecation warning from jupyter_client suppressed.")

Deprecation warning from jupyter_client suppressed.


In [3]:
import getpass
import os


def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Please provide your {var}")

_set_if_undefined("GOOGLE_API_KEY")
_set_if_undefined("TAVILY_API_KEY")
_set_if_undefined("SERPER_API_KEY")

Please provide your GOOGLE_API_KEY··········
Please provide your TAVILY_API_KEY··········
Please provide your SERPER_API_KEY··········


In [ ]:
from pydantic import BaseModel, Field, model_validator
from typing import ClassVar, Dict, Any, List, Optional, Annotated, Sequence, Literal
from langchain_core.documents import Document
from typing_extensions import TypedDict
from datetime import datetime, timezone
from enum import Enum
import uuid
import math
import logging
import operator

# -------------------- Setup Logging --------------------
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)  # Use INFO in production

# -------------------- Enums and Constants --------------------
class CrawlStatus(str, Enum):
    """Crawling status enumeration."""
    NEUTRAL = "neutral"
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    SUCCESS = "success"
    FAILED = "failed"
    PARTIAL = "partial"
    LAST_ERROR = "last_error"

class DocumentSource(str, Enum):
    """Document source types."""
    KNOWLEDGE_BASE = "knowledge_base"
    EXTERNAL_API = "external_api"
    WEB_CRAWL = "web_crawl"
    SEARCH_SNIPPET = "search_snippet"

class ContentSource(str, Enum):
    """Semantic categories assigned by LLM (e.g., academic, blog, forum)."""
    ACADEMIC = "academic"
    GOVERNMENT = "government"
    WIKI = "wikipedia"
    GIT_REPO = "git_repo"
    NEWS = "news"
    BLOG = "blog"
    FORUM = "forum"
    OTHER = "other"

class QueryComplexity(str, Enum):
    """Query complexity levels."""
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class QualityScoreConfig(float, Enum):
    RELEVANCE_BIAS = 0.1
    SOURCE_SCORE_FLOOR = 0.3
    RICHNESS_LOG_DIVISOR = 20
    GAP_PENALTY_FACTOR = 0.05
    COMPLEXITY_WEIGHT = 0.2
    DIVERSITY_WEIGHT = 0.1

# -------------------- Sub-Models --------------------
class SubQuery(BaseModel):
    """Represents a generated sub-query for targeted search."""
    query: str = Field(..., description="The sub-query text.")
    priority: float = Field(0.5, ge=0.0, le=1.0, description="Priority of this sub-query.")
    deep: bool = Field(False, description="Whether to perform a deep search for this query.")
    query_type: Literal["general", "technical", "trends"] = Field("general", description="Type of the sub-query.")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class DocumentResult(BaseModel):
    """Document result with standardized metadata."""
    page_content: str = Field(..., description="The document content")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Document metadata")
    content_length: Optional[int] = Field(default=0, ge=0, description="Content length in characters")
    source_type: DocumentSource = Field(default=DocumentSource.SEARCH_SNIPPET, description="Source type")
    content_source: Optional[ContentSource] = Field(default=ContentSource.OTHER, description="Content source or citation")
    relevance_score: Optional[float] = Field(default=None, ge=0.0, le=1.0, description="Relevance score")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

    @model_validator(mode="after")
    def compute_content_length(self) -> "DocumentResult":
        """Ensure content_length is always computed from page_content if missing/zero."""
        if self.page_content and (not self.content_length or self.content_length == 0):
            object.__setattr__(self, "content_length", len(self.page_content))
        return self

class ExtractedURL(BaseModel):
    """URL with extraction metadata."""
    query: str
    url: str = Field(..., description="The URL")
    domain: Optional[str] = Field(default=None, description="Domain extracted from URL")
    priority: float = Field(0.5, ge=0.0, le=1.0, description="Crawling priority")
    should_crawl: bool = Field(True, description="Whether this URL should be crawled")
    content_source: Optional[ContentSource] = Field(default=ContentSource.OTHER, description="Content source or citation")
    estimated_crawl_time: Optional[float] = Field(default=None, ge=1, description="Estimated crawl time in seconds")
    title: Optional[str] = Field(default=None, description="Page title if available")
    snippet: Optional[str] = Field(default=None, description="Content snippet")
    source_engine: Optional[str] = Field(default=None, description="Search engine that found this URL")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    retry_count: int = Field(default=0, description="Number of times crawling this URL has been retried.")

    @model_validator(mode="after")
    def compute_domain(self) -> "ExtractedURL":
        """Ensure domain is auto-computed from URL if missing."""
        if not self.domain and self.url:
            try:
                from urllib.parse import urlparse
                parsed_url = urlparse(self.url)
                if parsed_url.netloc:
                    object.__setattr__(self, "domain", parsed_url.netloc)
            except Exception:
                pass
        return self

class CrawlStats(BaseModel):
    """Crawling statistics."""
    attempted: int = Field(default=0, ge=0, description="Number of URLs attempted")
    successful: int = Field(default=0, ge=0, description="Number of successful crawls")
    failed: int = Field(default=0, ge=0, description="Number of failed crawls")
    time: float = Field(default=0.0, ge=0.0, description="Total response time in seconds")

class KnowledgeMetadata(BaseModel):
    """Session-level metadata."""
    session_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    last_updated: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    total_iterations: int = Field(default=0, ge=0)
    final: bool = Field(False)
    crawl_stats: CrawlStats = Field(default_factory=CrawlStats)

# -------------------- LangGraph TypedDict --------------------
class LangGraphKnowledgeState(TypedDict):
    """LangGraph state schema, using reducers for list updates."""
    main_query: str
    messages: Annotated[List[str], operator.add]
    sub_queries_to_generate: Annotated[List[SubQuery], operator.add]
    sub_queries_to_process: Annotated[List[SubQuery], operator.add]
    processed_queries: Annotated[List[str], operator.add]
    knowledge_base_results: Annotated[List[DocumentResult], operator.add]
    external_api_results: Annotated[List[DocumentResult], operator.add]
    crawler_results: Annotated[List[DocumentResult], operator.add]
    extracted_urls: Annotated[List[ExtractedURL], operator.add]
    crawl_queue: Annotated[List[ExtractedURL], operator.add]
    cleaned_chunks: Annotated[List[DocumentResult], operator.add]
    gaps: Annotated[List[str], operator.add]
    state_metrics: Dict[str, Any]
    metadata: KnowledgeMetadata
    errors: Annotated[List[str], operator.add]

# -------------------- Main KnowledgeState --------------------
class KnowledgeState(BaseModel):
    """Represents the state of the knowledge acquisition process."""
    main_query: str = Field(..., description="The main query driving the knowledge acquisition.")
    messages: List[Any] = Field(default_factory=list)
    sub_queries_to_generate: List[SubQuery] = Field(default_factory=list)
    sub_queries_to_process: List[SubQuery] = Field(default_factory=list)
    processed_queries: List[str] = Field(default_factory=list)

    knowledge_base_results: List[DocumentResult] = Field(default_factory=list)
    external_api_results: List[DocumentResult] = Field(default_factory=list)
    crawler_results: List[DocumentResult] = Field(default_factory=list)

    extracted_urls: List[ExtractedURL] = Field(default_factory=list)
    crawl_queue: List[ExtractedURL] = Field(default_factory=list)
    crawl_status: CrawlStatus = Field(default=CrawlStatus.NEUTRAL)

    cleaned_chunks: List[DocumentResult] = Field(default_factory=list)
    gaps: List[str] = Field(default_factory=list)
    state_metrics: Dict[str, Any] = Field(default_factory=dict)
    metadata: KnowledgeMetadata = Field(default_factory=KnowledgeMetadata)
    errors: List[str] = Field(default_factory=list)

    _LANGGRAPH_STATE_SCHEMA: ClassVar = LangGraphKnowledgeState

    # -------------------- Validators --------------------
    @model_validator(mode="after")
    def validate_counts(self) -> "KnowledgeState":
        if self.metadata.crawl_stats.attempted < len(self.extracted_urls):
            logger.warning("Crawl stats 'attempted' count is less than extracted URLs.")
        return self

    # -------------------- Properties --------------------
    @property
    def total_documents(self) -> int:
        return len(self.knowledge_base_results) + len(self.external_api_results) + len(self.crawler_results)

    @property
    def success_rate(self) -> float:
        stats = self.metadata.crawl_stats
        return stats.successful / stats.attempted if stats.attempted else 0.0

    @property
    def has_sufficient_content(self) -> bool:
        total_chunks = len(self.knowledge_base_results) + len(self.cleaned_chunks)
        min_required_chunks = self.state_metrics.get("min_required_chunks", 5)
        return total_chunks >= min_required_chunks

    @property
    def should_continue_crawling(self) -> bool:
        return self.metadata.total_iterations < 5 and not self.has_sufficient_content and bool(self.crawl_queue)

    # -------------------- Methods to Update State --------------------
    def add_queries(self, new_sub_queries: List[SubQuery]) -> None:
        self.sub_queries_to_generate.extend(new_sub_queries)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def add_results(self, new_results: List[DocumentResult], source: DocumentSource) -> None:
        if source == DocumentSource.KNOWLEDGE_BASE:
            self.knowledge_base_results.extend(new_results)
        elif source in (DocumentSource.EXTERNAL_API, DocumentSource.SEARCH_SNIPPET):
            self.external_api_results.extend(new_results)
        elif source == DocumentSource.WEB_CRAWL:
            self.crawler_results.extend(new_results)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def add_urls_to_crawl(self, urls: List[ExtractedURL]) -> None:
        self.crawl_queue.extend(urls)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def add_urls(self, new_urls: List[ExtractedURL], add_to_queue: bool = True) -> None:
        self.extracted_urls.extend(new_urls)
        if add_to_queue:
            self.crawl_queue.extend(new_urls)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def mark_sub_queries_processed(self, query_texts: List[str]) -> None:
        self.processed_queries.extend(query_texts)
        self.metadata.last_updated = datetime.now(timezone.utc)

    def add_error(self, error_msg: str) -> None:
        self.errors.append(f"[{datetime.now(timezone.utc).isoformat()}] {error_msg}")
        self.metadata.last_updated = datetime.now(timezone.utc)
        logger.error(f"Error occurred: {error_msg}")

    def update_crawl_stats(self, attempted: int = 0, successful: int = 0, failed: int = 0, time: float = 0.0) -> None:
        self.metadata.crawl_stats.attempted += attempted
        self.metadata.crawl_stats.successful += successful
        self.metadata.crawl_stats.failed += failed
        self.metadata.crawl_stats.total_response_time += time
        self.metadata.last_updated = datetime.now(timezone.utc)

    # -------------------- Quality Score --------------------
    def calculate_weighted_quality_score(self) -> float:
        if not self.total_documents:
            return 0.0

        combined_results = self.crawler_results + self.external_api_results
        total_content_length = sum(res.content_length for res in combined_results)
        richness_score = math.log1p(total_content_length / QualityScoreConfig.RICHNESS_LOG_DIVISOR)

        total_relevance = sum(
            res.relevance_score for res in self.knowledge_base_results + self.external_api_results + self.crawler_results if res.relevance_score is not None
        )
        relevance_score = (total_relevance / self.total_documents) * QualityScoreConfig.RELEVANCE_BIAS

        unique_sources = len(set(res.content_source for res in self.crawler_results + self.external_api_results))
        diversity_score = (unique_sources / len(ContentSource)) * QualityScoreConfig.DIVERSITY_WEIGHT

        complexity_level = self.state_metrics.get("query_complexity", QueryComplexity.MEDIUM)
        complexity_score = (
            1.0 if complexity_level == QueryComplexity.HIGH else 0.5 if complexity_level == QueryComplexity.MEDIUM else 0.1
        ) * QualityScoreConfig.COMPLEXITY_WEIGHT

        gap_penalty = len(self.gaps) * QualityScoreConfig.GAP_PENALTY_FACTOR

        score = (richness_score + relevance_score + diversity_score + complexity_score) - gap_penalty
        return max(0.0, score)

    # -------------------- Export Methods --------------------
    def to_dict(self) -> Dict[str, Any]:
        return self.model_dump()

    def to_summary(self) -> Dict[str, Any]:
        return {
            "main_query": self.main_query,
            "total_documents": self.total_documents,
            "crawl_status": self.metadata.crawl_stats.attempted,
            "success_rate": self.success_rate,
            "quality_score": self.calculate_weighted_quality_score(),
            "final": self.metadata.final,
            "errors": self.errors
        }

# -------------------- Helper Conversion Functions --------------------
def knowledge_state_to_typed_dict(state: KnowledgeState) -> Dict[str, Any]:
    return state.model_dump()

def typed_dict_to_knowledge_state(data: Dict[str, Any]) -> KnowledgeState:
    return KnowledgeState(**data)

def convert_langchain_doc_to_document_result(
    doc: Document,
    source_type: DocumentSource = DocumentSource.SEARCH_SNIPPET,
    relevance_score: Optional[float] = None
) -> DocumentResult:
    """Converts a LangChain Document to a Pydantic DocumentResult."""
    content_source = doc.metadata.get("content_source")
    if isinstance(content_source, str):
        try:
            content_source_enum = ContentSource(content_source)
        except ValueError:
            content_source_enum = ContentSource.OTHER # Default if conversion fails
    else:
        content_source_enum = ContentSource.OTHER # Default if not a string

    return DocumentResult(
        page_content=doc.page_content,
        metadata=doc.metadata,
        source_type=source_type,
        content_source=content_source_enum,
        relevance_score=relevance_score,
        content_length=len(doc.page_content) if doc.page_content else 0
    )

def convert_search_result_to_extracted_url(result: Dict[str, Any], query: str, priority: float = 0.5) -> ExtractedURL:
    return ExtractedURL(
        query=query,
        url=result.get('url', ''),
        title=result.get('title'),
        snippet=result.get('snippet'),
        source_engine=result.get('engine'),
        priority=priority,
        should_crawl=True
    )


In [49]:
# from pydantic import BaseModel, Field, field_validator, model_validator
# from typing import ClassVar, Dict, Any, List, Optional, Annotated, Sequence
# from langchain_core.documents import Document
# from typing_extensions import TypedDict
# from datetime import datetime, timezone
# from enum import Enum
# import uuid
# import math
# import logging
# import operator
# import asyncio # For async example

# # ---- Setup Logging ----
# logger = logging.getLogger(__name__)
# logger.setLevel(logging.INFO) # Use INFO in production


# # -------------------- Enums and Constants --------------------

# class CrawlStatus(str, Enum):
#     """Crawling status enumeration."""
#     NEUTRAL = "neutral"
#     PENDING = "pending"
#     IN_PROGRESS = "in_progress"
#     SUCCESS = "success"
#     FAILED = "failed"
#     PARTIAL = "partial"
#     LAST_ERROR = "last_error"

# class DocumentSource(str, Enum):
#     """Document source types."""
#     KNOWLEDGE_BASE = "knowledge_base"
#     EXTERNAL_API = "external_api"
#     WEB_CRAWL = "web_crawl"
#     SEARCH_SNIPPET = "search_snippet"

# class ContentSource(str, Enum):
#     """Semantic categories assigned by LLM (e.g., academic, blog, forum)."""
#     ACADEMIC = "academic"
#     GOVERNMENT = "government"
#     WIKI = "wikipedia"
#     GIT_REPO = "git_repo"
#     NEWS = "news"
#     BLOG = "blog"
#     FORUM = "forum"
#     OTHER = "other"

# class QueryComplexity(str, Enum):
#     """Query complexity levels."""
#     LOW = "low"
#     MEDIUM = "medium"
#     HIGH = "high"

# class QualityScoreConfig(float, Enum):
#     RELEVANCE_BIAS = 0.1
#     SOURCE_SCORE_FLOOR = 0.3
#     RICHNESS_LOG_DIVISOR = 20
#     GAP_PENALTY_FACTOR = 0.05
#     # New weights for quality scoring
#     COMPLEXITY_WEIGHT = 0.2
#     DIVERSITY_WEIGHT = 0.1


# # -------------------- Sub-Models --------------------

# class SubQuery(BaseModel):
#     """Sub-query model with metadata."""
#     query: str = Field(..., min_length=1, description="The sub-query text")
#     priority: float = Field(0.5, ge=0.0, le=1.0, description="Priority score")
#     deep: bool = Field(False, description="Whether this requires deep search")
#     query_type: str = Field("general", description="Type of query")
#     created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))


# class DocumentResult(BaseModel):
#     """Document result with standardized metadata."""
#     page_content: str = Field(..., description="The document content")
#     metadata: Dict[str, Any] = Field(default_factory=dict, description="Document metadata")
#     content_length: Optional[int] = Field(default=0, ge=0, description="Content length in characters")
#     source_type: DocumentSource = Field(default=DocumentSource.SEARCH_SNIPPET, description="Source type")
#     content_source: Optional[ContentSource] = Field(default=ContentSource.OTHER, description="Content source or citation")
#     relevance_score: Optional[float] = Field(default=None, ge=0.0, le=1.0, description="Relevance score")
#     created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

#     @model_validator(mode="after")
#     def compute_content_length(self) -> "DocumentResult":
#         """Ensure content_length is always computed from page_content if missing/zero."""
#         if self.page_content and (not self.content_length or self.content_length == 0):
#             object.__setattr__(self, "content_length", len(self.page_content))
#         return self


# class ExtractedURL(BaseModel):
#     """URL with extraction metadata."""
#     query: str
#     url: str = Field(..., description="The URL")
#     domain: Optional[str] = Field(default=None, description="Domain extracted from URL")
#     priority: float = Field(0.5, ge=0.0, le=1.0, description="Crawling priority")
#     should_crawl: bool = Field(True, description="Whether this URL should be crawled")
#     content_source: Optional[ContentSource] = Field(default=ContentSource.OTHER, description="Content source or citation")
#     estimated_crawl_time: Optional[float] = Field(default=None, ge=1, description="Estimated crawl time in seconds")
#     title: Optional[str] = Field(default=None, description="Page title if available")
#     snippet: Optional[str] = Field(default=None, description="Content snippet")
#     source_engine: Optional[str] = Field(default=None, description="Search engine that found this URL")
#     created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
#     retry_count: int = Field(default=0, description="Number of times crawling this URL has been retried.") # Added retry_count

#     @model_validator(mode="after")
#     def compute_domain(self) -> "ExtractedURL":
#         """Ensure domain is auto-computed from URL if missing."""
#         if not self.domain and self.url:
#             try:
#                 # Robustly extract domain
#                 from urllib.parse import urlparse
#                 parsed_url = urlparse(self.url)
#                 if parsed_url.netloc:
#                     object.__setattr__(self, "domain", parsed_url.netloc)
#             except Exception:
#                 pass
#         return self

# class CrawlStats(BaseModel):
#     """Crawling statistics."""
#     attempted: int = Field(default=0, ge=0, description="Number of URLs attempted")
#     successful: int = Field(default=0, ge=0, description="Number of successful crawls")
#     failed: int = Field(default=0, ge=0, description="Number of failed crawls")
#     total_response_time: float = Field(0.0, ge=0.0, description="Total response time in seconds")

# class KnowledgeMetadata(BaseModel):
#     """Session-level metadata."""
#     session_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
#     created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
#     last_updated: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
#     total_iterations: int = Field(default=0, ge=0)
#     final: bool = Field(False)
#     crawl_stats: CrawlStats = Field(default_factory=CrawlStats)
# # Consolidated Definitions

# from typing import Dict, Any, List, Optional, Protocol, Union, Literal
# from pydantic import BaseModel, Field, ConfigDict, model_validator
# from datetime import datetime, timezone, timedelta
# from enum import Enum

# # Assuming ExtractedURL, Document, DocumentSource, ContentSource are needed for the state
# from langchain_core.documents import Document # Assuming Document comes from langchain_core
# # Assume ExtractedURL and other Enums are defined elsewhere if not here
# # For now, let's define necessary enums and types if they're not guaranteed to be in the state import

# class DocumentSource(str, Enum):
#     KNOWLEDGE_BASE = "knowledge_base"
#     EXTERNAL_API = "external_api"
#     WEB_CRAWL = "web_crawl"
#     SEARCH_SNIPPET = "search_snippet"

# class ContentSource(str, Enum):
#     ACADEMIC = "academic"
#     GOVERNMENT = "government"
#     WIKI = "wiki"
#     GIT_REPO = "git_repo"
#     OTHER = "other"

# class QueryComplexity(str, Enum):
#     SIMPLE = "simple"
#     MEDIUM = "medium"
#     HIGH = "high"

# class CrawlStatus(str, Enum):
#     NEUTRAL = "neutral"
#     IN_PROGRESS = "in_progress"
#     SUCCESS = "success"
#     PARTIAL = "partial"
#     FAILED = "failed"


# # Assuming a definition for ExtractedURL is needed for the crawl_queue
# # Based on previous code, it looks like this:
# class ExtractedURL(BaseModel):
#     """URL with extraction metadata."""
#     query: str
#     url: str = Field(..., description="The URL")
#     domain: Optional[str] = Field(default=None, description="Domain extracted from URL")
#     priority: float = Field(0.5, ge=0.0, le=1.0, description="Crawling priority")
#     should_crawl: bool = Field(True, description="Whether this URL should be crawled")
#     content_source: Optional[ContentSource] = Field(default=ContentSource.OTHER, description="Content source or citation")
#     estimated_crawl_time: Optional[float] = Field(default=None, ge=1, description="Estimated crawl time in seconds")
#     title: Optional[str] = Field(default=None, description="Page title if available")
#     snippet: Optional[str] = Field(default=None, description="Content snippet")
#     source_engine: Optional[str] = Field(default=None, description="Search engine that found this URL")
#     created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
#     retry_count: int = Field(default=0, description="Number of times crawling this URL has been retried.")

#     @model_validator(mode="after")
#     def compute_domain(self) -> "ExtractedURL":
#         """Ensure domain is auto-computed from URL if missing."""
#         if not self.domain and self.url:
#             try:
#                 # Robustly extract domain
#                 from urllib.parse import urlparse
#                 parsed_url = urlparse(self.url)
#                 if parsed_url.netloc:
#                     object.__setattr__(self, "domain", parsed_url.netloc)
#             except Exception:
#                 pass
#         return self

# # Assuming a definition for SubQuery is needed
# class SubQuery(BaseModel):
#     """Represents a generated sub-query for targeted search."""
#     query: str = Field(..., description="The sub-query text.")
#     priority: float = Field(0.5, ge=0.0, le=1.0, description="Priority of this sub-query.")
#     deep: bool = Field(False, description="Whether to perform a deep search for this query.")
#     query_type: Literal["general", "technical", "trends"] = Field("general", description="Type of the sub-query.")
#     created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

# # Assuming a definition for DocumentResult is needed
# class DocumentResult(BaseModel):
#     """Represents a processed document chunk with metadata."""
#     page_content: str
#     metadata: Dict[str, Any] = Field(default_factory=dict)
#     source_type: DocumentSource
#     content_source: Optional[ContentSource] = None # Explicitly handle Optional
#     relevance_score: Optional[float] = Field(default=None, ge=0.0, le=1.0)
#     content_length: Optional[int] = None # Explicitly handle Optional


# # Helper function needed by nodes.py (e.g., kb_lookup_node)
# def convert_langchain_doc_to_document_result(
#     doc: Document,
#     source_type: DocumentSource,
#     relevance_score: Optional[float] = None
# ) -> DocumentResult:
#     """Converts a LangChain Document to a Pydantic DocumentResult."""
#     content_source = doc.metadata.get("content_source")
#     if isinstance(content_source, str):
#         try:
#             content_source_enum = ContentSource(content_source)
#         except ValueError:
#             content_source_enum = ContentSource.OTHER # Default if conversion fails
#     else:
#         content_source_enum = ContentSource.OTHER # Default if not a string

#     return DocumentResult(
#         page_content=doc.page_content,
#         metadata=doc.metadata,
#         source_type=source_type,
#         content_source=content_source_enum,
#         relevance_score=relevance_score,
#         content_length=len(doc.page_content) if doc.page_content else 0
#     )

# # Assuming Metadata and CrawlStats are needed
# class CrawlStats(BaseModel):
#     attempted: int = 0
#     successful: int = 0
#     failed: int = 0
#     time: float = 0.0

# class Metadata(BaseModel):
#     session_id: str = Field(default_factory=lambda: str(uuid.uuid4())) # Assuming uuid is available or imported
#     created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
#     last_updated: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
#     total_iterations: int = 0
#     final: bool = False
#     crawl_stats: CrawlStats = Field(default_factory=CrawlStats)
#     # Add other metadata fields as needed



# # -------------------- Main LangGraph State Model --------------------

# # Define an internal TypedDict for LangGraph compatibility with Pydantic BaseModel state
# # This works for setting the initial state and using reducers
# class LangGraphKnowledgeState(TypedDict):
#     """LangGraph state schema, using reducers for list updates."""
#     main_query: str
#     messages: Annotated[List[str], operator.add]
#     sub_queries_to_generate: Annotated[List[SubQuery], operator.add]
#     sub_queries_to_process: Annotated[List[SubQuery], operator.add]
#     processed_queries: Annotated[List[str], operator.add]
#     knowledge_base_results: Annotated[List[DocumentResult], operator.add]
#     external_api_results: Annotated[List[DocumentResult], operator.add]
#     crawler_results: Annotated[List[DocumentResult], operator.add]
#     extracted_urls: Annotated[List[ExtractedURL], operator.add]
#     crawl_queue: Annotated[List[ExtractedURL], operator.add]
#     cleaned_chunks: Annotated[List[Document], operator.add]
#     gaps: Annotated[List[str], operator.add]
#     state_metrics: Dict[str, Any]
#     metadata: KnowledgeMetadata
#     errors: Annotated[List[str], operator.add]



# # Main KnowledgeState model
# class KnowledgeState(BaseModel):
#     """
#     Represents the state of the knowledge acquisition process.
#     Designed as a Pydantic model for structure and validation.
#     """
#     messages: List[Any] = Field(default_factory=list) # Using Any for flexibility with message types
#     main_query: str = Field(..., description="The main query driving the knowledge acquisition.")
#     sub_queries_to_generate: List[SubQuery] = Field(default_factory=list)
#     sub_queries_to_process: List[SubQuery] = Field(default_factory=list) # Or a separate list for active?
#     processed_queries: List[str] = Field(default_factory=list) # Keep track of processed query strings

#     knowledge_base_results: List[DocumentResult] = Field(default_factory=list) # Results from KB lookup
#     external_api_results: List[DocumentResult] = Field(default_factory=list) # Results from search engines etc.

#     crawler_results: List[DocumentResult] = Field(default_factory=list) # Results from web crawling
#     crawl_queue: List[ExtractedURL] = Field(default_factory=list) # URLs prioritized for crawling
#     crawl_status: CrawlStatus = Field(default=CrawlStatus.NEUTRAL, description="Current crawling status")

#     extracted_urls: List[ExtractedURL] = Field(default_factory=list) # URLs extracted from search results
#     cleaned_chunks: List[DocumentResult] = Field(default_factory=list) # Final cleaned and chunked documents

#     gaps: List[str] = Field(default_factory=list, description="Identified knowledge gaps.")
#     state_metrics: Dict[str, Any] = Field(default_factory=dict, description="Metrics about the state (e.g., confidence, quality).")
#     metadata: KnowledgeMetadata = Field(default_factory=KnowledgeMetadata)
#     errors: List[str] = Field(default_factory=list, description="List of errors encountered during the process.")

#     _LANGGRAPH_STATE_SCHEMA: ClassVar = LangGraphKnowledgeState

#     # --- Validation ---
#     @model_validator(mode="after")
#     def validate_counts(self) -> "KnowledgeState":
#         # Example validation: ensure crawl stats match reality
#         if self.metadata.crawl_stats.attempted < len(self.extracted_urls):
#             logger.warning("Crawl stats 'attempted' count is less than extracted URLs.")
#         return self


#     property
#     def total_documents(self) -> int:
#         return (
#             len(self.knowledge_base_results) +
#             len(self.external_api_results) +
#             len(self.crawler_results)
#         )

#     # Methods to update state
#     def add_queries(self, new_sub_queries: List[SubQuery]) -> None:
#         self.sub_queries_to_generate.extend(new_sub_queries)
#         self.metadata.last_updated = datetime.now(timezone.utc)

#     def add_results(self, new_results: List[DocumentResult], source: DocumentSource) -> None:
#         # Add results to the appropriate list based on source
#         if source == DocumentSource.KNOWLEDGE_BASE:
#             self.knowledge_base_results.extend(new_results)
#         elif source == DocumentSource.EXTERNAL_API:
#              # Convert external API results (which might be SearchResultSchema initially) to DocumentResult
#              # Assuming external_api_results will hold DocumentResult objects after extraction/ranking
#              self.external_api_results.extend(new_results)
#         elif source == DocumentSource.WEB_CRAWL:
#             self.crawler_results.extend(new_results)
#         elif source == DocumentSource.SEARCH_SNIPPET:
#              # Search snippet results might be added directly as DocumentResult or converted
#              self.external_api_results.extend(new_results) # Or a separate list for snippets? Let's add to external for now
#         # Add results to the cleaned_chunks list as they are processed
#         # No, cleaned_chunks should be populated by the CleanerEngine node, not here.
#         # This method should only add raw/initial results.
#         self.metadata.last_updated = datetime.now(timezone.utc)

#     def add_urls_to_crawl(self, urls: List[ExtractedURL]):
#          self.crawl_queue.extend(urls)
#          self.metadata.last_updated = datetime.now(timezone.utc)

#     def add_error(self, error_message: str):
#          self.errors.append(f"[{datetime.now(timezone.utc).isoformat()}] {error_message}")
#          self.metadata.last_updated = datetime.now(timezone.utc)

#     def add_urls(self, new_urls: List[ExtractedURL], add_to_queue: bool = True) -> None:
#         self.extracted_urls.extend(new_urls)
#         if add_to_queue:
#             self.crawl_queue.extend(new_urls)
#         self.metadata.last_updated = datetime.now(timezone.utc)

#     def update_crawl_stats(self, attempted: int = 0,
#                            successful: int = 0,
#                            failed: int = 0,
#                            time: float = 0.0
#          ) -> None:
#          self.metadata.crawl_stats.attempted += attempted
#          self.metadata.crawl_stats.successful += successful
#          self.metadata.crawl_stats.failed += failed
#          self.metadata.crawl_stats.time += time
#          self.metadata.last_updated = datetime.now(timezone.utc)

#     property
#     def success_rate(self) -> float:
#         stats = self.metadata.crawl_stats
#         if stats.attempted == 0:
#             return 0.0
#         return stats.successful / stats.attempted

#     property
#     def should_continue_crawling(self) -> bool:
#         # Placeholder logic
#         return self.metadata.total_iterations < 5 and \
#                not self.has_sufficient_content and \
#                len(self.crawl_queue) > 0
#     property
#     def has_sufficient_content(self) -> bool:
#          # Define what "sufficient" means, e.g., a minimum number of chunks or relevance score
#          total_chunks = len(self.knowledge_base_results) + len(self.cleaned_chunks)
#          min_required_chunks = self.state_metrics.get("min_required_chunks", 5) # Example threshold
#          return total_chunks >= min_required_chunks

#     property
#     def success_rate(self) -> float:
#          stats = self.metadata.crawl_stats
#          if stats.attempted == 0:
#               return 0.0
#          return stats.successful / stats.attempted

#     def calculate_weighted_quality_score(self) -> float:
#         """Calculates a quality score based on relevance, source type, recency, etc."""
#         if not self.cleaned_chunks:
#             return 0.0

#         total_score = 0.0
#         for doc in self.cleaned_chunks:
#             score = doc.relevance_score or 0.5 # Default relevance
#             # Add points based on source type and content type
#             source_weight = {
#                 DocumentSource.KNOWLEDGE_BASE: 1.5,
#                 DocumentSource.WEB_CRAWL: 1.2,
#                 DocumentSource.SEARCH_SNIPPET: 1.0,
#                 DocumentSource.EXTERNAL_API: 1.0 # Generic external
#             }.get(doc.source_type, 1.0)

#             content_weight = {
#                 ContentSource.ACADEMIC: 1.3,
#                 ContentSource.GOVERNMENT: 1.2,
#                 ContentSource.WIKI: 1.1,
#                 ContentSource.GIT_REPO: 1.0,
#                 ContentSource.OTHER: 1.0
#             }.get(doc.content_source, 1.0) if doc.content_source else 1.0

#             # Incorporate recency (example: more recent = higher score)
#             recency_score = 1.0 # Placeholder - needs actual date comparison logic

#             total_score += score * source_weight * content_weight * recency_score

#         # Normalize the score
#         return total_score / len(self.cleaned_chunks) if self.cleaned_chunks else 0.0

#     def calculate_weighted_quality_score(self) -> float:
#         if not self.total_documents:
#             return 0.0

#         # --- Richness ---
#         combined_results = []
#         combined_results.extend(self.crawler_results)
#         combined_results.extend(self.external_api_results)
#         total_content_length = sum(res.content_length for res in combined_results)
#         richness_score = math.log1p(total_content_length / QualityScoreConfig.RICHNESS_LOG_DIVISOR)

#         # --- Relevance ---
#         total_relevance = sum(res.relevance_score for res in self.knowledge_base_results + self.external_api_results + self.crawler_results if res.relevance_score is not None)
#         average_relevance = total_relevance / self.total_documents if self.total_documents else 0
#         relevance_score = average_relevance * QualityScoreConfig.RELEVANCE_BIAS

#         # --- Source Diversity (NEW) ---
#         unique_sources = len(set(res.content_source for res in self.crawler_results + self.external_api_results))
#         diversity_score = (unique_sources / len(ContentSource)) * QualityScoreConfig.DIVERSITY_WEIGHT

#         # --- Query Complexity (NEW) ---
#         # Assuming complexity is stored somewhere, e.g., in state_metrics
#         complexity_level = self.state_metrics.get("query_complexity", QueryComplexity.MEDIUM)
#         complexity_score = (
#             1.0 if complexity_level == QueryComplexity.HIGH else
#             0.5 if complexity_level == QueryComplexity.MEDIUM else
#             0.1
#         ) * QualityScoreConfig.COMPLEXITY_WEIGHT

#         # --- Coverage (Gaps) ---
#         gap_penalty = len(self.gaps) * QualityScoreConfig.GAP_PENALTY_FACTOR

#         score = (richness_score + relevance_score + diversity_score + complexity_score) - gap_penalty
#         return max(0.0, score)

#     # Factory methods for creating instances (optional but can be useful)
#     classmethod
#     def create_knowledge_state(cls, main_query: str, **kwargs):
#         """Factory method to create a KnowledgeState instance."""
#         return cls(main_query=main_query, **kwargs)


#     classmethod
#     def create_knowledge_state_from_dict(cls, data: Dict[str, Any]):
#         # Ensure nested models are correctly instantiated from dicts
#         if 'metadata' in data and isinstance(data['metadata'], dict):
#              data['metadata'] = Metadata(**data['metadata'])
#         # Need similar checks/conversions for lists of models like sub_queries, results, urls etc.
#         # This can get complex. Pydantic's parse_obj or model_validate should handle this if data structure matches.
#         # A safer way might be to pass the entire dict to the constructor if the structure is consistent.
#         return cls(**data)


#     def mark_sub_queries_processed(self, query_texts: List[str]) -> None:
#         self.processed_queries.extend(query_texts)
#         self.metadata.last_updated = datetime.now(timezone.utc)

#     def add_error(self, error_msg: str) -> None:
#         self.errors.append(error_msg)
#         self.metadata.last_updated = datetime.now(timezone.utc)
#         logger.error(f"Error occurred: {error_msg}")



#     # --- Export Methods ---
#     def to_dict(self) -> Dict[str, Any]:
#         """Exports the full state as a dictionary for LangGraph compatibility."""
#         return self.model_dump()

#     def to_summary(self) -> Dict[str, Any]:
#         """Provides a compact summary of the state."""
#         return {
#             "main_query": self.main_query,
#             "total_documents": self.total_documents,
#             "crawl_status": self.metadata.crawl_stats.attempted,
#             "success_rate": self.success_rate,
#             "quality_score": self.calculate_weighted_quality_score(),
#             "final": self.metadata.final,
#             "errors": self.errors
#         }

#     # --- Factory Functions (Kept for clarity) ---
#     # classmethod
#     # def create_knowledge_state(cls, main_query: str) -> "KnowledgeState":
#     #     return cls(main_query=main_query, messages=[]) # Added messages field

#     # classmethod
#     # def create_knowledge_state_from_dict(cls, data: Dict[str, Any]) -> "KnowledgeState":
#     #     # Ensure messages and main_query are present when loading from dict if they might be missing
#     #     data['messages'] = data.get('messages', [])
#     #     if 'main_query' not in data:
#     #          raise ValueError("main_query is required when creating KnowledgeState from dict")
#     #     return cls.model_validate(data)


# def knowledge_state_to_typed_dict(state: KnowledgeState) -> Dict[str, Any]:
#     """Convert Pydantic model to TypedDict for LangGraph."""
#     return state.model_dump()

# def typed_dict_to_knowledge_state(data: Dict[str, Any]) -> KnowledgeState:
#     """Convert TypedDict to Pydantic model."""
#     return KnowledgeState(**data)

# def convert_langchain_doc_to_document_result(
#     doc: Document,
#     source_type: DocumentSource = DocumentSource.SEARCH_SNIPPET,
#     content_source: ContentSource = ContentSource.OTHER,
#     relevance_score: Optional[float] = None
# ) -> DocumentResult:
#     """Convert LangChain Document to our DocumentResult."""
#     return DocumentResult(
#         page_content=doc.page_content,
#         metadata=doc.metadata,
#         source_type=source_type,
#         content_source=content_source,
#         relevance_score=relevance_score
#     )


# def convert_search_result_to_extracted_url(
    # result: Dict[str, Any],
#     query: str,
#     priority: float = 0.5
# ) -> ExtractedURL:
#     """Convert search result to ExtractedURL."""
#     return ExtractedURL(
#         query=query,
#         url=result.get('url', ''),
#         title=result.get('title'),
#         snippet=result.get('snippet'),
#         source_engine=result.get('engine'),
#         priority=priority,
#         should_crawl=True
#     )

In [50]:
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, Dict, Any, List
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from duckduckgo_search import DDGS
from langchain_community.embeddings import SentenceTransformerEmbeddings # Import SentenceTransformerEmbeddings

load_dotenv()

# Assuming these config classes are defined elsewhere or will be defined:
# from tools import URLAnalysisConfig, CrawlerConfig, VectorStoreConfig
# from KnowledgeState import LLMConfig # LLMConfig is often separate or simple

# --- Define Sub-Config Models used within GlobalConfig ---
# If these are already defined in your tools/other files, you can remove these definitions
# and ensure they are imported correctly. Assuming they are not fully defined yet for demo.

class SearchConfig(BaseModel):
    """Configuration for the SearchEngine."""
    external_max_results: int = Field(default=8, ge=1, description="Maximum search results per query for external searches.")
    subquery_max_search: int = Field(default=3, ge=1, description="Maximum number of subqueries to run in a batch.")
    subquery_results_per_query: int = Field(default=5, ge=1, description="Maximum search results per subquery.")
    # Add other search engine specific configs here if needed (e.g., API keys)
    # Tavily and DDGS clients are complex types, use Any or specific protocol if defined elsewhere
    tavily_client: Optional[Any] = Field(default=None, description="Initialized TavilySearch client.")
    ddgs_client: Optional[Any] = Field(default=None, description="Initialized DuckDuckGoSearch client.") # Use Any for now

    model_config = ConfigDict(extra='forbid', arbitrary_types_allowed=True)

class URLAnalysisConfig(BaseModel):
    """Configuration for the URLAnalyzer."""
    max_urls: int = Field(default=10, ge=1, le=50, description="Maximum number of URLs to return.")
    priority_threshold: float = Field(default=0.3, ge=0.0, le=1.0, description="Minimum priority for a URL to be considered crawlable.")
    enable_ml_scoring: bool = Field(default=True, description="Enable ML-based relevance scoring via LLM.")
    high_value_domains: Dict[str, float] = Field(
        default={
            'wikipedia.org': 0.8, 'arxiv.org': 0.7, 'github.com': 0.6,
            'medium.com': 0.5, 'towardsdatascience.com': 0.5,
            'ieee.org': 0.8, 'acm.org': 0.8, '.edu': 0.8, '.gov': 0.8
        },
        description="Domains with assigned priority scores."
    )
    low_value_patterns: List[str] = Field(
        default=[
            r'facebook\.com', r'twitter\.com', r'linkedin\.com',
            r'instagram\.com', r'pinterest\.com', r'reddit\.com/r/\w+/comments',
            r'youtube\.com/watch', r'tiktok\.com', r'^\s*javascript:'
        ],
        description="Regex patterns for low-value URLs."
    )
    domain_to_content_source: Dict[str, str] = Field( # Using str here assuming ContentSource Enum will be handled
         default={
            'wikipedia.org': "wiki",
            'arxiv.org': "academic",
            'github.com': "git_repo",
            'nature.com': "academic",
            'ieee.org': "academic",
            'acm.org': "academic",
            '.gov': "government",
            '.edu': "academic",
        },
        description="Mapping of domains to ContentSource strings."
    )
    model_config = ConfigDict(extra='forbid') # Ensure no extra fields are allowed


class URLAnalysisConfig(BaseModel):
    """Configuration for the URLAnalyzer."""
    max_urls: int = Field(default=10, ge=1, le=50, description="Maximum number of URLs to return.")
    priority_threshold: float = Field(default=0.3, ge=0.0, le=1.0, description="Minimum priority for a URL to be considered crawlable.")
    enable_ml_scoring: bool = Field(default=True, description="Enable ML-based relevance scoring via LLM.")
    high_value_domains: Dict[str, float] = Field(
        default={
            'wikipedia.org': 0.8, 'arxiv.org': 0.7, 'github.com': 0.6,
            'medium.com': 0.5, 'towardsdatascience.com': 0.5,
            'ieee.org': 0.8, 'acm.org': 0.8, '.edu': 0.8, '.gov': 0.8
        },
        description="Domains with assigned priority scores."
    )
    low_value_patterns: List[str] = Field(
        default=[
            r'facebook\.com', r'twitter\.com', r'linkedin\.com',
            r'instagram\.com', r'pinterest\.com', r'reddit\.com/r/\w+/comments',
            r'youtube\.com/watch', r'tiktok\.com', r'^\s*javascript:'
        ],
        description="Regex patterns for low-value URLs."
    )
    domain_to_content_source: Dict[str, str] = Field( # Using str here assuming ContentSource Enum will be handled
         default={
            'wikipedia.org': "wiki",
            'arxiv.org': "academic",
            'github.com': "git_repo",
            'nature.com': "academic",
            'ieee.org': "academic",
            'acm.org': "academic",
            '.gov': "government",
            '.edu': "academic",
        },
        description="Mapping of domains to ContentSource strings."
    )
    model_config = ConfigDict(extra='forbid') # Ensure no extra fields are allowed

class CrawlerConfig(BaseModel):
    """Configuration for the CrawlerEngine."""
    chunk_size: int = Field(default=1000, ge=100, description="Size of text chunks.")
    chunk_overlap: int = Field(default=200, ge=0, description="Overlap between text chunks.")
    max_concurrent: int = Field(default=5, ge=1, description="Maximum concurrent crawl tasks.")
    timeout: int = Field(default=15, ge=5, description="Timeout for each crawl request in seconds.")
    max_content_length: int = Field(default=150000, ge=10000, description="Max length of crawled content to prevent memory issues.")
    enable_embedding_content_scoring: bool = Field(default=True, description="Enable embedding-based scoring of crawled content.") # Changed to embedding scoring
    embedding_model_name: str = Field("sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use for content scoring.") # Added embedding model config
    embedding_relevance_threshold: float = Field(default=0.4, ge=0.0, le=1.0, description="Relevance score threshold for embedding-scored chunks.") # Changed threshold name
    max_retries: int = Field(default=2, ge=0, description="Maximum number of times to retry a failed URL crawl.") # Added max_retries

class CleanerConfig(BaseModel):
    """Configuration for the CleanerEngine."""
    chunk_size: int = Field(default=1200, ge=200, description="Size of text chunks.")
    chunk_overlap: int = Field(default=160, ge=0, description="Overlap between text chunks.")
    use_semantic_chunking: bool = Field(default=False, description="Use embedding-based semantic chunking.")
    embedding_model_name: str = Field("sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use.")
    model_config = ConfigDict(extra='forbid') # Ensure no extra fields are allowed


class VectorStoreConfig(BaseModel):
    """Configuration for the VectorStoreNode."""
    vector_store_type: str = Field(
        default="chroma",
        description="Type of vector store to use: 'chroma', 'postgres', or 'sqlite'."
    )
    collection_name: str = Field(default="default-collection", description="The name of the vector collection/table.")
    chroma_persist_directory: str = Field(default="./chroma_db", description="Directory for Chroma persistence.")
    # Add embedding model config here, or keep it separate if preferred
    embedding_model_name: str = Field(default="sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use.")
    embedding_model_provider: str = Field(default="huggingface", description="Provider for the embedding model: 'huggingface' or 'sentence-transformer'.")


class LLMConfig(BaseModel):
    """Configuration for Language Models."""
    chat_model_name: str = Field(default="gpt-4o", description="Name of the main chat model.")
    tool_model_name: str = Field(default="gpt-4o", description="Name of the tool-calling model.")
    embedding_model_name: str = Field(default="text-embedding-ada-002", description="Name of the embedding model.") # This one might be redundant if SentenceTransformer is used directly
    temperature: float = Field(default=0.0, ge=0.0, le=2.0, description="Temperature for creative tasks.")
    # Add other LLM-specific configs (e.g., API keys, max tokens)

class DecisionsConfig(BaseModel):
    """Configuration for Graph Decision Logic."""
    confidence_threshold: float = Field(default=0.7, ge=0.0, le=1.0, description="KB confidence threshold to skip external search.")
    low_success_rate_threshold: float = Field(default=0.3, ge=0.0, le=1.0, description="Crawl success rate below which crawling is stopped.")
    quality_threshold: float = Field(default=0.6, ge=0.0, le=1.0, description="Content quality score threshold to stop searching.")
    max_iterations: int = Field(default=5, ge=1, description="Maximum number of graph iterations.")
    min_results: int = Field(default=2, ge=0, description="Minimum number of results needed for quality assessment.")
    max_final_documents: int = Field(default=20, ge=1, description="Maximum number of documents in the final result set.")


class GlobalConfig(BaseModel):
    """
    Comprehensive configuration for the Knowledge Acquisition workflow.
    """
    search: "SearchConfig" = Field(default_factory=lambda: SearchConfig())
    url_analyzer: "URLAnalysisConfig" = Field(default_factory=lambda: URLAnalysisConfig())
    crawler: "CrawlerConfig" = Field(default_factory=lambda: CrawlerConfig())
    vector_store: "VectorStoreConfig" = Field(default_factory=lambda: VectorStoreConfig())
    llm: LLMConfig = Field(default_factory=LLMConfig)
    decisions: "DecisionsConfig" = Field(default_factory=lambda: DecisionsConfig())

    model_config = ConfigDict(extra="forbid")

# --- Global LLM Registry ---
class LLMRegistry:
    _models: Dict[str, Any] = {}

    @classmethod
    def init_from_config(cls, config: LLMConfig):
        if "chat" not in cls._models:
            cls._models["chat"] = init_chat_model(
                config.chat_model_name,
                temperature=config.temperature,
            )
        if "tool" not in cls._models:
            cls._models["tool"] = init_chat_model(
                config.tool_model_name,
                temperature=config.temperature,
            )
        return cls._models

    @classmethod
    def get(cls, role: str = "chat"):
        if role not in cls._models:
            raise ValueError(f"LLM '{role}' not initialized. Call init_from_config first.")
        return cls._models[role]
# Example usage:
# global_config = GlobalConfig()
# print(global_config.model_dump_json(indent=2))

Here's a test script to demonstrate the functionality of the `KnowledgeState` model.

In [51]:
# Create an initial KnowledgeState instance
initial_state = KnowledgeState.create_knowledge_state(main_query="What are the latest advancements in AI?")
print("Initial State:")
print(initial_state.to_summary())

# Add some sub-queries
sub_queries = [
    SubQuery(query="Recent breakthroughs in natural language processing"),
    SubQuery(query="New developments in computer vision", priority=0.8),
    SubQuery(query="Ethical considerations in AI", priority=0.6), # Add another query
]
initial_state.add_queries(sub_queries)
print("\nState after adding sub-queries:")
print(initial_state.sub_queries_to_generate)

# Add some document results including edge cases for relevance and length
kb_results = [
    DocumentResult(page_content="Content about NLP breakthrough 1", source_type=DocumentSource.KNOWLEDGE_BASE, relevance_score=0.9),
    DocumentResult(page_content="Content about CV development 1", source_type=DocumentSource.KNOWLEDGE_BASE, relevance_score=0.7),
    DocumentResult(page_content="Short but highly relevant fact.", source_type=DocumentSource.KNOWLEDGE_BASE, relevance_score=1.0), # Short, high relevance
]
web_results = [
    DocumentResult(page_content="Blog post on AI trends" * 50, source_type=DocumentSource.WEB_CRAWL, content_source=ContentSource.BLOG, relevance_score=0.1), # Long, low relevance
    DocumentResult(page_content="Academic paper on new algorithm" * 10, source_type=DocumentSource.WEB_CRAWL, content_source=ContentSource.ACADEMIC, relevance_score=0.95),
    DocumentResult(page_content="Irrelevant content.", source_type=DocumentSource.WEB_CRAWL, content_source=ContentSource.FORUM, relevance_score=0.0), # Short, zero relevance
    DocumentResult(page_content="Very long and irrelevant document content. " * 200, source_type=DocumentSource.WEB_CRAWL, content_source=ContentSource.OTHER, relevance_score=0.05), # Very long, very low relevance
]
initial_state.add_results(kb_results, source=DocumentSource.KNOWLEDGE_BASE)
initial_state.add_results(web_results, source=DocumentSource.WEB_CRAWL)
print("\nState after adding results:")
print(f"Total documents: {initial_state.total_documents}")
print(f"Knowledge base results: {len(initial_state.knowledge_base_results)}")
print(f"Crawler results: {len(initial_state.crawler_results)}")


# Add some extracted URLs
extracted_urls = [
    ExtractedURL(query=initial_state.main_query, url="https://example.com/blog/ai-trends", content_source=ContentSource.BLOG),
    ExtractedURL(query=initial_state.main_query, url="https://anothersite.org/paper.pdf", content_source=ContentSource.ACADEMIC, should_crawl=False),
    ExtractedURL(query=initial_state.main_query, url="https://yetanothersite.com/forum/irrelevant", content_source=ContentSource.FORUM), # Add another URL
]
initial_state.add_urls(extracted_urls)
print("\nState after adding URLs:")
print(f"Extracted URLs: {len(initial_state.extracted_urls)}")
print(f"Crawl Queue: {len(initial_state.crawl_queue)}")

# Update crawl stats - reflecting more attempts
initial_state.update_crawl_stats(attempted=3, successful=2, failed=1, time=7.8)
print("\nState after updating crawl stats:")
print(f"Crawl Stats: {initial_state.metadata.crawl_stats}")
print(f"Success Rate: {initial_state.success_rate:.2f}")

# Mark sub-queries as processed
initial_state.mark_sub_queries_processed([sub_queries[0].query, sub_queries[1].query]) # Mark more queries processed
print("\nState after marking queries processed:")
print(f"Processed Queries: {initial_state.processed_queries}")
print(f"Sub-queries to generate: {len(initial_state.sub_queries_to_generate)}")

# Add a gap and an error
initial_state.gaps.append("Missing information on ethical considerations")
initial_state.add_error("Failed to parse a document from irrelevant source")
initial_state.add_error("Timeout during crawl attempt") # Add another error
print("\nState after adding gap and error:")
print(f"Gaps: {initial_state.gaps}")
print(f"Errors: {initial_state.errors}")

# Check computed properties
print("\nComputed Properties:")
# To better test has_sufficient_content, let's add some 'cleaned_chunks'
# In a real scenario, these would come from processing DocumentResults
initial_state.cleaned_chunks.append(Document(page_content="Cleaned chunk 1 " * 50))
initial_state.cleaned_chunks.append(Document(page_content="Cleaned chunk 2 " * 60))
print(f"Has sufficient content (placeholder): {initial_state.has_sufficient_content}")
initial_state.metadata.total_iterations = 3 # Simulate more iterations
print(f"Should continue crawling (placeholder): {initial_state.should_continue_crawling}")

# Calculate quality score (demonstration)
# Manually set complexity for demonstration
initial_state.state_metrics["query_complexity"] = QueryComplexity.HIGH
quality_score = initial_state.calculate_weighted_quality_score()
print(f"\nCalculated Quality Score: {quality_score:.2f}")

# Export to dict
state_dict = initial_state.to_dict()
# print("\nState exported to dictionary:")
# print(state_dict)

# Create a new state from the dictionary (demonstration of loading)
loaded_state = KnowledgeState.create_knowledge_state_from_dict(state_dict)
print("\nState loaded from dictionary:")
print(loaded_state.to_summary())

ERROR:__main__:Error occurred: Failed to parse a document from irrelevant source
ERROR:__main__:Error occurred: Timeout during crawl attempt


Initial State:
{'main_query': 'What are the latest advancements in AI?', 'total_documents': 0, 'crawl_status': 0, 'success_rate': 0.0, 'quality_score': 0.0, 'final': False, 'errors': []}

State after adding sub-queries:
[SubQuery(query='Recent breakthroughs in natural language processing', priority=0.5, deep=False, query_type='general', created_at=datetime.datetime(2025, 9, 7, 5, 43, 21, 100937, tzinfo=datetime.timezone.utc)), SubQuery(query='New developments in computer vision', priority=0.8, deep=False, query_type='general', created_at=datetime.datetime(2025, 9, 7, 5, 43, 21, 100950, tzinfo=datetime.timezone.utc)), SubQuery(query='Ethical considerations in AI', priority=0.6, deep=False, query_type='general', created_at=datetime.datetime(2025, 9, 7, 5, 43, 21, 101134, tzinfo=datetime.timezone.utc))]

State after adding results:
Total documents: 7
Knowledge base results: 3
Crawler results: 4

State after adding URLs:
Extracted URLs: 3
Crawl Queue: 3

State after updating crawl stats:
C

In [43]:
from pydantic import BaseModel, Field, field_validator, ConfigDict
from typing import ClassVar, Dict, Any, List, Optional, Annotated, Sequence, Protocol
from typing_extensions import TypedDict
from datetime import datetime, timezone
from langchain_core.documents import Document
from enum import Enum
import uuid
import math
import logging
import operator
import asyncio
import re
from urllib.parse import urlparse
from abc import ABC, abstractmethod

# -------------------- Pydantic models for configuration --------------------

class URLAnalysisConfig(BaseModel):
    """Configuration for the URLAnalyzer."""
    max_urls: int = Field(default=10, ge=1, le=50, description="Maximum number of URLs to return.")
    priority_threshold: float = Field(default=0.3, ge=0.0, le=1.0, description="Minimum priority for a URL to be considered crawlable.")
    enable_ml_scoring: bool = Field(default=True, description="Enable ML-based relevance scoring via LLM.")
    high_value_domains: Dict[str, float] = Field(
        default={
            'wikipedia.org': 0.8, 'arxiv.org': 0.7, 'github.com': 0.6,
            'medium.com': 0.5, 'towardsdatascience.com': 0.5,
            'ieee.org': 0.8, 'acm.org': 0.8, '.edu': 0.8, '.gov': 0.8
        },
        description="Domains with assigned priority scores."
    )
    low_value_patterns: List[str] = Field(
        default=[
            r'facebook\.com', r'twitter\.com', r'linkedin\.com',
            r'instagram\.com', r'pinterest\.com', r'reddit\.com/r/\w+/comments',
            r'youtube\.com/watch', r'tiktok\.com', r'^\s*javascript:'
        ],
        description="Regex patterns for low-value URLs."
    )
    domain_to_content_source: Dict[str, ContentSource] = Field(
        default={
            'wikipedia.org': ContentSource.WIKI,
            'arxiv.org': ContentSource.ACADEMIC,
            'github.com': ContentSource.GIT_REPO,
            'nature.com': ContentSource.ACADEMIC,
            'ieee.org': ContentSource.ACADEMIC,
            'acm.org': ContentSource.ACADEMIC,
            '.gov': ContentSource.GOVERNMENT,
            '.edu': ContentSource.ACADEMIC,
        },
        description="Mapping of domains to ContentSource enums."
    )
    model_config = ConfigDict(extra='forbid')


# -------------------- Async URL Analyzer Protocol --------------------

class AsyncURLAnalyzerProtocol(Protocol):
    """Protocol for an asynchronous URL analyzer."""
    async def analyze_batch(self, search_results: List[Dict[str, Any]], query: str) -> None:
        """Analyze a batch of search results asynchronously and update state."""
        ...


# -------------------- Enhanced URL Analysis with Pydantic Integration --------------------

class URLAnalyzer(AsyncURLAnalyzerProtocol):
    """Enhanced, state-aware, async URL analyzer that works with Pydantic models."""

    def __init__(self, state: "KnowledgeState", config: URLAnalysisConfig, llm: Optional[Any] = None):
        self.state = state
        self.config = config
        self.llm = llm

    async def analyze_batch(self, search_results: List[Dict[str, Any]], query: str) -> None:
        """Analyze a batch of search results asynchronously and update state."""
        tasks = [self._analyze_single_url(result, query) for result in search_results]
        analyzed_urls = await asyncio.gather(*tasks, return_exceptions=True)

        extracted_urls = []
        for res in analyzed_urls:
            if isinstance(res, Exception):
                self.state.add_error(f"URL analysis error: {str(res)}")
            elif res:
                extracted_urls.append(res)

        # Filter, sort by priority, and add to state
        crawlable_urls = [url for url in extracted_urls if url.should_crawl]
        crawlable_urls.sort(key=lambda x: x.priority, reverse=True)

        final_urls = crawlable_urls[:self.config.max_urls]
        self.state.add_urls(final_urls, add_to_queue=True)

    async def _analyze_single_url(self, result: Dict[str, Any], query: str) -> Optional[ExtractedURL]:
        """Analyze a single URL asynchronously."""
        url = result.get('url', '')
        snippet = result.get('snippet', '')

        if not url:
            return None

        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            path = parsed.path.lower()

            # Use dynamic scoring
            domain_score = await self._calculate_domain_score(domain)
            relevance_score = await self._calculate_relevance_score(url, query, snippet)
            content_type_score = self._calculate_content_type_score(path)

            # Overall priority score (0-1)
            priority = (domain_score * 0.4 + relevance_score * 0.4 + content_type_score * 0.2)
            should_crawl = priority > self.config.priority_threshold and not self._is_low_value_url(url)

            # Determine content source from config
            content_source = self._determine_content_source(domain)

            return ExtractedURL(
                url=url,
                priority=priority,
                should_crawl=should_crawl,
                snippet=snippet,
                title=result.get('title'),
                source_engine=result.get('engine'),
                content_source=content_source,
                # Using a dummy value for now, could be dynamic
                estimated_crawl_time=5.0
            )

        except Exception as e:
            raise Exception(f"Failed to analyze URL {url}: {e}")

    async def _calculate_domain_score(self, domain: str) -> float:
        """Calculate domain authority score using dynamic config."""
        score = 0.3 # Base score
        for hv_domain, hv_score in self.config.high_value_domains.items():
            if hv_domain in domain:
                score += hv_score
                break
        return min(1.0, score)

    async def _calculate_relevance_score(self, url: str, query: str, snippet: str) -> float:
        """Calculate relevance score using either LLM or traditional method."""
        if self.config.enable_ml_scoring and self.llm:
            return await self._calculate_ml_relevance_score_llm(url, query, snippet)
        else:
            return self._calculate_traditional_relevance(url, query, snippet)

    def _calculate_traditional_relevance(self, url: str, query: str, snippet: str) -> float:
        """Fallback for traditional relevance scoring."""
        query_terms = query.lower().split()
        url_lower = url.lower()
        snippet_lower = snippet.lower()

        url_matches = sum(1 for term in query_terms if term in url_lower)
        snippet_matches = sum(1 for term in query_terms if term in snippet_lower)

        url_score = min(1.0, url_matches / len(query_terms)) if query_terms else 0
        snippet_score = min(1.0, snippet_matches / len(query_terms)) if snippet and query_terms else 0

        return (url_score * 0.3 + snippet_score * 0.7)

    async def _calculate_ml_relevance_score_llm(self, url: str, query: str, snippet: str) -> float:
        """Use LLM for semantic relevance scoring."""
        prompt = f"""
        Rate the relevance of this URL to the query on a scale of 0.0 to 1.0:
        Query: {query}
        URL: {url}
        Snippet: {snippet[:200]}

        Consider: semantic relevance, content authority, freshness
        Respond with just a number between 0.0 and 1.0
        """
        try:
            response = await self.llm.ainvoke(prompt)
            # Add robustness for non-numeric LLM output
            score = float(re.search(r"(\d+(\.\d+)?)", response.content).group(1))
            return max(0.0, min(1.0, score))
        except Exception as e:
            self.state.add_error(f"ML scoring failed for {url}: {str(e)}")
            return self._calculate_traditional_relevance(url, query, snippet)

    def _calculate_content_type_score(self, path: str) -> float:
        """Score based on URL path content type indicators."""
        high_value_patterns = [
            r'/article/', r'/blog/', r'/post/', r'/news/', r'/research/',
            r'/paper/', r'/guide/', r'/tutorial/', r'/doc/', r'/wiki/'
        ]

        low_value_patterns = [
            r'/login', r'/register', r'/cart', r'/checkout', r'/contact',
            r'/about', r'/privacy', r'/terms'
        ]

        for pattern in high_value_patterns:
            if re.search(pattern, path):
                return 0.8

        for pattern in low_value_patterns:
            if re.search(pattern, path):
                return 0.1

        return 0.5

    def _determine_content_source(self, domain: str) -> ContentSource:
        """Determine content source based on domain using dynamic config."""
        for domain_pattern, content_source in self.config.domain_to_content_source.items():
            if domain_pattern in domain:
                return content_source
        return ContentSource.OTHER

    def _is_low_value_url(self, url: str) -> bool:
        """Check if URL matches low-value patterns."""
        return any(re.search(pattern, url) for pattern in self.config.low_value_patterns)


Here is a test script to demonstrate the functionality of the `URLAnalyzer` and `URLAnalysisConfig`.

In [44]:
import asyncio
import nest_asyncio
from unittest.mock import AsyncMock, MagicMock

nest_asyncio.apply()

# Assume KnowledgeState, URLAnalysisConfig, URLAnalyzer, etc., are defined above

# 1. Create a mock KnowledgeState for the URLAnalyzer to update
mock_state = MagicMock(spec=KnowledgeState)
mock_state.add_error = MagicMock()
mock_state.add_urls = MagicMock()

# 2. Create a URLAnalysisConfig instance
config = URLAnalysisConfig(
    max_urls=5,
    priority_threshold=0.4,
    enable_ml_scoring=True,
    high_value_domains={'example.com': 0.9, 'testsite.org': 0.7},
    low_value_patterns=[r'forum\.invalid']
)
print("URL Analysis Config:")
print(config.model_dump_json(indent=2))

# 3. Create a mock LLM with an async `ainvoke` method for ML scoring
mock_llm = AsyncMock()
# Configure the mock LLM's ainvoke method to return a mock response with content
mock_llm.ainvoke.return_value = MagicMock(content="0.85") # Simulate LLM returning a score

# 4. Create a URLAnalyzer instance
url_analyzer = URLAnalyzer(state=mock_state, config=config, llm=mock_llm)
print("\nURL Analyzer created.")

# 5. Prepare a list of simulated search results
search_results = [
    {'url': 'https://example.com/relevant-article-1', 'title': 'Relevant Article 1', 'snippet': 'This is a highly relevant snippet.', 'engine': 'google'}, # High domain, relevant snippet
    {'url': 'https://testsite.org/less-relevant-page', 'title': 'Less Relevant Page', 'snippet': 'Some content here.', 'engine': 'bing'}, # Medium domain, less relevant
    {'url': 'https://another.com/irrelevant-stuff', 'title': 'Irrelevant', 'snippet': 'Completely unrelated content.', 'engine': 'google'}, # Low domain, irrelevant
    {'url': 'https://forum.invalid/thread/123', 'title': 'Forum Discussion', 'snippet': 'A discussion on a forum.', 'engine': 'duckduckgo'}, # Low value pattern
    {'url': 'https://example.com/another/highly/relevant/doc', 'title': 'Relevant Doc', 'snippet': 'More relevant information.', 'engine': 'google'}, # High domain, relevant via path
    {'url': 'https://lowprior.com/page', 'title': 'Low Priority Page', 'snippet': 'Standard page content.', 'engine': 'bing'}, # No special domain/path, average snippet
    {'url': 'https://yetanother.com/article', 'title': 'Yet Another Article', 'snippet': 'Content that might be relevant.', 'engine': 'google'}, # Average relevance
    {'url': 'https://example.com/login', 'title': 'Login Page', 'snippet': 'Please log in.', 'engine': 'google'}, # High domain, but low value path
]

print("\nAnalyzing batch of search results...")
main_query = "Latest advancements in AI"

# 6. Run the async analyze_batch method
asyncio.run(url_analyzer.analyze_batch(search_results, main_query))

print("\nAnalysis complete.")

# 7. Check how the mock state methods were called
print("\nMock State Calls:")
print(f"add_error called: {mock_state.add_error.call_count} times")
print(f"add_urls called: {mock_state.add_urls.call_count} times")

if mock_state.add_urls.call_count > 0:
    # Print the arguments passed to the last add_urls call
    print("\nURLs added to state (via add_urls call):")
    # Assuming the last call is the one from analyze_batch
    added_urls_list = mock_state.add_urls.call_args[0][0]
    print(f"Number of URLs added: {len(added_urls_list)}")
    for url_obj in added_urls_list:
        print(f"- URL: {url_obj.url}, Priority: {url_obj.priority:.2f}, Should Crawl: {url_obj.should_crawl}, Source: {url_obj.content_source}")

if mock_llm.ainvoke.call_count > 0:
    print(f"\nMock LLM ainvoke called: {mock_llm.ainvoke.call_count} times (should match relevant URLs)")
    # You could inspect call_args_list here if needed to see prompts sent to LLM

URL Analysis Config:
{
  "max_urls": 5,
  "priority_threshold": 0.4,
  "enable_ml_scoring": true,
  "high_value_domains": {
    "example.com": 0.9,
    "testsite.org": 0.7
  },
  "low_value_patterns": [
    "forum\\.invalid"
  ],
  "domain_to_content_source": {
    "wikipedia.org": "wikipedia",
    "arxiv.org": "academic",
    "github.com": "git_repo",
    "nature.com": "academic",
    "ieee.org": "academic",
    "acm.org": "academic",
    ".gov": "government",
    ".edu": "academic"
  }
}

URL Analyzer created.

Analyzing batch of search results...

Analysis complete.

Mock State Calls:
add_error called: 8 times
add_urls called: 1 times

URLs added to state (via add_urls call):
Number of URLs added: 0

Mock LLM ainvoke called: 8 times (should match relevant URLs)


In [45]:
warnings.filterwarnings( "ignore")

!pip install langchain_tavily langchain_core ddg duckduckgo_search htmldate google-search-results --quiet
!pip install -U ddgs --quiet

from htmldate import find_date
from datetime import datetime, timedelta
from pydantic import BaseModel, Field
from typing import ClassVar, Dict, Any, List, Optional, AsyncGenerator
from langchain_core.documents import Document
from langchain_tavily import TavilySearch
# from duckduckgo_search import DDGS  # Assuming DDGS is the modern async-compatible way
from langchain_community.tools import DuckDuckGoSearchResults # Import the LangChain DDG tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper # Import the DDG wrapper
from langchain_community.utilities import GoogleSerperAPIWrapper # Import Google Serper
# from KnowledgeState import KnowledgeState, DocumentSource # Assuming these are available
# from config import SearchConfig # Assuming SearchConfig is available
import asyncio
import logging
import re

logger = logging.getLogger(__name__)

# -------------------- Pydantic Model for Search Results --------------------

class SearchResultSchema(BaseModel):
    """Schema for a single search result."""
    url: str = Field(..., description="The URL of the search result.")
    snippet: str = Field(..., description="A snippet of the content.")
    title: str = Field(..., description="The title of the document.")
    engine: str = Field(..., description="The search engine used.")
    query: str = Field(..., description="The query used to find this result.")
    metadata: Dict[str, Any] = Field(default_factory=dict, description="Additional metadata.")


# -------------------- Enhanced Search Engine with Async Integration --------------------
class SearchEngine:
    """Enhanced search class for async, state-aware operations."""

    def __init__(self, state: KnowledgeState, config: SearchConfig):
        self.state = state
        self.config = config
        # Access clients from config
        self._tavily = config.tavily_client
        # self._ddgs = config.ddgs_client
        # Use the LangChain DuckDuckGo tool
        self._ddg_tool = DuckDuckGoSearchResults(
            api_wrapper=DuckDuckGoSearchAPIWrapper(region="wt-wt", time="y", max_results=config.external_max_results),
            source="text" # Use text source for general search
        )
        # Initialize Google Serper
        try:
            self._serper = GoogleSerperAPIWrapper()
        except Exception as e:
            logger.warning(f"Failed to initialize Google Serper: {e}")
            self._serper = None


        self.engines = {}
        if self._tavily:
            self.engines["tavily"] = self._tavily_search
        # if self._ddgs:
        #     self.engines["ddg"] = self._ddg_search
        if self._ddg_tool:
             self.engines["ddg_langchain"] = self._ddg_langchain_search
        if self._serper:
            self.engines["serper"] = self._serper_search


    async def run_searches(self, queries: List[str], max_results_per_query: int = 10) -> None:
        """
        Runs multiple searches concurrently across all available engines, prioritizing recent data.
        Updates the state with search results.
        """
        if not queries:
            return

        all_tasks = []
        for query in queries:
            if self._tavily:
                all_tasks.append(self._tavily_search(query=query, max_results=self.config.external_max_results, time_range="year"))
            # if self._ddgs:
            #      # Pass only expected arguments to DDG search
            #     all_tasks.append(self._ddg_search(query=query, max_results=self.config.external_max_results))
            if self._ddg_tool:
                 all_tasks.append(self._ddg_langchain_search(query=query, max_results=self.config.external_max_results))
            if self._serper:
                # Google Serper doesn't have a direct time_range param in the tool,
                # but the API wrapper might support 'tbs' parameter indirectly.
                # For now, we'll use the default search.
                all_tasks.append(self._serper_search(query=query, max_results=self.config.external_max_results))


        # Run all search tasks concurrently
        raw_results_list = await asyncio.gather(*all_tasks, return_exceptions=True)

        all_typed_results = []
        for raw_results in raw_results_list:
            if isinstance(raw_results, Exception):
                self.state.add_error(f"Search task failed: {str(raw_results)}")
                continue
            all_typed_results.extend(raw_results)

        # Convert SearchResultSchema to DocumentResult, ensuring URL and publication_date are included in metadata
        document_results = [
            DocumentResult(
                page_content=res.snippet,
                metadata={
                    **res.metadata, # Include existing metadata
                    "title": res.title,
                    "engine": res.engine,
                    "query": res.query,
                    "url": res.url, # Explicitly include URL
                    "publication_date": res.metadata.get("publication_date") # Explicitly include publication_date
                },
                source_type=DocumentSource.SEARCH_SNIPPET,
            )
            for res in all_typed_results
        ]

        self.state.add_results(document_results, source=DocumentSource.SEARCH_SNIPPET)


    async def _tavily_search(self, query: str, max_results: int, time_range: str = "year") -> List[SearchResultSchema]:
        """Tavily search implementation using async methods with a time filter."""
        try:
            # Tavily's `time_range` options are 'day', 'week', 'month', or 'year'.
            # We use 'year' to cover the last 3 years as requested.
            results = await self._tavily.ainvoke({"query": query, "max_results": max_results, "time_range": time_range})

            search_results = []
            if results and isinstance(results, list):
                for r in results:
                    url = r.get("url") or r.get("link")
                    snippet = r.get("snippet") or r.get("body")
                    title = r.get("title", "")

                    if url and snippet:
                        metadata = {
                            "raw_score": r.get("score", 0.0),
                            "publication_date": r.get("publication_time") # Note: You'll need to parse this if you want a datetime object
                        }
                        search_results.append(SearchResultSchema(
                            url=url,
                            snippet=snippet,
                            title=title,
                            engine="tavily",
                            query=query,
                            metadata=metadata
                        ))
            return search_results
        except Exception as e:
            logger.error(f"Tavily search error for query '{query}': {e}")
            raise e

    # def _add_date_filters(self, query: str, years: int) -> str:
    #     """
    #     Adds Google-style date range filters to a query for DDG.

    #     Args:
    #         query (str): The original search query.
    #         years (int): The number of years to search back.

    #     Returns:
    #         str: The modified query string with date filters.
    #     """
    #     end_date = datetime.now().date()
    #     start_date = end_date - timedelta(days=365 * years)
    #     start_str = start_date.strftime("%Y-%m-%d")
    #     end_str = end_date.strftime("%Y-%m-%d")
    #     return f'{query} after:{start_str} before:{end_str}'

    # async def _ddg_search(self, query: str, max_results: int, years_back: int = 3) -> List[SearchResultSchema]:
    #     """
    #     DuckDuckGo search using the DDGS async client with a date filter.
    #     """
    #     try:
    #         # Modify the query to filter by date range
    #         filtered_query = self._add_date_filters(query, years=years_back)

    #         # Use asyncio.to_thread for DDGS, which is not natively async
    #         results = await asyncio.to_thread(self._ddgs.text, filtered_query, max_results=max_results)

    #         search_results = []
    #         if results:
    #             for r in results:
    #                 url = r.get("href") or r.get("url")
    #                 content = r.get("body") or r.get("snippet")
    #                 title = r.get("title", "")

    #                 if url and content:
    #                     metadata = {
    #                          "publication_date": r.get("date") # Assuming DDGS returns a date field
    #                     }
    #                     search_results.append(SearchResultSchema(
    #                         url=url,
    #                         snippet=content,
    #                         title=title,
    #                         engine="ddg",
    #                         query=query,
    #                         metadata=metadata
    #                     ))
    #             return search_results
    #         except Exception as e:
    #             logger.error(f"DuckDuckGo search error for query '{query}': {e}")
    #             raise e

    async def _ddg_langchain_search(self, query: str, max_results: int) -> List[SearchResultSchema]:
        """DuckDuckGo search using LangChain's DuckDuckGoSearchResults tool."""
        try:
            # The tool's result is a string, need to parse it.
            # The time filter is set in the wrapper during initialization.
            result_string = await self._ddg_tool.ainvoke(query)

            # Parse the string result from the tool
            # The default format is typically a list of dicts represented as a string
            import json
            # Attempt to find and parse the JSON-like part of the string
            # This regex looks for a list of dictionaries
            match = re.search(r"\[.*?\]", result_string, re.DOTALL)
            if match:
                # Clean up the string to be valid JSON if necessary (e.g., fix single quotes)
                json_string = match.group(0).replace("'", '"') # Simple fix for single quotes
                # More robust parsing might be needed depending on tool output
                results = json.loads(json_string)
            else:
                 logger.warning(f"Could not parse DuckDuckGo search result string: {result_string}")
                 return []


            search_results = []
            if results:
                for r in results:
                    # Assuming the structure is list of dicts with 'link', 'title', 'snippet'
                    url = r.get("link") or r.get("url")
                    snippet = r.get("snippet") or r.get("body")
                    title = r.get("title", "")

                    if url and snippet:
                        # LangChain's DDG tool might not provide publication date directly in metadata
                        # We can try to extract it from the snippet or title later in the pipeline (e.g., in CrawlerEngine)
                        metadata = {
                            # No raw score from this tool by default
                            "publication_date": None # Placeholder - date extraction happens later
                        }
                        search_results.append(SearchResultSchema(
                            url=url,
                            snippet=snippet,
                            title=title,
                            engine="ddg_langchain",
                            query=query,
                            metadata=metadata
                        ))
            return search_results
        except Exception as e:
            logger.error(f"LangChain DuckDuckGo search error for query '{query}': {e}")
            raise e

    async def _serper_search(self, query: str, max_results: int) -> List[SearchResultSchema]:
        """Google Serper search implementation."""
        try:
            # Serper API wrapper doesn't directly support max_results in invoke,
            # it's configured during initialization or via specific parameters.
            # We'll use the default for now or explore adding a custom run method.
            # For this example, we'll call the sync method in a thread.
            results = await asyncio.to_thread(self._serper.results, query)

            search_results = []
            if results and 'organic' in results:
                for r in results['organic']:
                    url = r.get("link") or r.get("url")
                    snippet = r.get("snippet") or r.get("description") # Serper uses 'description'
                    title = r.get("title", "")

                    if url and snippet:
                        # Serper often provides a date
                        publication_date_str = r.get("date")
                        metadata = {
                            "publication_date": publication_date_str
                        }
                        search_results.append(SearchResultSchema(
                            url=url,
                            snippet=snippet,
                            title=title,
                            engine="serper",
                            query=query,
                            metadata=metadata
                        ))
            return search_results
        except Exception as e:
            logger.error(f"Google Serper search error for query '{query}': {e}")
            raise e

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
htmldate 1.9.3 requires lxml<6,>=5.3.0; platform_system != "Darwin" or python_version > "3.8", but you have lxml 6.0.1 which is incompatible.


Here is a test script to demonstrate the functionality of the `SearchEngine` class.

In [46]:
warnings.filterwarnings( "ignore")

import asyncio
import nest_asyncio
from unittest.mock import AsyncMock, MagicMock
from langchain_core.documents import Document
import time # Import time for simulating crawl duration
# from langchain_tavily import TavilySearch # Keep Tavily if you want to test it
# from duckduckgo_search import DDGS # Remove DDGS import
from langchain_community.tools import DuckDuckGoSearchResults # Import the LangChain DDG tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper # Import the DDG wrapper
from langchain_community.utilities import GoogleSerperAPIWrapper # Import Google Serper
import os
# from KnowledgeState import KnowledgeState, DocumentSource

nest_asyncio.apply()
# Assume SearchEngine, KnowledgeState, DocumentSource, TavilySearch, DDGS are defined above

# 1. Create a mock KnowledgeState for the SearchEngine to update
mock_state = MagicMock(spec=KnowledgeState)
mock_state.add_error = MagicMock()
mock_state.add_results = MagicMock()
# Initialize required fields for KnowledgeState mock
mock_state.main_query = "Test Query"
mock_state.messages = []


# 2. Create actual search clients (Tavily, DDG Tool, and Serper)
# Ensure API keys are set in the environment (e.g., using _set_if_undefined or .env)
tavily_api_key = os.environ.get("TAVILY_API_KEY")
if not tavily_api_key:
    print("TAVILY_API_KEY not found. Skipping Tavily test.")
    tavily_client = None
else:
    tavily_client = TavilySearch(api_key=tavily_api_key)
    print("Tavily client initialized.")

serper_api_key = os.environ.get("SERPER_API_KEY")
if not serper_api_key:
    print("SERPER_API_KEY not found. Skipping Serper test.")
    serper_client = None
else:
    serper_client = GoogleSerperAPIWrapper() # Serper API key is read from env var by the wrapper
    print("Google Serper client initialized.")


# Create the LangChain DuckDuckGoSearchResults tool
# The wrapper configuration (region, time, max_results) is now part of the tool setup
ddg_tool = DuckDuckGoSearchResults(
    api_wrapper=DuckDuckGoSearchAPIWrapper(region="wt-wt", time="y", max_results=5),
    source="text" # Use text source for general search
)
print("LangChain DuckDuckGoSearchResults tool initialized.")


# Create a SearchConfig with the actual clients/tools
# The SearchConfig model in cell pCU1hau5LdNU was updated to allow Any for client types
mock_search_config = SearchConfig(
    external_max_results=5, # This will be used by Tavily, and potentially overridden by tool wrapper config
    subquery_max_search=2,
    subquery_results_per_query=3,
    tavily_client=tavily_client,
    # ddgs_client=None, # DDGS client is no longer used directly
    # Serper client is initialized directly within SearchEngine __init__ based on env var
)


# 3. Create a SearchEngine instance
# The SearchEngine constructor now initializes Serper if the API key is available
search_engine = SearchEngine(
    state=mock_state,
    config=mock_search_config # Pass the config with actual clients
)
# Manually set the ddg_tool on the engine instance for testing if not handled by config init
# In the updated SearchEngine __init__, the tool is created internally based on config
# search_engine._ddg_tool = ddg_tool # This line might be needed if __init__ doesn't fully handle it

print("Search Engine created with actual clients from config.")

# 4. Define queries to run
queries_to_run = ["latest AI models", "future of machine learning"]

print(f"\nRunning searches for queries: {queries_to_run}")

# 5. Run the async run_searches method
# Note: This will make actual API calls if clients are initialized
asyncio.run(search_engine.run_searches(queries_to_run, max_results_per_query=3))

print("\nSearch complete.")

# 6. Check how the mock state methods were called
print("\nMock State Calls:")
print(f"add_error called: {mock_state.add_error.call_count} times")
print(f"add_results called: {mock_state.add_results.call_count} times")

if mock_state.add_results.call_count > 0:
    # Print the arguments passed to the last add_results call
    print("\nDocuments added to state (via add_results call):")
    all_added_docs = []
    for call_args in mock_state.add_results.call_args_list:
         added_docs_list = call_args[0][0]
         all_added_docs.extend(added_docs_list)

    print(f"Number of documents added: {len(all_added_docs)}")
    if all_added_docs:
        # Access the 'source' keyword argument from the last call if needed, or assume it's consistent
        source_type = mock_state.add_results.call_args[1].get('source', 'N/A')
        print(f"Source Type (from last call): {source_type}")
        for i, doc in enumerate(all_added_docs[:10]): # Print up to 10 documents
            print(f"- Doc {i+1}: Title: {doc.metadata.get('title', 'N/A')}, Engine: {doc.metadata.get('engine', 'N/A')}, Query: {doc.metadata.get('query', 'N/A')}, URL: {doc.metadata.get('url', 'N/A')}, Publication Date: {doc.metadata.get('publication_date', 'N/A')}")


# 7. Check if the mock client methods were called as expected
print("\nClient/Tool Calls (actual):")
# Check if ainvoke was called on Tavily if initialized
if tavily_client:
    # Cannot directly count calls on actual clients without mocking, but can check if results were added
    print("Check the 'Engine' field in the added documents to see which engines/tools were used (tavily).")
else:
    print("Tavily client was not initialized due to missing API key.")

# Check if ainvoke was called on the DDG tool
if ddg_tool:
     # Cannot directly count calls on actual tools without mocking, but can check results
     print("Check the 'Engine' field in the added documents to see which engines/tools were used (ddg_langchain).")

# Check if the Serper client was initialized and potentially used
if search_engine._serper: # Access the internal attribute
     print("Check the 'Engine' field in the added documents to see which engines were used (serper).")
else:
     print("Google Serper client was not initialized due to missing API key.")

Tavily client initialized.
Google Serper client initialized.
LangChain DuckDuckGoSearchResults tool initialized.
Search Engine created with actual clients from config.

Running searches for queries: ['latest AI models', 'future of machine learning']



Search complete.

Mock State Calls:
add_error called: 0 times
add_results called: 1 times

Documents added to state (via add_results call):
Number of documents added: 19
Source Type (from last call): DocumentSource.SEARCH_SNIPPET
- Doc 1: Title: Two in-house models in support of our mission - Microsoft AI, Engine: serper, Query: latest AI models, URL: https://microsoft.ai/news/two-new-in-house-models/, Publication Date: Aug 28, 2025
- Doc 2: Title: Models - Google DeepMind, Engine: serper, Query: latest AI models, URL: https://deepmind.google/models/, Publication Date: None
- Doc 3: Title: Comparison of AI Models across Intelligence, Performance, Price, Engine: serper, Query: latest AI models, URL: https://artificialanalysis.ai/models, Publication Date: None
- Doc 4: Title: What AI Models are expected to be released in the next 2 months ..., Engine: serper, Query: latest AI models, URL: https://www.reddit.com/r/singularity/comments/1lnayv6/what_ai_models_are_expected_to_be_released_in

In [ ]:
!pip install playwright --quiet
!playwright install --with-deps chromium firefox # Install necessary browser binaries

!pip install --upgrade htmldate --quiet

In [47]:
!pip install htmldate --quiet
!pip install --upgrade htmldate --quiet # Ensure latest version

class CrawlerEngine(AsyncCrawlerProtocol):
    """
    Enhanced, state-aware, async crawler for intelligent knowledge acquisition.
    Uses AsyncChromiumLoader for dynamic sites and falls back to WebBaseLoader,
    now including logic to prioritize recently published content and using embeddings
    for content relevance scoring.
    """
    def __init__(self, state: KnowledgeState, config: CrawlerConfig, embedding_model: Optional[Any] = None): # Added embedding_model
        self.state = state
        self.config = config
        self.embedding_model = embedding_model # Store embedding model
        self.semaphore = asyncio.Semaphore(self.config.max_concurrent)
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap
        )
        self.time_threshold = datetime.now(timezone.utc) - timedelta(days=3 * 365) # For 3 years

        if self.config.enable_embedding_content_scoring and not self.embedding_model:
             logger.warning("Embedding content scoring enabled but no embedding model provided. Disabling.")
             self.config.enable_embedding_content_scoring = False


    async def crawl_urls_batch(self, urls_to_crawl: List[ExtractedURL]) -> None:
        """
        Crawl a list of ExtractedURLs, process the content, and update the state.
        This method is the main entry point for crawling within the LangGraph node.
        """
        if not urls_to_crawl:
            return

        # Sort URLs to prioritize those with fewer retries and higher priority
        # Simple approach: sort by retry_count (ascending) then by priority (descending)
        urls_to_process_in_batch = sorted(urls_to_crawl, key=lambda x: (x.retry_count, -x.priority))
        urls_to_process_in_batch = urls_to_process_in_batch[:self.config.max_concurrent] # Take only max_concurrent

        # Create a set of URLs being processed in this batch for easy removal later
        urls_being_processed_set = {url.url for url in urls_to_process_in_batch}


        crawl_tasks = [self._crawl_single_url(url_info) for url_info in urls_to_process_in_batch]
        results = await asyncio.gather(*crawl_tasks, return_exceptions=True)

        successful_crawls = 0
        failed_crawls = 0
        total_time = 0.0

        # Process results and handle retries
        processed_urls_count = len(urls_to_process_in_batch)

        # Remove the URLs processed in this batch from the state's crawl_queue
        # This needs to be done carefully to avoid issues with modifying a list while iterating
        # A robust way is to rebuild the queue or filter it.
        initial_queue = list(self.state.crawl_queue) # Get a snapshot
        self.state.crawl_queue.clear() # Clear the queue

        # Rebuild the queue with URLs NOT in the processed batch and any that were re-queued by _crawl_single_url
        # URLs re-queued by _crawl_single_url are already added back to state.crawl_queue in that method.
        # So, we just need to add back the URLs from the initial queue that were *not* processed in this batch.
        for url_info in initial_queue:
            if url_info.url not in urls_being_processed_set:
                self.state.crawl_queue.append(url_info)

        # Now process the results from the batch
        for result in results:
            if isinstance(result, Exception):
                # Error was handled within _crawl_single_url (either retried or discarded)
                failed_crawls += 1 # Still count as a failed attempt for stats
            elif result:
                # Successful crawl result (Document, crawl_time)
                document, crawl_time = result

                # Check publication date before further processing
                publication_date_str = document.metadata.get("publication_date")
                if publication_date_str:
                    try:
                        # Ensure date parsing is robust to different formats
                        pub_date = dateutil.parser.parse(publication_date_str)
                        # Ensure comparison is between timezone-aware datetimes
                        if pub_date.tzinfo is None:
                            # Assume UTC if no timezone info
                            pub_date = pub_date.replace(tzinfo=timezone.utc)

                        if pub_date < self.time_threshold:
                            logger.info(f"Skipping older document from {document.metadata.get('url')}")
                            continue # Skip processing this document
                    except (ValueError, TypeError) as e:
                        logger.warning(f"Could not parse or compare publication date '{publication_date_str}': {e}")

                successful_crawls += 1
                total_time += crawl_time

                # Split and add documents to state
                chunks = await self._split_document(document)

                if self.config.enable_embedding_content_scoring and self.embedding_model:
                    # Use embedding-based scoring
                    scored_chunks = await self._score_content_embedding(chunks, self.state.main_query)
                    relevant_chunks = [
                        c for c in scored_chunks
                        if c.metadata.get("embedding_relevance_score", 0) >= self.config.embedding_relevance_threshold
                    ]
                    if relevant_chunks:
                        self.state.add_results(relevant_chunks, source=DocumentSource.WEB_CRAWL)
                    else:
                         logger.info(f"No relevant chunks found after embedding scoring for {document.metadata.get('url')}")
                else:
                    # Add all chunks if embedding scoring is disabled or not possible
                    self.state.add_results(chunks, source=DocumentSource.WEB_CRAWL)

        self.state.update_crawl_stats(
            attempted=processed_urls_count, # Use the count of URLs processed in this batch
            successful=successful_crawls,
            failed=failed_crawls,
            time=total_time
        )

        # URLs that were successfully crawled or discarded are implicitly removed
        # because they are not re-added to the queue by _crawl_single_url.
        # Only URLs needing retry are added back by _crawl_single_url.
        pass # The queue manipulation is done above


    async def _crawl_single_url(self, url_info: ExtractedURL) -> Optional[tuple[Document, float]]:
        """Crawl a single URL safely and return the result or None if retried/discarded."""
        url = url_info.url
        start_time = time.perf_counter()

        try:
            async with self.semaphore:
                # Check if the URL has already been retried too many times
                if url_info.retry_count >= self.config.max_retries: # Use >= for correct check
                    logger.warning(f"Discarding URL {url} after {url_info.retry_count} retries.")
                    # Do not return anything for this URL, it's discarded
                    return None

                document = await self._load_and_extract(url)

                if not document or not document.page_content:
                    self.state.add_error(f"No content extracted from {url}")
                    return None

                end_time = time.perf_counter()
                crawl_time = end_time - start_time

                if len(document.page_content) > self.config.max_content_length:
                    document.page_content = document.page_content[:self.config.max_content_length] + "..."
                    document.metadata["truncated"] = True

                document.metadata.update({
                    "crawl_timestamp": datetime.now(timezone.utc).isoformat(),
                    "content_length": len(document.page_content),
                    "crawl_time": crawl_time,
                    "url": url,
                    "query": self.state.main_query
                })

                # Add publication and modification dates to metadata
                publication_date = await self._extract_publication_date(document.page_content)
                if publication_date:
                    document.metadata["publication_date"] = publication_date

                modification_date = await self._extract_modification_date(document.page_content)
                if modification_date:
                    document.metadata["modification_date"] = modification_date

                # Reset retry count on successful crawl
                url_info.retry_count = 0

                return (document, crawl_time)

        except (httpx.TimeoutException, httpx.ConnectError) as e:
            # Handle specific network errors for retry
            url_info.retry_count += 1
            if url_info.retry_count <= self.config.max_retries:
                logger.warning(f"Network error for {url} (attempt {url_info.retry_count}/{self.config.max_retries}). Re-queueing.")
                # Add the URL back to the crawl queue for retry
                self.state.crawl_queue.append(url_info)
                # Do not return anything for this URL yet, it's being retried
                return None
            else:
                logger.error(f"Network error for {url} after {url_info.retry_count} retries. Discarding URL. Error: {e}")
                self.state.add_error(f"Failed to crawl {url} after retries: {str(e)}")
                # Do not return anything, it's discarded
                return None

        except Exception as e:
            # Handle other exceptions (parsing errors, etc.) - do not retry for these
            logger.error(f"Failed to crawl {url} due to unexpected error: {e}")
            self.state.add_error(f"Failed to crawl {url}: {str(e)}")
            # For other errors, we don't retry and don't return content
            return None


    async def _load_and_extract(self, url: str) -> Optional[Document]:
        """
        Attempts to load a document using AsyncChromiumLoader, falling back to WebBaseLoader.
        Includes basic error handling for URL loading.
        """
        doc = None
        # Try AsyncChromiumLoader first for dynamic content
        try:
            loader = AsyncChromiumLoader([url])
            docs = await loader.aload()
            if docs and docs[0].page_content and docs[0].page_content.strip():
                docs[0].metadata["loader"] = "chromium"
                doc = docs[0]
                logger.info(f"Successfully loaded {url} with ChromiumLoader.")
            else:
                 logger.warning(f"Chromium loader got empty content for {url}.")
        except Exception as e:
            # Log the specific error but don't fail the crawl task yet
            logger.warning(f"Chromium loader failed for {url}: {e}")

        # Fallback to WebBaseLoader if Chromium fails or is not installed, or returned empty content
        if not doc:
            try:
                def blocking_load():
                     return WebBaseLoader(web_paths=(url,)).load()
                docs = await asyncio.to_thread(blocking_load)
                if docs and docs[0].page_content and docs[0].page_content.strip():
                    docs[0].metadata["loader"] = "webbase"
                    doc = docs[0]
                    logger.info(f"Successfully loaded {url} with WebBaseLoader.")
                else:
                    logger.warning(f"WebBaseLoader got empty content for {url}.")
            except Exception as e:
                logger.warning(f"WebBaseLoader failed for {url}: {e}")
                # If both loaders fail, return None
                return None

        return doc


    async def _extract_publication_date(self, html_content: str) -> Optional[str]:
        """
        Extracts the publication date from HTML content using htmldate.
        Uses asyncio.to_thread as htmldate is not natively async.
        Handles potential errors during date extraction.
        """
        try:
            # Using original_date=True to prioritize explicitly marked dates
            # Removed timeout argument as it seems unsupported or causing issues
            return await asyncio.to_thread(find_date, html_content, original_date=True)
        except Exception as e:
            # Log htmldate specific errors
            logger.warning(f"Failed to extract publication date with htmldate: {e}")
            return None

    async def _extract_modification_date(self, html_content: str) -> Optional[str]:
        """
        Extracts the modification date from HTML content using htmldate.
        Uses asyncio.to_thread as htmldate is not natively async.
        Handles potential errors during date extraction.
        Note: htmldate's ability to find modification dates might be limited or require specific arguments not available in all versions.
        Trying without specific arguments first, or exploring alternative libraries might be necessary.
        """
        try:
            # Calling find_date without the problematic 'lastmod=True' or 'timeout'
            # It might still find a date, but less likely to be the modification date specifically
            return await asyncio.to_thread(find_date, html_content)
        except Exception as e:
            logger.warning(f"Failed to extract modification date with htmldate: {e}")
            return None


    async def _split_document(self, doc: Document) -> List[Document]:
        """Split a document into chunks asynchronously."""
        return await asyncio.to_thread(self.splitter.split_documents, [doc])

    async def _score_content_embedding(self, chunks: List[Document], query: str) -> List[Document]:
        """
        Use embeddings to score content chunks against the query based on semantic similarity.
        Returns the chunks with an added 'embedding_relevance_score' in their metadata.
        """
        if not self.embedding_model or not chunks:
            return chunks

        try:
            # Embed the query
            query_embedding = await asyncio.to_thread(self.embedding_model.embed_query, query)

            # Embed the documents/chunks
            # Embedding a list of documents can be done in a single call
            chunk_texts = [chunk.page_content for chunk in chunks]
            # Corrected the call to embed_documents - it should be a method call, not subscripting
            chunk_embeddings = await asyncio.to_thread(self.embedding_model.embed_documents, chunk_texts)

            # Calculate cosine similarity between query embedding and each chunk embedding
            # Cosine similarity ranges from -1 (opposite) to 1 (identical).
            # We want scores closer to 1.0.
            scores = []
            for chunk_embedding in chunk_embeddings:
                # Using dot product for cosine similarity if embeddings are normalized
                # If not normalized, need to normalize or use a different similarity metric
                # SentenceTransformer embeddings are usually normalized.
                score = sum(q * c for q, c in zip(query_embedding, chunk_embedding))
                # Ensure score is within [0, 1] range if needed, though cosine similarity is [-1, 1]
                # Map [-1, 1] to [0, 1] for thresholding: (score + 1) / 2
                normalized_score = (score + 1) / 2
                scores.append(normalized_score)


            # Add scores to chunk metadata
            for i, chunk in enumerate(chunks):
                chunk.metadata["embedding_relevance_score"] = scores[i]

            return chunks

        except Exception as e:
            logger.error(f"Embedding scoring failed: {e}")
            # If embedding fails, return original chunks without scores or with a default low score
            for chunk in chunks:
                 chunk.metadata["embedding_relevance_score"] = 0.0 # Assign a low score on failure
            return chunks


    async def _score_content_llm(self, chunks: List[Document], query: str) -> List[Document]:
        """
        Use an LLM to score content chunks against the query.
        Returns the chunks with an added 'llm_relevance_score' in their metadata.
        (Kept for reference if needed, but embedding is preferred as per user feedback)
        """
        if not self.llm or not chunks:
            return chunks

        scoring_prompt = """
        You are a quality assurance assistant for a web crawler. Your task is to score document chunks based on their relevance to a given query, and prioritize more recent content.
        Provide a single relevance score from 0.0 to 1.0 for each chunk.
        Take into account both the topical relevance and the recency of the content.
        Query: {query}

        Example Output Format:
        [
          {{ "chunk_index": 0, "score": 0.85 }},
          {{ "chunk_index": 1, "score": 0.20 }},
          ...
        ]

        Now, score the following chunks:
        """

        batch_size = 5
        for i in range(0, len(chunks), batch_size):
            chunk_batch = chunks[i:i+batch_size]
            prompt = scoring_prompt.format(query=query)
            for j, chunk in enumerate(chunk_batch):
                # Include date in the prompt for LLM context
                date_info = f"Publication Date: {chunk.metadata.get('publication_date', 'N/A')}\nModification Date: {chunk.metadata.get('modification_date', 'N/A')}\n"
                prompt += f"\n--- Chunk {i+j} ---\n{date_info}{chunk.page_content[:500]}...\n" # Corrected typo here
            try:
                response = await self.llm.ainvoke(prompt)
                scores = self._parse_llm_response(response.content)

                for score_info in scores:
                    chunk_index = score_info.get("chunk_index")
                    score = score_info.get("score")
                    if chunk_index is not None and score is not None and 0 <= chunk_index < len(chunks):
                         chunks[chunk_index].metadata["llm_relevance_score"] = score
            except Exception as e:
                logger.error(f"LLM scoring failed for batch {i}: {e}")

        return chunks

    def _parse_llm_response(self, text: str) -> List[Dict[str, Union[int, float]]]:
        """Parses the LLM's response to extract scores."""
        try:
            json_match = re.search(r'\[\s*\{.*?\}\s*\]', text, re.DOTALL)
            if json_match:
                # Use json.loads for safer parsing
                import json
                return json.loads(json_match.group(0))
        except Exception as e:
            logger.error(f"Failed to parse LLM response: {e}")
            raise e
        return []

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ddgs 9.5.5 requires lxml>=6.0.0, but you have lxml 5.4.0 which is incompatible.


Here is a test script to demonstrate the functionality of the `CrawlerEngine` class. It includes several edge cases.

In [52]:
import asyncio
import nest_asyncio
from unittest.mock import MagicMock
from langchain_core.documents import Document
import time # Import time for simulating operations
import logging
from datetime import datetime, timedelta, timezone
from typing import Dict, Any, List, Optional, Protocol, Union
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader, AsyncChromiumLoader
import bs4
import re
import httpx
from pydantic import BaseModel, Field
from datetime import datetime, timezone
from htmldate import find_date # New import for date extraction
import dateutil.parser # Needed for flexible date parsing
from bs4 import BeautifulSoup # Import BeautifulSoup for parsing HTML metadata
from langchain_community.embeddings import SentenceTransformerEmbeddings # Import embeddings

nest_asyncio.apply()

# 1. Create a mock KnowledgeState for the CrawlerEngine to update
mock_state = MagicMock(spec=KnowledgeState)
mock_state.add_error = MagicMock()
mock_state.add_results = MagicMock()
mock_state.update_crawl_stats = MagicMock()
mock_state.main_query = "Latest advancements in AI"
mock_state.crawl_queue = [] # Simulate crawl_queue as a list attribute that can be modified

# 2. Create a CrawlerConfig instance, enabling embedding scoring for testing
config = CrawlerConfig(
    chunk_size=500,
    chunk_overlap=100,
    max_concurrent=3, # Test concurrency
    timeout=5, # Set a lower timeout to trigger simulated timeouts
    max_content_length=10000, # Test truncation
    enable_embedding_content_scoring=True, # Enable embedding scoring for testing
    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2", # Specify embedding model
    embedding_relevance_threshold=0.4, # Set a threshold for filtering
    max_retries=2 # Set max retries for testing
)
print("Crawler Config:")
print(config.model_dump_json(indent=2))

# 3. Initialize a real Embedding Model for scoring
try:
    real_embedding_model = SentenceTransformerEmbeddings(model_name=config.embedding_model_name)
    print("\nInitialized real Embedding Model for scoring.")
except Exception as e:
    print(f"\nFailed to initialize real Embedding Model: {e}. Embedding scoring will not run.")
    real_embedding_model = None
    config.enable_embedding_content_scoring = False # Disable if embedding model fails

# 4. Create a CrawlerEngine instance, passing the real Embedding Model
crawler_engine = CrawlerEngine(state=mock_state, config=config, embedding_model=real_embedding_model)
print("\nCrawler Engine created.")

# 5. Prepare a list of simulated ExtractedURLs to crawl, including edge cases
# Simulate URLs that will succeed, fail and be retried, and fail permanently
urls_to_crawl_initial = [
    # Real URLs that should succeed
    ExtractedURL(query=mock_state.main_query, url="https://microsoft.ai/news/two-new-in-house-models/", priority=0.9, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://deepmind.google/models/", priority=0.85, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://huggingface.co/", priority=0.7, should_crawl=True),

    # Simulate URLs that will timeout and need retries
    # Use httpbin.org/delay/ to simulate delays longer than the timeout
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/6", priority=0.6, should_crawl=True), # Will timeout, needs retry 1
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/7", priority=0.6, should_crawl=True), # Will timeout, needs retry 1

    # Simulate a URL that will fail consistently and be discarded after max_retries
    # Start it with retry_count = max_retries - 1 so it fails one more time and is discarded
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/8", priority=0.5, should_crawl=True, retry_count=config.max_retries - 1),

    # Simulate a URL that should NOT be crawled based on should_crawl=False
    ExtractedURL(query=mock_state.main_query, url="https://github.com/", priority=0.2, should_crawl=False),
]

# Add the initial URLs to the mock state's crawl_queue
mock_state.crawl_queue.extend([url for url in urls_to_crawl_initial if url.should_crawl])

print(f"\nInitial Crawl Queue ({len(mock_state.crawl_queue)} URLs):")
for url_info in mock_state.crawl_queue:
     print(f"- URL: {url_info.url}, Priority: {url_info.priority}, Retry Count: {url_info.retry_count}")


print(f"\nStarting crawling process with max_concurrent={config.max_concurrent}, max_retries={config.max_retries}...")

# 6. Run the async crawl_urls_batch method in multiple iterations to simulate retries
num_iterations = 5 # Run multiple iterations to allow retries to happen

for iteration in range(num_iterations):
    print(f"\n--- Running Crawl Iteration {iteration + 1} ---")
    if not mock_state.crawl_queue:
        print("Crawl queue is empty. Stopping.")
        break

    # Pass a copy of the current crawl queue to crawl_urls_batch
    # The method will select its batch internally and modify state.crawl_queue directly
    current_queue_snapshot = list(mock_state.crawl_queue)
    asyncio.run(crawler_engine.crawl_urls_batch(current_queue_snapshot))

    print(f"Crawl Queue after iteration {iteration + 1} ({len(mock_state.crawl_queue)} URLs):")
    for url_info in mock_state.crawl_queue:
         print(f"- URL: {url_info.url}, Priority: {url_info.priority}, Retry Count: {url_info.retry_count}")


print("\nCrawling process finished.")

# 7. Check the final state and mock calls
print("\nMock State Calls (Final):")
print(f"add_error called: {mock_state.add_error.call_count} times")
print(f"add_results called: {mock_state.add_results.call_count} times")
print(f"update_crawl_stats called: {mock_state.update_crawl_stats.call_count} times (should match num_iterations or fewer if queue emptied early)")

# Print final crawl stats summary
if mock_state.update_crawl_stats.call_count > 0:
    print("\nFinal Crawl Stats Summary:")
    # Sum up stats from all update_crawl_stats calls
    total_attempted = sum(call_kwargs.get('attempted', 0) for call_args, call_kwargs in mock_state.update_crawl_stats.call_args_list)
    total_successful = sum(call_kwargs.get('successful', 0) for call_args, call_kwargs in mock_state.update_crawl_stats.call_args_list)
    total_failed = sum(call_kwargs.get('failed', 0) for call_args, call_kwargs in mock_state.update_crawl_stats.call_args_list)
    total_time = sum(call_kwargs.get('time', 0.0) for call_args, call_kwargs in mock_state.update_crawl_stats.call_args_list)

    print(f"Total Attempted: {total_attempted}")
    print(f"Total Successful: {total_successful}")
    print(f"Total Failed (attempts): {total_failed}")
    print(f"Total Time: {total_time:.2f}")


# Verify that URLs that timed out max_retries times are no longer in the queue
discarded_url = "https://httpbin.org/delay/8"
is_discarded_url_in_queue = any(url_info.url == discarded_url for url_info in mock_state.crawl_queue)
print(f"\nIs discarded URL ({discarded_url}) still in queue? {is_discarded_url_in_queue}")

# Verify that URLs that timed out but still have retries are in the queue with updated retry_count
retry_url_1 = "https://httpbin.org/delay/6"
retry_url_2 = "https://httpbin.org/delay/7"
retry_url_1_in_queue = next((url_info for url_info in mock_state.crawl_queue if url_info.url == retry_url_1), None)
retry_url_2_in_queue = next((url_info for url_info in mock_state.crawl_queue if url_info.url == retry_url_2), None)

print(f"\nIs retry URL 1 ({retry_url_1}) in queue? {retry_url_1_in_queue is not None}")
if retry_url_1_in_queue:
    print(f"  Retry URL 1 final retry_count: {retry_url_1_in_queue.retry_count}")

print(f"Is retry URL 2 ({retry_url_2}) in queue? {retry_url_2_in_queue is not None}")
if retry_url_2_in_queue:
    print(f"  Retry URL 2 final retry_count: {retry_url_2_in_queue.retry_count}")

# Verify that successfully crawled URLs are NOT in the queue
successful_urls = ["https://microsoft.ai/news/two-new-in-house-models/", "https://deepmind.google/models/", "https://huggingface.co/"]
for url in successful_urls:
    is_successful_url_in_queue = any(url_info.url == url for url_info in mock_state.crawl_queue)
    print(f"\nIs successful URL ({url}) still in queue? {is_successful_url_in_queue}")

# Remove the line that checks LLM call count as it's no longer relevant
# if config.enable_ml_content_scoring and real_llm:
#     print(f"\nLLM ainvoke called: {real_llm.ainvoke.call_count} times (for content scoring batches)")

Crawler Config:
{
  "chunk_size": 500,
  "chunk_overlap": 100,
  "max_concurrent": 3,
  "timeout": 5,
  "max_content_length": 10000,
  "enable_embedding_content_scoring": true,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_relevance_threshold": 0.4,
  "max_retries": 2
}



Initialized real Embedding Model for scoring.

Crawler Engine created.

Initial Crawl Queue (6 URLs):
- URL: https://microsoft.ai/news/two-new-in-house-models/, Priority: 0.9, Retry Count: 0
- URL: https://deepmind.google/models/, Priority: 0.85, Retry Count: 0
- URL: https://huggingface.co/, Priority: 0.7, Retry Count: 0
- URL: https://httpbin.org/delay/6, Priority: 0.6, Retry Count: 0
- URL: https://httpbin.org/delay/7, Priority: 0.6, Retry Count: 0
- URL: https://httpbin.org/delay/8, Priority: 0.5, Retry Count: 1

Starting crawling process with max_concurrent=3, max_retries=2...

--- Running Crawl Iteration 1 ---


INFO:__main__:Successfully loaded https://microsoft.ai/news/two-new-in-house-models/ with ChromiumLoader.
INFO:__main__:Successfully loaded https://deepmind.google/models/ with ChromiumLoader.
INFO:__main__:Successfully loaded https://huggingface.co/ with ChromiumLoader.


Crawl Queue after iteration 1 (3 URLs):
- URL: https://httpbin.org/delay/6, Priority: 0.6, Retry Count: 0
- URL: https://httpbin.org/delay/7, Priority: 0.6, Retry Count: 0
- URL: https://httpbin.org/delay/8, Priority: 0.5, Retry Count: 1

--- Running Crawl Iteration 2 ---


INFO:__main__:Successfully loaded https://httpbin.org/delay/6 with ChromiumLoader.
INFO:__main__:Successfully loaded https://httpbin.org/delay/7 with ChromiumLoader.
INFO:__main__:Successfully loaded https://httpbin.org/delay/8 with ChromiumLoader.


Crawl Queue after iteration 2 (0 URLs):

--- Running Crawl Iteration 3 ---
Crawl queue is empty. Stopping.

Crawling process finished.

Mock State Calls (Final):
add_error called: 0 times
add_results called: 6 times
update_crawl_stats called: 2 times (should match num_iterations or fewer if queue emptied early)

Final Crawl Stats Summary:
Total Attempted: 6
Total Successful: 6
Total Failed (attempts): 0
Total Time: 51.84

Is discarded URL (https://httpbin.org/delay/8) still in queue? False

Is retry URL 1 (https://httpbin.org/delay/6) in queue? False
Is retry URL 2 (https://httpbin.org/delay/7) in queue? False

Is successful URL (https://microsoft.ai/news/two-new-in-house-models/) still in queue? False

Is successful URL (https://deepmind.google/models/) still in queue? False

Is successful URL (https://huggingface.co/) still in queue? False


In [53]:
import asyncio
import logging
from typing import ClassVar, Dict, Any, List, Optional, Protocol, Union
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field
# from KnowledgeState import KnowledgeState

# Assume your embedding model is available
# from langchain_community.embeddings import SomeEmbeddingModel

logger = logging.getLogger(__name__)

# -------------------- Async Cleaner Protocol --------------------

class AsyncCleanerProtocol(Protocol):
    """Protocol for an asynchronous document cleaner."""
    async def clean_and_chunk_documents(self, docs: List[Document]) -> None:
        """Asynchronously clean, chunk, and update state with documents."""
        ...

# -------------------- Enhanced Cleaner Engine --------------------

class CleanerEngine(AsyncCleanerProtocol):
    """
    Enhanced, state-aware, async cleaner for documents.
    Supports intelligent, embedding-based chunking.
    """

    def __init__(self, state: KnowledgeState, config: CleanerConfig, embedding_model: Optional[Any] = None):
        self.state = state
        self.config = config
        self.embedding_model = embedding_model
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap
        )
        if self.config.use_semantic_chunking and not self.embedding_model:
            logger.warning("Semantic chunking enabled but no embedding model provided. Falling back to recursive splitting.")
            self.config.use_semantic_chunking = False

    async def clean_and_chunk_documents(self, docs: List[Document]) -> None:
        """
        Cleans and chunks documents, then updates the shared KnowledgeState.
        """
        if not docs:
            return

        tasks = [self._process_single_doc(doc) for doc in docs]
        processed_chunks = await asyncio.gather(*tasks, return_exceptions=True)

        all_cleaned_chunks = []
        for result in processed_chunks:
            if isinstance(result, Exception):
                self.state.add_error(f"Document processing failed: {str(result)}")
            elif result:
                all_cleaned_chunks.extend(result)

        if all_cleaned_chunks:
            self.state.cleaned_chunks.extend(all_cleaned_chunks)
            logger.info(f"Added {len(all_cleaned_chunks)} total chunks to state.")

    async def _process_single_doc(self, doc: Document) -> Optional[List[Document]]:
        """
        Cleans and chunks a single document.
        """
        try:
            if not doc or not doc.page_content or not doc.page_content.strip():
                return None

            cleaned_doc = await self._strip_html(doc)
            if not cleaned_doc.page_content or not cleaned_doc.page_content.strip():
                return None

            if self.config.use_semantic_chunking:
                return await self._semantic_chunk(cleaned_doc)
            else:
                return await self._recursive_chunk(cleaned_doc)

        except Exception as e:
            raise Exception(f"Failed to process document from {doc.metadata.get('url', 'unknown')}: {e}")

    async def _strip_html(self, doc: Document) -> Document:
        """
        Removes HTML tags from a document's content using a separate thread.
        """
        return await asyncio.to_thread(self._strip_html_sync, doc)

    def _strip_html_sync(self, doc: Document) -> Document:
        """Synchronous HTML stripping with BeautifulSoup."""
        try:
            if not doc.page_content:
                return Document(page_content="", metadata=doc.metadata)

            text = BeautifulSoup(doc.page_content, "lxml").get_text(" ", strip=True)
            # Add additional cleaning logic here (e.g., removing extra whitespace)
            cleaned_text = re.sub(r'\s+', ' ', text).strip()
            return Document(page_content=cleaned_text, metadata=doc.metadata)

        except Exception as e:
            logger.warning(f"Failed to strip HTML from document: {e}")
            return Document(page_content="", metadata=doc.metadata)

    async def _recursive_chunk(self, doc: Document) -> List[Document]:
        """
        Chunks a document using the RecursiveCharacterTextSplitter.
        """
        # The splitting process can be CPU-bound, so run in a thread
        return await asyncio.to_thread(self.splitter.split_documents, [doc])

    async def _semantic_chunk(self, doc: Document) -> List[Document]:
        """
        Intelligent, semantic-based chunking using embeddings.
        Requires an embedding model to be configured.
        """
        if not self.embedding_model:
            logger.warning("Embedding model not set for semantic chunking. Falling back to recursive.")
            return await self._recursive_chunk(doc)

        # Example of semantic chunking logic
        # This is a simplified, non-LangChain specific implementation for illustration.
        sentences = doc.page_content.split('. ')
        if len(sentences) <= 1:
            return await self._recursive_chunk(doc) # Fallback if not enough sentences

        # This part requires an actual embedding model and clustering logic
        # Example using placeholder logic:
        chunks = []
        current_chunk_sentences = []
        for sentence in sentences:
            current_chunk_sentences.append(sentence)
            if len(' '.join(current_chunk_sentences)) > self.config.chunk_size:
                chunks.append(Document(page_content=' '.join(current_chunk_sentences), metadata=doc.metadata))
                current_chunk_sentences = []
        if current_chunk_sentences:
            chunks.append(Document(page_content=' '.join(current_chunk_sentences), metadata=doc.metadata))

        return chunks


Here is a test script to demonstrate the functionality of the `CleanerEngine` class. It includes several edge cases.

In [54]:
import asyncio
import nest_asyncio
from unittest.mock import AsyncMock, MagicMock
from langchain_core.documents import Document
import time # Import time for simulating operations
# Assume KnowledgeState, CleanerConfig, CleanerEngine are defined above

nest_asyncio.apply()

# 1. Create a mock KnowledgeState for the CleanerEngine to update
mock_state = MagicMock(spec=KnowledgeState)
mock_state.add_error = MagicMock()
mock_state.cleaned_chunks = [] # Simulate the actual list attribute for inspection

# 2. Create a CleanerConfig instance, testing both chunking methods
config_recursive = CleanerConfig(
    chunk_size=200, # Corrected chunk_size to meet validation
    chunk_overlap=20,
    use_semantic_chunking=False # Test recursive chunking
)
config_semantic = CleanerConfig(
    chunk_size=200, # Also corrected chunk_size to meet validation
    chunk_overlap=30,
    use_semantic_chunking=True # Test semantic chunking
)
print("Cleaner Config (Recursive):")
print(config_recursive.model_dump_json(indent=2))
print("\nCleaner Config (Semantic):")
print(config_semantic.model_dump_json(indent=2))


# 3. Create mock Embedding Model (needed for semantic chunking config)
# It doesn't need complex behavior for this test, just to exist.
mock_embedding_model = MagicMock()

# 4. Create CleanerEngine instances for both configurations
cleaner_recursive = CleanerEngine(state=mock_state, config=config_recursive)
cleaner_semantic = CleanerEngine(state=mock_state, config=config_semantic, embedding_model=mock_embedding_model)
print("\nCleaner Engines created.")

# 5. Prepare a list of simulated documents, including edge cases
documents_to_clean = [
    # Standard document with HTML
    Document(page_content="<html><body><h1>Title</h1><p>This is some content with <b>bold</b> text and <a href='#'>a link</a>.</p></body></html>", metadata={"url": "https://example.com/html-page"}),
    # Document that should be recursively chunked
    Document(page_content="This document is long enough to be split into multiple recursive chunks. It has several sentences and paragraphs that exceed the configured chunk size. We need to ensure the splitter works correctly and maintains overlap.", metadata={"url": "https://example.com/long-text"}),
    # Document for semantic chunking (if enabled)
    Document(page_content="Apple is a technology company. They make iPhones and Macs. Semantic chunking might group related sentences.  Banana is a fruit. It grows on trees. This sentence is about something else.", metadata={"url": "https://example.com/semantic-text"}),
    # Document with empty content
    Document(page_content="", metadata={"url": "https://example.com/empty-page"}),
    # Document with only whitespace
    Document(page_content="   \n\n  \t ", metadata={"url": "https://example.com/whitespace-only"}),
    # Document with basic text
    Document(page_content="Simple text document.", metadata={"url": "https://example.com/simple-text"}),
     # Document with complex HTML and scripts (will be stripped)
    Document(page_content="""
    <html><body>
    <script>console.log('hello');</script>
    <div>Content inside div.</div>
    <!-- comment -->
    <p>Another paragraph.</p>
    </body></html>
    """, metadata={"url": "https://example.com/complex-html"}),
]

print(f"\nProcessing batch of documents ({len(documents_to_clean)} total)...")
mock_state.cleaned_chunks = [] # Reset the mock list before processing

# 6. Run the async clean_and_chunk_documents method with recursive cleaner
print("\n--- Testing Recursive Chunking ---")
asyncio.run(cleaner_recursive.clean_and_chunk_documents(documents_to_clean))

print("\nProcessing complete for recursive chunking.")

# 7. Check mock state calls for recursive cleaner
print("\nMock State Calls (Recursive):")
print(f"add_error called: {mock_state.add_error.call_count} times (should be at least 2 for empty/whitespace docs)")
print(f"Total chunks added to state: {len(mock_state.cleaned_chunks)}")

if len(mock_state.cleaned_chunks) > 0:
    print("\nSample Cleaned Chunks (Recursive):")
    for i, chunk in enumerate(mock_state.cleaned_chunks[:5]): # Print first 5
        print(f"- Chunk {i+1}: Length {len(chunk.page_content)}, URL: {chunk.metadata.get('url', 'N/A')}")
        # print(f"  Content: {chunk.page_content[:100]}...") # Uncomment to see snippet

# Reset mock state for semantic testing
mock_state.reset_mock()
mock_state.add_error = MagicMock()
mock_state.cleaned_chunks = [] # Reset the mock list attribute

print("\n--- Testing Semantic Chunking (with mock embedding) ---")
# Mock the internal _semantic_chunk method to control output for testing
async def mock_semantic_chunk(doc: Document) -> List[Document]:
    # Simple simulation: split into chunks based on sentence end for demonstration
    if not doc.page_content:
        return []
    sentences = doc.page_content.split('. ')
    chunks = []
    current_chunk_sentences = []
    current_length = 0
    for i, sentence in enumerate(sentences):
         sentence_with_dot = sentence + '.' if i < len(sentences) - 1 else sentence # Add dot back
         sentence_length = len(sentence_with_dot) + (1 if current_chunk_sentences else 0) # Add space if not first
         if current_length + sentence_length > cleaner_semantic.config.chunk_size and current_chunk_sentences:
              chunks.append(Document(page_content=' '.join(current_chunk_sentences), metadata=doc.metadata))
              current_chunk_sentences = [sentence_with_dot]
              current_length = len(sentence_with_dot)
         else:
              current_chunk_sentences.append(sentence_with_dot)
              current_length += sentence_length

    if current_chunk_sentences:
        chunks.append(Document(page_content=' '.join(current_chunk_sentences), metadata=doc.metadata))

    # Simulate adding some metadata from semantic process if needed
    for chunk in chunks:
        chunk.metadata['chunking_method'] = 'semantic_mock'

    return chunks

cleaner_semantic._semantic_chunk = AsyncMock(side_effect=mock_semantic_chunk)

# Run the async clean_and_chunk_documents method with semantic cleaner
asyncio.run(cleaner_semantic.clean_and_chunk_documents(documents_to_clean))

print("\nProcessing complete for semantic chunking.")

# 8. Check mock state calls for semantic cleaner
print("\nMock State Calls (Semantic):")
print(f"add_error called: {mock_state.add_error.call_count} times (should be at least 2 for empty/whitespace docs)")
print(f"Total chunks added to state: {len(mock_state.cleaned_chunks)}")

if len(mock_state.cleaned_chunks) > 0:
    print("\nSample Cleaned Chunks (Semantic):")
    for i, chunk in enumerate(mock_state.cleaned_chunks[:5]): # Print first 5
        print(f"- Chunk {i+1}: Length {len(chunk.page_content)}, URL: {chunk.metadata.get('url', 'N/A')}, Method: {chunk.metadata.get('chunking_method', 'N/A')}")
        # print(f"  Content: {chunk.page_content[:100]}...") # Uncomment to see snippet

# Check if embedding model was accessed (shouldn't be if _semantic_chunk is mocked)
# If _semantic_chunk was not mocked, you would check mock_embedding_model calls here.

INFO:__main__:Added 6 total chunks to state.
INFO:__main__:Added 6 total chunks to state.


Cleaner Config (Recursive):
{
  "chunk_size": 200,
  "chunk_overlap": 20,
  "use_semantic_chunking": false,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2"
}

Cleaner Config (Semantic):
{
  "chunk_size": 200,
  "chunk_overlap": 30,
  "use_semantic_chunking": true,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2"
}

Cleaner Engines created.

Processing batch of documents (7 total)...

--- Testing Recursive Chunking ---

Processing complete for recursive chunking.

Mock State Calls (Recursive):
add_error called: 0 times (should be at least 2 for empty/whitespace docs)
Total chunks added to state: 6

Sample Cleaned Chunks (Recursive):
- Chunk 1: Length 54, URL: https://example.com/html-page
- Chunk 2: Length 198, URL: https://example.com/long-text
- Chunk 3: Length 38, URL: https://example.com/long-text
- Chunk 4: Length 184, URL: https://example.com/semantic-text
- Chunk 5: Length 21, URL: https://example.com/simple-text

--- Testing Semantic Chunking

In [55]:
!pip install chromadb --quiet
!pip install langchain-core --quiet
!pip install --upgrade langchain --quiet
!pip install -U langchain-community upstash-redis --quiet
!pip install --upgrade  sqlite-vec  --quiet

import os
import logging
import asyncio
import time
from typing import Dict, Any, Optional, List

from pydantic import BaseModel, Field, PrivateAttr, ValidationError
from langchain_core.documents import Document
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import RedisStore, InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings, SentenceTransformerEmbeddings
from tenacity import retry, stop_after_attempt, wait_exponential, RetryError

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class AsyncVectorStoreProtocol(Protocol):
    """Protocol for an asynchronous vector store node."""
    async def index_documents(self, state: KnowledgeState) -> Dict[str, Any]:
        """Indexes cleaned chunks from state and updates vector store."""
        ...

# -------------------------------------------------------------------
# --- VectorStoreNode (Chroma only)
# -------------------------------------------------------------------
class VectorStoreNode(AsyncVectorStoreProtocol):
    _vectorstore: Any = PrivateAttr()
    _embeddings: Any = PrivateAttr()

    def __init__(self, config: VectorStoreConfig):
        self.config = config
        try:
            self._initialize_components()
        except (ValueError, ValidationError) as e:
            logger.critical(f"Failed to initialize VectorStoreNode: {e}")
            raise

    # ----------------- Helpers -----------------
    def _get_embedding_model(self):
        if self.config.embedding_model_provider == "huggingface":
            return HuggingFaceEmbeddings(model_name=self.config.embedding_model_name)
        elif self.config.embedding_model_provider == "sentence-transformer":
            return SentenceTransformerEmbeddings(model_name=self.config.embedding_model_name)
        else:
            raise ValueError(f"Unknown embedding provider: {self.config.embedding_model_provider}")

    def _initialize_components(self):
        # Ensure persistence dir exists
        os.makedirs(self.config.chroma_persist_directory, exist_ok=True)

        # Embeddings
        base_embedder = self._get_embedding_model()
        cache_store = InMemoryStore()
        self._embeddings = CacheBackedEmbeddings.from_bytes_store(
            base_embedder,
            document_embedding_cache=cache_store,
            namespace=self.config.collection_name
        )

        # Chroma setup
        self._vectorstore = Chroma(
            collection_name=self.config.collection_name,
            embedding_function=self._embeddings,
            persist_directory=self.config.chroma_persist_directory,
        )
        logger.info(f"Chroma initialized at {self.config.chroma_persist_directory}")

    # ----------------- Indexing -----------------
    @retry(wait=wait_exponential(multiplier=1, min=2, max=10), stop=stop_after_attempt(5), reraise=True)
    async def _index_with_retry(self, documents: List[Document]):
        start = time.perf_counter()
        await asyncio.to_thread(self._vectorstore.add_documents, documents)
        end = time.perf_counter()
        logger.info(f"Indexed {len(documents)} docs in {end - start:.2f}s")

    async def index_documents(self, state: KnowledgeState) -> Dict[str, Any]:
        if not state.cleaned_chunks:
            logger.info("No docs to index.")
            return {"cleaned_chunks": [], "errors": []}

        # Convert to LC Documents
        docs = []
        for d in state.cleaned_chunks:
            if isinstance(d, Document):
                docs.append(d)
            elif isinstance(d, dict):
                docs.append(Document(
                    page_content=d["page_content"],
                    metadata=d.get("metadata", {})
                ))
            else:
                raise TypeError(f"Unsupported type in cleaned_chunks: {type(d)}")

        try:
            await self._index_with_retry(docs)
            return {"cleaned_chunks": [], "errors": []}
        except RetryError as e:
            msg = f"Indexing failed after retries: {e}"
            state.errors.append(msg)
            logger.error(msg)
            return {"errors": state.errors}
        except Exception as e:
            msg = f"Indexing error: {e}"
            state.errors.append(msg)
            logger.error(msg, exc_info=True)
            return {"errors": state.errors}

    # ----------------- Search -----------------
    async def search(self, query: str, k: int = 3):
        return await asyncio.to_thread(self._vectorstore.similarity_search, query, k)


In [56]:
import asyncio
import logging
from langchain_core.documents import Document

# Import from your module
# from vector_store_node import VectorStoreNode, VectorStoreConfig, KnowledgeState

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


async def test_chroma_with_cache():
    logger.info("=" * 50)
    logger.info("Starting test scenario: ChromaDB with InMemory Cache")
    logger.info("=" * 50)

    # --- Config ---
    config = VectorStoreConfig(
        collection_name="test_collection",
        chroma_persist_directory="./chroma_db_test",
        embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
        embedding_model_provider="huggingface"
    )

    # --- Init Node ---
    try:
        node = VectorStoreNode(config)
        logger.info("VectorStoreNode initialized with Chroma.")
    except Exception as e:
        logger.error(f"Failed to initialize Chroma node: {e}")
        return

    # --- Prepare State ---
    state = KnowledgeState(
        main_query="Test query for indexing", # Add main_query
        messages=[], # Add messages
        cleaned_chunks=[
            Document(page_content="Chroma is a vector database for embeddings.", metadata={"source": "test1"}),
            Document(page_content="It supports persistence and fast similarity search.", metadata={"source": "test2"}),
        ]
    )

    # --- Index Docs ---
    result = await node.index_documents(state)
    logger.info(f"Indexing result: {result}")

    # --- Search ---
    try:
        query = "What is Chroma?"
        docs = await node.search(query, k=2)
        if docs:
            logger.info(f"✅ Success: Document found for query '{query}'")
            for d in docs:
                logger.info(f"- {d.page_content} (meta={d.metadata})")
        else:
            logger.warning(f"❌ No docs found for query '{query}'")
    except Exception as e:
        logger.error(f"Search failed: {e}")


if __name__ == "__main__":
    asyncio.run(test_chroma_with_cache())

INFO:__main__:==================================================
INFO:__main__:Starting test scenario: ChromaDB with InMemory Cache
INFO:__main__:==================================================
INFO:__main__:Chroma initialized at ./chroma_db_test
INFO:__main__:VectorStoreNode initialized with Chroma.
INFO:__main__:Indexed 2 docs in 0.18s
INFO:__main__:Indexing result: {'cleaned_chunks': [], 'errors': []}
INFO:__main__:✅ Success: Document found for query 'What is Chroma?'
INFO:__main__:- Chroma is a vector database for embeddings. (meta={'source': 'test1'})
INFO:__main__:- Chroma is a vector database for embeddings. (meta={'source': 'test1'})


In [ ]:
# nodes.py

import logging
from typing import Dict, Any, List
import asyncio
import time
import hashlib
from langchain_core.documents import Document
from pydantic import ValidationError

# from KnowledgeState import KnowledgeState, CrawlStatus, SubQuery, convert_langchain_doc_to_document_result, DocumentSource, ContentSource
# from tools import SearchEngine, CrawlerEngine, CleanerEngine, VectorStoreNode
# from config import GlobalConfig
from datetime import datetime, timezone

logger = logging.getLogger(__name__)

# --- Helper function for LLM calls with retry logic ---

async def _call_llm(llm: Any, prompt: str) -> str:
    """Helper to call the LLM and extract the response with retry logic."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Assuming an async-capable LLM client
            response = await llm.ainvoke(prompt)
            if hasattr(response, 'content'):
                return response.content
            return str(response)
        except Exception as e:
            logger.warning(f"LLM call attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                raise Exception(f"All LLM call attempts failed: {e}")
            await asyncio.sleep(1)

# ---- Node functions for LangGraph ----

async def kb_lookup_node(state: KnowledgeState, vectorstore_node: VectorStoreNode) -> KnowledgeState:
    """LangGraph node for KB lookup."""
    try:
        results_with_scores = await asyncio.to_thread(
            vectorstore_node._vectorstore.similarity_search_with_score,
            query=state.main_query,
            k=6
        )

        state.knowledge_base_results.clear()

        for doc, score in results_with_scores:
            doc_result = convert_langchain_doc_to_document_result(
                doc,
                source_type=DocumentSource.KNOWLEDGE_BASE,
                relevance_score=1.0 - score
            )
            state.knowledge_base_results.append(doc_result)

        if results_with_scores:
            avg_score = sum(score for _, score in results_with_scores) / len(results_with_scores)
            confidence = max(0.0, min(1.0, 1.0 - (avg_score / 2.0)))
            state.state_metrics["kb_confidence"] = confidence
        else:
            state.state_metrics["kb_confidence"] = 0.0

        logger.info(f"KB lookup complete. Found {len(results_with_scores)} docs, confidence: {state.state_metrics.get('kb_confidence', 0.0):.2f}")

    except Exception as e:
        logger.error(f"KB lookup failed: {e}")
        state.add_error(f"KB lookup failed: {str(e)}")
        state.state_metrics["kb_confidence"] = 0.0

    return state


async def generate_subqueries_node(state: KnowledgeState, llm: Any) -> KnowledgeState:
    """LangGraph node to generate subqueries."""
    try:
        prompt = f"""
        Given the main query: "{state.main_query}"
        Generate 3-4 focused sub-queries that would help find comprehensive knowledge.
        Consider different aspects like:
        - Technical details and implementation
        - Recent developments and trends
        - Practical applications and examples
        - Comparative analysis and alternatives
        Format your response as a list of queries, one per line:
        - [subquery 1]
        - [subquery 2]
        - [subquery 3]
        - [subquery 4]
        """
        response_content = await _call_llm(llm, prompt)

        lines = [line.strip() for line in response_content.split('\n') if line.strip().startswith('-')]
        subqueries_texts = [line[1:].strip() for line in lines if len(line) > 1]

        subqueries = []
        for i, sq in enumerate(subqueries_texts):
            if sq:
                priority = 0.9 if i == 0 else (0.8 if i == 1 else 0.7)
                deep = i < 2
                query_type = ("technical" if "technical" in sq.lower() or "implementation" in sq.lower() else
                              "trends" if "recent" in sq.lower() or "developments" in sq.lower() else "general")
                subqueries.append(SubQuery(query=sq.strip(), priority=priority, deep=deep, query_type=query_type))

        if subqueries:
            state.add_subqueries(subqueries)
            return state

        logger.info("Generated no new subqueries.")

    except Exception as e:
        logger.error(f"Failed to generate subqueries: {e}")
        state.add_error(f"Subquery generation failed: {str(e)}")

    return state


async def external_search_node(state: KnowledgeState, search_engine: SearchEngine) -> KnowledgeState:
    """LangGraph node for external search."""
    try:
        await search_engine.run_searches(queries=[state.main_query], max_results_per_query=8)
        logger.info("External search tool invoked successfully.")
    except Exception as e:
        logger.error(f"External search failed: {e}")
        state.add_error(f"External search failed: {str(e)}")
    return state


async def extract_and_rank_urls_node(state: KnowledgeState, url_analyzer: URLAnalyzer) -> KnowledgeState:
    """LangGraph node to extract and rank URLs."""
    try:
        search_results_dicts = [
            {'url': r.metadata.get('url', ''), 'snippet': r.page_content}
            for r in state.external_api_results
        ]

        await url_analyzer.analyze_batch(search_results=search_results_dicts, query=state.main_query)
        logger.info("URL analyzer tool invoked successfully.")

    except Exception as e:
        logger.error(f"URL extraction failed: {e}")
        state.add_error(f"URL extraction failed: {str(e)}")

    return state


async def web_crawling_node(state: KnowledgeState, crawler_engine: CrawlerEngine) -> KnowledgeState:
    """LangGraph node for web crawling."""
    try:
        if not state.crawl_queue:
            logger.info("No URLs in crawl queue. Skipping crawling.")
            return {"crawl_status": CrawlStatus.NEUTRAL}

        state.crawl_status = CrawlStatus.IN_PROGRESS

        max_crawls = crawler_engine.config.max_concurrent
        urls_to_crawl = state.crawl_queue[:max_crawls]

        await crawler_engine.crawl_urls_batch(urls_to_crawl)

        state.crawl_queue = state.crawl_queue[max_crawls:]

        if state.metadata.crawl_stats.successful > 0:
            state.crawl_status = CrawlStatus.SUCCESS
        else:
            state.crawl_status = CrawlStatus.FAILED

        logger.info(f"Crawling complete. Status: {state.crawl_status}")

    except Exception as e:
        logger.error(f"Web crawling failed: {e}")
        state.add_error(f"Web crawling failed: {str(e)}")
        state.crawl_status = CrawlStatus.FAILED

    return state


async def assess_content_quality_node(state: KnowledgeState) -> KnowledgeState:
    """LangGraph node for assessing content quality."""
    try:
        quality_score = state.calculate_weighted_quality_score()
        state.state_metrics["quality_score"] = quality_score

        gaps = []
        if len(state.knowledge_base_results) + len(state.crawler_results) < 2:
            gaps.append("Insufficient knowledge base or crawled results")
        if state.state_metrics.get("kb_confidence", 0.0) < 0.5:
            gaps.append("Low knowledge base confidence")
        if quality_score < 0.3:
            gaps.append("Overall low content quality")

        state.gaps = gaps

        logger.info(f"Content quality assessment: score={quality_score:.2f}, gaps={len(gaps)}")

    except Exception as e:
        logger.error(f"Content quality assessment failed: {e}")
        state.add_error(f"Content quality assessment failed: {str(e)}")

    return state


async def subquery_search_node(state: KnowledgeState, search_engine: SearchEngine) -> KnowledgeState:
    """LangGraph node for targeted search using subqueries."""
    state = state.ensure
    try:
        subqueries_to_search = [sq.query for sq in state.sub_queries_to_generate][:3]

        if not subqueries_to_search:
            logger.info("No subqueries available for search. Skipping.")
            return {}

        await search_engine.run_searches(queries=subqueries_to_search, max_results_per_query=5)

        # Move subqueries from `to_generate` to `processed`
        processed_subqueries = [sq.query for sq in state.sub_queries_to_generate[:3]]
        state.processed_queries.extend(processed_subqueries)
        state.sub_queries_to_generate = state.sub_queries_to_generate[3:]

        logger.info(f"Subquery search invoked for: {processed_subqueries}")

    except Exception as e:
        logger.error(f"Subquery search failed: {e}")
        state.add_error(f"Subquery search failed: {str(e)}")

    return state


async def process_documents_node(state: KnowledgeState, cleaner_engine: CleanerEngine) -> KnowledgeState:
    """LangGraph node for cleaning and chunking documents."""
    state = KnowledgeState.create_knowledge_state_from_dict(state)
    try:
        all_docs = []
        all_docs.extend(state.knowledge_base_results)
        all_docs.extend(state.crawler_results)
        all_docs.extend(state.external_api_results)

        if not all_docs:
            logger.info("No documents to process. Skipping.")
            return state

        # Call the refactored cleaner tool
        await cleaner_engine.clean_and_chunk_documents(all_docs)

        logger.info(f"Document processing complete. Chunks ready.")

    except Exception as e:
        logger.error(f"Document processing failed: {e}")
        state.add_error(f"Document processing failed: {str(e)}")

    return state


async def finalize_results_node(state: KnowledgeState) -> KnowledgeState:
    """LangGraph node for finalizing and sorting results."""
    try:
        all_results = state.knowledge_base_results + state.cleaned_chunks

        if not all_results:
            logger.info("No documents to finalize.")
            return state

        seen_hashes = set()
        final_docs = []
        for doc_result in all_results:
            if doc_result.page_content:
                content_hash = hashlib.sha256(doc_result.page_content.encode()).hexdigest()
                if content_hash not in seen_hashes:
                    seen_hashes.add(content_hash)
                    final_docs.append(doc_result)

        def sort_key(doc_result):
            source_priority = {
                DocumentSource.KNOWLEDGE_BASE: 4,
                DocumentSource.EXTERNAL_API: 3,
                DocumentSource.WEB_CRAWL: 2,
                DocumentSource.SEARCH_SNIPPET: 1
            }.get(doc_result.source_type, 1)

            content_boost = {
                ContentSource.ACADEMIC: 1.3,
                ContentSource.GOVERNMENT: 1.2,
                ContentSource.WIKI: 1.1,
                ContentSource.OTHER: 1.0
            }.get(doc_result.content_source, 1.0) if doc_result.content_source else 1.0

            relevance = doc_result.relevance_score or 0.5
            length_factor = min(1.0, (doc_result.content_length or 0) / 1000)

            return state

        final_docs.sort(key=sort_key, reverse=True)

        max_final_docs = state.state_metrics.get('max_final_documents', 20)
        state.cleaned_chunks = final_docs[:max_final_docs]
        state.metadata.total_iterations += 1

        final_quality = state.calculate_weighted_quality_score()
        state.state_metrics["final_quality_score"] = final_quality

        state.metadata.final = True

        logger.info(f"Finalization complete. {len(state.cleaned_chunks)} final documents.")

    except Exception as e:
        logger.error(f"Finalization failed: {e}")
        state.add_error(f"Finalization failed: {str(e)}")

    return state


async def error_handler_node(state: KnowledgeState) -> KnowledgeState:
    """
    Handles errors recorded in the state.
    """
    if state.errors:
        logger.error(f"Graph encountered errors: {state.errors}")
    return state


In [ ]:
# decisions.py

import logging
from typing import Dict, Any
# from KnowledgeState import KnowledgeState, CrawlStatus
# from config import GlobalConfig

logger = logging.getLogger(__name__)

# --- Decision functions for LangGraph conditional edges ---

def should_search_external_node(state: KnowledgeState, config: GlobalConfig) -> str:
    """Decision logic based on KB confidence, using GlobalConfig."""
    try:
        confidence_threshold = config.decisions.confidence_threshold
        kb_confidence = state.state_metrics.get("kb_confidence", 0.0)

        if kb_confidence >= confidence_threshold:
            return "finalize"
        else:
            return "external_search"
    except Exception as e:
        logger.error(f"External search decision failed: {e}")
        # Transition to error handler on failure
        state.add_error(f"External search decision failed: {e}")
        return "error_handler"

def should_crawl_urls_node(state: KnowledgeState, config: GlobalConfig) -> str:
    """Decision logic for web crawling, using GlobalConfig."""
    try:
        if not state.should_continue_crawling:
            return "assess_quality"

        crawl_stats = state.metadata.crawl_stats
        if crawl_stats.attempted > 0 and state.success_rate < config.decisions.low_success_rate_threshold:
            logger.info("Skipping crawling due to low success rate.")
            return "assess_quality"

        if state.crawl_queue:
            return "web_crawling"
        else:
            return "assess_quality"

    except Exception as e:
        logger.error(f"Crawl decision failed: {e}")
        # Transition to error handler on failure
        state.add_error(f"Crawl decision failed: {e}")
        return "error_handler"

def should_do_subquery_search_node(state: KnowledgeState, config: GlobalConfig) -> str:
    """Decision logic for subquery search, using GlobalConfig."""
    try:
        quality_threshold = config.decisions.quality_threshold
        max_iterations = config.decisions.max_iterations

        if state.has_sufficient_content and state.calculate_weighted_quality_score() >= quality_threshold:
            return "process_documents"

        if state.metadata.total_iterations >= max_iterations:
            return "process_documents"

        if state.sub_queries_to_generate:
            return "subquery_search"
        else:
            return "process_documents"

    except Exception as e:
        logger.error(f"Subquery search decision failed: {e}")
        # Transition to error handler on failure
        state.add_error(f"Subquery search decision failed: {e}")
        return "error_handler"


In [ ]:
# workflow.py

!pip install langgraph --quiet

import asyncio
import logging
from typing import Dict, Any
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# from nodes import (
#     kb_lookup_node,
#     generate_subqueries_node,
#     external_search_node,
#     extract_and_rank_urls_node,
#     web_crawling_node,
#     assess_content_quality_node,
#     subquery_search_node,
#     process_documents_node,
#     finalize_results_node,
#     error_handler_node
# )
# from KnowledgeState import LangGraphKnowledgeState
# from decisions import (
#     should_search_external_node,
#     should_crawl_urls_node,
#     should_do_subquery_search_node
# )
# from config import GlobalConfig

logger = logging.getLogger(__name__)

def create_knowledge_graph(tools_and_configs: Dict[str, Any]) -> StateGraph:
    """
    Creates and compiles the LangGraph for knowledge acquisition.
    Uses the modern TypedDict-based state for compatibility.
    """
    try:
        workflow = StateGraph(KnowledgeState)
        config: GlobalConfig = tools_and_configs["config"]
        # Add all nodes
        workflow.add_node("kb_lookup", lambda state: asyncio.run(kb_lookup_node(state, tools_and_configs['vectorstore_node'])))
        workflow.add_node("generate_subqueries", lambda state: asyncio.run(generate_subqueries_node(state, tools_and_configs['llm'])))
        workflow.add_node("external_search", lambda state: asyncio.run(external_search_node(state, tools_and_configs['search_engine'])))
        workflow.add_node("extract_urls", lambda state: asyncio.run(extract_and_rank_urls_node(state, tools_and_configs['url_analyzer'])))
        workflow.add_node("web_crawling", lambda state: asyncio.run(web_crawling_node(state, tools_and_configs['crawler_engine'])))
        workflow.add_node("assess_quality", lambda state: asyncio.run(assess_content_quality_node))
        workflow.add_node("subquery_search", lambda state: asyncio.run(subquery_search_node(state, tools_and_configs['search_engine'])))
        workflow.add_node("process_documents", lambda state: asyncio.run(process_documents_node(state, tools_and_configs['cleaner_engine'])))
        workflow.add_node("finalize", lambda state: asyncio.run(finalize_results_node))
        workflow.add_node("error_handler", lambda state: asyncio.run(error_handler_node))

        # Define the graph flow
        workflow.set_entry_point("kb_lookup")

        # Decision point after KB lookup
        workflow.add_conditional_edges(
            "kb_lookup",
            lambda state: should_search_external_node(state, config),
            {
                "external_search": "external_search",
                "finalize": "finalize"
            }
        )

        # Subquery generation and external search
        workflow.add_edge("generate_subqueries", "external_search")
        workflow.add_edge("external_search", "extract_urls")

        # Crawling decision
        workflow.add_conditional_edges(
            "extract_urls",
            lambda state: should_crawl_urls_node(state, config),
            {
                "web_crawling": "web_crawling",
                "assess_quality": "assess_quality"
            }
        )

        workflow.add_edge("web_crawling", "assess_quality")

        # Quality-based decisions
        workflow.add_conditional_edges(
            "assess_quality",
            lambda state: should_do_subquery_search_node(state, config),
            {
                "subquery_search": "subquery_search",
                "process_documents": "process_documents"
            }
        )

        # Subquery search loops back to URL extraction
        workflow.add_edge("subquery_search", "extract_urls")

        # Final processing chain
        workflow.add_edge("process_documents", "finalize")
        workflow.add_edge("finalize", END)

        # Add error transitions (fallback to the error handler)
        workflow.add_edge("kb_lookup", "error_handler")
        workflow.add_edge("generate_subqueries", "error_handler")
        workflow.add_edge("external_search", "error_handler")
        workflow.add_edge("extract_urls", "error_handler")
        workflow.add_edge("web_crawling", "error_handler")
        workflow.add_edge("assess_quality", "error_handler")
        workflow.add_edge("subquery_search", "error_handler")
        workflow.add_edge("process_documents", "error_handler")
        workflow.add_edge("finalize", "error_handler")
        workflow.add_edge("error_handler", END)

        return workflow

    except Exception as e:
        logger.error(f"Graph creation failed: {e}")
        raise


In [ ]:
# # nodes.py

# import logging
# from typing import Dict, Any, List
# import asyncio
# from KnowledgeState import KnowledgeState, CrawlStatus, SubQuery, convert_langchain_doc_to_document_result, DocumentSource, ContentSource
# from tools import SearchEngine, CrawlerEngine, CleanerEngine, VectorStoreNode
# from config import GlobalConfig
# from datetime import datetime, timezone
# import time
# from langchain_core.documents import Document
# from pydantic import ValidationError

# logger = logging.getLogger(__name__)

# # --- Helper function for LLM calls with retry logic ---

# async def _call_llm(llm: Any, prompt: str) -> str:
#     """Helper to call the LLM and extract the response with retry logic."""
#     max_retries = 3
#     for attempt in range(max_retries):
#         try:
#             # Assuming an async-capable LLM client
#             response = await llm.ainvoke(prompt)
#             if hasattr(response, 'content'):
#                 return response.content
#             return str(response)
#         except Exception as e:
#             logger.warning(f"LLM call attempt {attempt + 1} failed: {e}")
#             if attempt == max_retries - 1:
#                 raise Exception(f"All LLM call attempts failed: {e}")
#             await asyncio.sleep(1)

# # ---- Node functions for LangGraph ----

# async def kb_lookup_node(state: KnowledgeState, vectorstore_node: VectorStoreNode) -> Dict[str, Any]:
#     """LangGraph node for KB lookup."""
#     try:
#         results_with_scores = await asyncio.to_thread(
#             vectorstore_node._vectorstore.similarity_search_with_score,
#             query=state.main_query,
#             k=6
#         )

#         state.knowledge_base_results.clear()

#         for doc, score in results_with_scores:
#             doc_result = convert_langchain_doc_to_document_result(
#                 doc,
#                 source_type=DocumentSource.KNOWLEDGE_BASE,
#                 relevance_score=1.0 - score
#             )
#             state.knowledge_base_results.append(doc_result)

#         if results_with_scores:
#             avg_score = sum(score for _, score in results_with_scores) / len(results_with_scores)
#             confidence = max(0.0, min(1.0, 1.0 - (avg_score / 2.0)))
#             state.state_metrics["kb_confidence"] = confidence
#         else:
#             state.state_metrics["kb_confidence"] = 0.0

#         logger.info(f"KB lookup complete. Found {len(results_with_scores)} docs, confidence: {state.state_metrics.get('kb_confidence', 0.0):.2f}")

#     except Exception as e:
#         logger.error(f"KB lookup failed: {e}")
#         state.add_error(f"KB lookup failed: {str(e)}")
#         state.state_metrics["kb_confidence"] = 0.0

#     return {"knowledge_base_results": state.knowledge_base_results, "state_metrics": state.state_metrics, "errors": state.errors}


# async def generate_subqueries_node(state: KnowledgeState, llm: Any) -> Dict[str, Any]:
#     """LangGraph node to generate subqueries."""
#     try:
#         prompt = f"""
#         Given the main query: "{state.main_query}"
#         Generate 3-4 focused sub-queries that would help find comprehensive knowledge.
#         Consider different aspects like:
#         - Technical details and implementation
#         - Recent developments and trends
#         - Practical applications and examples
#         - Comparative analysis and alternatives
#         Format your response as a list of queries, one per line:
#         - [subquery 1]
#         - [subquery 2]
#         - [subquery 3]
#         - [subquery 4]
#         """
#         response_content = await _call_llm(llm, prompt)

#         lines = [line.strip() for line in response_content.split('\n') if line.strip().startswith('-')]
#         subqueries_texts = [line[1:].strip() for line in lines if len(line) > 1]

#         subqueries = []
#         for i, sq in enumerate(subqueries_texts):
#             if sq:
#                 priority = 0.9 if i == 0 else (0.8 if i == 1 else 0.7)
#                 deep = i < 2
#                 query_type = ("technical" if "technical" in sq.lower() or "implementation" in sq.lower() else
#                               "trends" if "recent" in sq.lower() or "developments" in sq.lower() else "general")
#                 subqueries.append(SubQuery(query=sq.strip(), priority=priority, deep=deep, query_type=query_type))

#         if subqueries:
#             return {"sub_queries_to_generate": subqueries}

#         logger.info("Generated no new subqueries.")

#     except Exception as e:
#         logger.error(f"Failed to generate subqueries: {e}")
#         state.add_error(f"Subquery generation failed: {str(e)}")

#     return {"sub_queries_to_generate": [], "errors": state.errors}


# async def external_search_node(state: KnowledgeState, search_engine: SearchEngine) -> Dict[str, Any]:
#     """LangGraph node for external search."""
#     try:
#         await search_engine.run_searches(queries=[state.main_query], max_results_per_query=8)
#         logger.info("External search tool invoked successfully.")
#     except Exception as e:
#         logger.error(f"External search failed: {e}")
#         state.add_error(f"External search failed: {str(e)}")
#     return {"errors": state.errors, "external_api_results": state.external_api_results}


# async def extract_and_rank_urls_node(state: KnowledgeState, url_analyzer: URLAnalyzer) -> Dict[str, Any]:
#     """LangGraph node to extract and rank URLs."""
#     try:
#         search_results_dicts = [
#             {'url': r.metadata.get('url', ''), 'snippet': r.page_content}
#             for r in state.external_api_results
#         ]

#         await url_analyzer.analyze_batch(search_results=search_results_dicts, query=state.main_query)
#         logger.info("URL analyzer tool invoked successfully.")

#     except Exception as e:
#         logger.error(f"URL extraction failed: {e}")
#         state.add_error(f"URL extraction failed: {str(e)}")

#     return {"errors": state.errors, "extracted_urls": state.extracted_urls, "crawl_queue": state.crawl_queue}


# async def web_crawling_node(state: KnowledgeState, crawler_engine: CrawlerEngine) -> Dict[str, Any]:
#     """LangGraph node for web crawling."""
#     try:
#         if not state.crawl_queue:
#             logger.info("No URLs in crawl queue. Skipping crawling.")
#             return {"crawl_status": CrawlStatus.NEUTRAL}

#         state.crawl_status = CrawlStatus.IN_PROGRESS

#         max_crawls = crawler_engine.config.max_concurrent
#         urls_to_crawl = state.crawl_queue[:max_crawls]

#         await crawler_engine.crawl_urls_batch(urls_to_crawl)

#         state.crawl_queue = state.crawl_queue[max_crawls:]

#         if state.metadata.crawl_stats.successful > 0:
#             state.crawl_status = CrawlStatus.SUCCESS
#         else:
#             state.crawl_status = CrawlStatus.FAILED

#         logger.info(f"Crawling complete. Status: {state.crawl_status}")

#     except Exception as e:
#         logger.error(f"Web crawling failed: {e}")
#         state.add_error(f"Web crawling failed: {str(e)}")
#         state.crawl_status = CrawlStatus.FAILED

#     return {"crawl_queue": state.crawl_queue, "crawl_status": state.crawl_status, "metadata": state.metadata, "errors": state.errors, "crawler_results": state.crawler_results}


# async def assess_content_quality_node(state: KnowledgeState) -> Dict[str, Any]:
#     """LangGraph node for assessing content quality."""
#     try:
#         quality_score = state.calculate_weighted_quality_score()
#         state.state_metrics["quality_score"] = quality_score

#         g


In [ ]:
# # workflow.py

# import logging
# from typing import Dict, Any
# from langgraph.graph import StateGraph, END
# from langgraph.checkpoint.memory import MemorySaver

# from nodes import (
#     kb_lookup_node,
#     generate_subqueries_node,
#     external_search_node,
#     extract_and_rank_urls_node,
#     web_crawling_node,
#     assess_content_quality_node,
#     subquery_search_node,
#     process_documents_node,
#     finalize_results_node,
#     error_handler_node
# )
# from KnowledgeState import LangGraphKnowledgeState
# from decisions import (
#     should_search_external_node,
#     should_crawl_urls_node,
#     should_do_subquery_search_node
# )
# from config import GlobalConfig

# logger = logging.getLogger(__name__)

# def create_knowledge_graph(tools_and_configs: Dict[str, Any]) -> StateGraph:
#     """
#     Creates and compiles the LangGraph for knowledge acquisition.
#     Uses the modern TypedDict-based state for compatibility.
#     """
#     try:
#         workflow = StateGraph(LangGraphKnowledgeState)
#         config: GlobalConfig = tools_and_configs["config"]

#         # Add all nodes
#         workflow.add_node("kb_lookup", lambda state: kb_lookup_node(state, tools_and_configs['vectorstore_node']))
#         workflow.add_node("generate_subqueries", lambda state: generate_subqueries_node(state, tools_and_configs['llm']))
#         workflow.add_node("external_search", lambda state: external_search_node(state, tools_and_configs['search_engine']))
#         workflow.add_node("extract_urls", lambda state: extract_and_rank_urls_node(state, tools_and_configs['url_analyzer']))
#         workflow.add_node("web_crawling", lambda state: web_crawling_node(state, tools_and_configs['crawler_engine']))
#         workflow.add_node("assess_quality", assess_content_quality_node)
#         workflow.add_node("subquery_search", lambda state: subquery_search_node(state, tools_and_configs['search_engine']))
#         workflow.add_node("process_documents", lambda state: process_documents_node(state, tools_and_configs['cleaner_engine']))
#         workflow.add_node("finalize", finalize_results_node)
#         workflow.add_node("error_handler", error_handler_node)

#         # Define the graph flow
#         workflow.set_entry_point("kb_lookup")

#         # Decision point after KB lookup
#         workflow.add_conditional_edges(
#             "kb_lookup",
#             lambda state: should_search_external_node(state, config),
#             {
#                 "external_search": "external_search",
#                 "finalize": "finalize"
#             }
#         )

#         # Subquery generation and external search
#         workflow.add_edge("generate_subqueries", "external_search")
#         workflow.add_edge("external_search", "extract_urls")

#         # Crawling decision
#         workflow.add_conditional_edges(
#             "extract_urls",
#             lambda state: should_crawl_urls_node(state, config),
#             {
#                 "web_crawling": "web_crawling",
#                 "assess_quality": "assess_quality"
#             }
#         )

#         workflow.add_edge("web_crawling", "assess_quality")

#         # Quality-based decisions
#         workflow.add_conditional_edges(
#             "assess_quality",
#             lambda state: should_do_subquery_search_node(state, config),
#             {
#                 "subquery_search": "subquery_search",
#                 "process_documents": "process_documents"
#             }
#         )

#         # Subquery search loops back to URL extraction
#         workflow.add_edge("subquery_search", "extract_urls")

#         # Final processing chain
#         workflow.add_edge("process_documents", "finalize")
#         workflow.add_edge("finalize", END)

#         # Add error transitions (fallback to the error handler)
#         workflow.add_edge("kb_lookup", "error_handler")
#         workflow.add_edge("generate_subqueries", "error_handler")
#         workflow.add_edge("external_search", "error_handler")
#         workflow.add_edge("extract_urls", "error_handler")
#         workflow.add_edge("web_crawling", "error_handler")
#         workflow.add_edge("assess_quality", "error_handler")
#         workflow.add_edge("subquery_search", "error_handler")
#         workflow.add_edge("process_documents", "error_handler")
#         workflow.add_edge("finalize", "error_handler")
#         workflow.add_edge("error_handler", END)

#         return workflow

#     except Exception as e:
#         logger.error(f"Graph creation failed: {e}")
#         raise


In [ ]:
# # decisions.py

# import logging
# from typing import Dict, Any
# from KnowledgeState import KnowledgeState, CrawlStatus
# from config import GlobalConfig

# logger = logging.getLogger(__name__)

# # --- Decision functions for LangGraph conditional edges ---

# def should_search_external_node(state: KnowledgeState, config: GlobalConfig) -> str:
#     """Decision logic based on KB confidence, using GlobalConfig."""
#     try:
#         confidence_threshold = config.decisions.confidence_threshold
#         kb_confidence = state.state_metrics.get("kb_confidence", 0.0)

#         if kb_confidence >= confidence_threshold:
#             return "finalize"
#         else:
#             return "external_search"
#     except Exception as e:
#         logger.error(f"External search decision failed: {e}")
#         state.add_error(f"External search decision failed: {e}")
#         return "error_handler"

# def should_crawl_urls_node(state: KnowledgeState, config: GlobalConfig) -> str:
#     """Decision logic for web crawling, using GlobalConfig."""
#     try:
#         if not state.should_continue_crawling:
#             return "assess_quality"

#         crawl_stats = state.metadata.crawl_stats
#         if crawl_stats.attempted > 0 and state.success_rate < config.decisions.low_success_rate_threshold:
#             logger.info("Skipping crawling due to low success rate.")
#             return "assess_quality"

#         if state.crawl_queue:
#             return "web_crawling"
#         else:
#             return "assess_quality"

#     except Exception as e:
#         logger.error(f"Crawl decision failed: {e}")
#         state.add_error(f"Crawl decision failed: {e}")
#         return "error_handler"

# def should_do_subquery_search_node(state: KnowledgeState, config: GlobalConfig) -> str:
#     """Decision logic for subquery search, using GlobalConfig."""
#     try:
#         quality_threshold = config.decisions.quality_threshold
#         max_iterations = config.decisions.max_iterations

#         if state.has_sufficient_content and state.calculate_weighted_quality_score() >= quality_threshold:
#             return "process_documents"

#         if state.metadata.total_iterations >= max_iterations:
#             return "process_documents"

#         if state.sub_queries_to_generate:
#             return "subquery_search"
#         else:
#             return "process_documents"

#     except Exception as e:
#         logger.error(f"Subquery search decision failed: {e}")
#         state.add_error(f"Subquery search decision failed: {e}")
#         return "error_handler"

!pip uninstall langchain-google-vertexai --quiet

In [62]:
# main.py

!pip install -U langchain-google-genai --quiet
!pip install sentence-transformers --quiet # Ensure embedding model is available

import logging
import asyncio
from typing import Optional, Dict, Any
import time

from langgraph.checkpoint.memory import MemorySaver
# from langchain_openai import ChatOpenAI # If using OpenAI
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.chat_models import init_chat_model

# Assume KnowledgeState, GlobalConfig, SearchEngine, CrawlerEngine, CleanerEngine, URLAnalyzer, VectorStoreNode are defined
# Assume create_knowledge_graph is defined
# Assume LLMRegistry is defined

logger = logging.getLogger(__name__)

async def process_query(query: str, global_config: GlobalConfig) -> KnowledgeState:
    """Enhanced query processing returning Pydantic state."""
    if not query or not query.strip():
        raise ValueError("Query cannot be empty")

    try:
        # Initialize LLMs using the global registry (assuming it's properly set up elsewhere)
        # If LLMRegistry is not used, you would initialize LLMs directly here:
        # llm = init_chat_model(...)
        # tool_llm = init_chat_model(...)
        # For now, we'll assume LLMRegistry is the way to go and pass the chat LLM to tools that need one.
        # Ensure LLMRegistry is initialized before this function is called if needed by tools.
        try:
             llm = LLMRegistry.get("chat")
             # tool_llm = LLMRegistry.get("tool") # Get tool LLM if needed by tools
        except ValueError:
             logger.warning("LLMRegistry not initialized. Initializing default chat model.")
             llm = init_chat_model(global_config.llm.chat_model_name, temperature=global_config.llm.temperature)
             # tool_llm = init_chat_model(global_config.llm.tool_model_name, temperature=global_config.llm.temperature) # Init tool LLM if needed
             # Note: If using LLMRegistry, it should be initialized once before calling process_query

        # Initialize Embedding Model (needed by CrawlerEngine and VectorStoreNode)
        # Use the model name from CrawlerConfig as it's explicitly for content scoring/embedding
        try:
            embedding_model = SentenceTransformerEmbeddings(model_name=global_config.crawler.embedding_model_name)
            logger.info(f"Initialized Embedding Model: {global_config.crawler.embedding_model_name}")
        except Exception as e:
             logger.error(f"Failed to initialize Embedding Model: {e}")
             embedding_model = None # Handle case where embedding model fails to init

        # Create initial state
        initial_state = KnowledgeState.create_knowledge_state(query.strip())

        # Initialize tools, passing the state, relevant configs, and dependencies (LLM, embedding model)
        # Assuming tool constructors are updated to accept these
        # Placeholder clients for SearchEngine (Tavily, DDGS/DDG Tool) - should be initialized based on config/env vars
        # For this test, we'll pass None and rely on the SearchEngine's __init__ to handle it
        # based on its config which should contain clients if available.
        # Let's create dummy clients or assume config handles it for now.
        # Based on SearchEngine definition (AKIslzdPORin), it takes config with clients.
        # Let's create dummy clients for the config for this test.
        class DummyTavily:
             async def ainvoke(self, *args, **kwargs):
                  logger.info("DummyTavily ainvoke called")
                  return [] # Return empty list
        class DummyDDGTool:
             async def ainvoke(self, *args, **kwargs):
                  logger.info("DummyDDGTool ainvoke called")
                  return "[]" # Return empty string representing empty list
        class DummySerper:
             def results(self, *args, **kwargs):
                  logger.info("DummySerper results called")
                  return {'organic': []} # Return empty organic results

        # Update SearchConfig in global_config with dummy clients for the test
        # In a real application, you would initialize these clients properly based on env vars
        global_config.search.tavily_client = DummyTavily() # Replace with actual client if API key available
        # DDGTool and Serper are initialized *within* SearchEngine based on config/env vars


        # Now initialize the tools, passing config and necessary dependencies
        url_analyzer = URLAnalyzer(state=initial_state, config=global_config.url_analyzer, llm=llm) # URL Analyzer needs LLM
        search_engine = SearchEngine(state=initial_state, config=global_config.search) # Search Engine uses its config for clients
        crawler_engine = CrawlerEngine(state=initial_state, config=global_config.crawler, embedding_model=embedding_model) # Crawler needs embedding model
        cleaner_engine = CleanerEngine(state=initial_state, config=global_config.cleaner, embedding_model=embedding_model) # Cleaner needs embedding model for semantic chunking
        # VectorStoreNode needs config and embedding model
        vectorstore_node = VectorStoreNode(config=global_config.vector_store, embedding_model=embedding_model)


        # Bundle tools and config for graph initialization
        tools_and_configs = {
            "llm": llm, # Pass the main chat LLM
            "config": global_config,
            "vectorstore_node": vectorstore_node,
            "url_analyzer": url_analyzer,
            "search_engine": search_engine,
            "crawler_engine": crawler_engine,
            "cleaner_engine": cleaner_engine
            # Add other tools here as they are created
        }

        # Create the graph
        workflow = create_knowledge_graph(tools_and_configs)

        # Compile with memory for state persistence
        memory = MemorySaver()
        compiled_workflow = workflow.compile(checkpointer=memory)

        # Run the workflow
        workflow_config = {"configurable": {"thread_id": f"knowledge_subgraph_{hash(query)}_{int(time.time())}"}}
        # The workflow operates on and returns the Pydantic KnowledgeState instance directly now
        final_state = await compiled_workflow.ainvoke(initial_state, workflow_config)

        logger.info(f"Knowledge acquisition completed for query: '{query}'")
        # The returned object is already the Pydantic state
        return final_state

    except Exception as e:
        logger.error(f"Processing query '{query}' failed: {e}")
        # Create an error state using the class method for consistency
        error_state = KnowledgeState.create_knowledge_state(main_query=query)
        error_state.add_error(f"Workflow execution failed: {str(e)}")
        return error_state


async def main():
    # Step 1: Load config
    # Create instances of the config classes
    search_config = SearchConfig() # Use defaults or load from file
    url_analyzer_config = URLAnalysisConfig()
    crawler_config = CrawlerConfig()
    cleaner_config = CleanerConfig()
    vector_store_config = VectorStoreConfig() # Ensure chroma_persist_directory is set
    llm_config = LLMConfig()
    decisions_config = DecisionsConfig()


    global_config = GlobalConfig(
        search=search_config,
        url_analyzer=url_analyzer_config,
        crawler=crawler_config,
        vector_store=vector_store_config,
        llm=llm_config,
        decisions=decisions_config
    )

    # Step 2: Initialize LLM Registry (if used)
    # Ensure you have GOOGLE_API_KEY or OPENAI_API_KEY set in your environment or Colab secrets
    try:
        LLMRegistry.init_from_config(global_config.llm)
        logger.info("LLMRegistry initialized.")
    except Exception as e:
        logger.error(f"Failed to initialize LLMRegistry: {e}")
        # Handle case where LLM initialization fails

    # Step 3: Run your agent workflow
    query = "What are the latest developments in large language models?"
    # Set up initial state explicitly
    # Use direct instantiation now that create_knowledge_state is added
    initial_state = KnowledgeState.create_knowledge_state(main_query=query)

    # Pass the initial state object to process_query
    final_state = await process_query(query, global_config)

    print("\n--- Final State Summary ---")
    print(f"Main Query: {final_state.main_query}")
    print(f"Total Errors: {len(final_state.errors)}")
    if final_state.errors:
        print("Errors:", final_state.errors)
    print(f"KB Results: {len(final_state.knowledge_base_results)} documents")
    print(f"Crawler Results: {len(final_state.crawler_results)} documents")
    print(f"External API Results (Snippets): {len(final_state.external_api_results)} documents")
    print(f"Final Cleaned Chunks: {len(final_state.cleaned_chunks)} documents")
    print(f"Remaining Crawl Queue: {len(final_state.crawl_queue)} URLs")
    print(f"Crawl Stats - Attempted: {final_state.metadata.crawl_stats.attempted}, Successful: {final_state.metadata.crawl_stats.successful}, Failed: {final_state.metadata.crawl_stats.failed}")
    print(f"Final Quality Score: {final_state.state_metrics.get('final_quality_score', 'N/A'):.2f}")


if __name__ == "__main__":
    # Clear previous Chroma DB data for a clean run if needed
    # import shutil
    # if os.path.exists(global_config.vector_store.chroma_persist_directory):
    #     shutil.rmtree(global_config.vector_store.chroma_persist_directory)

    # Ensure asyncio is run correctly
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        print("\nWorkflow interrupted.")
    except Exception as e:
        logger.critical(f"An unexpected error occurred during main execution: {e}")

ERROR:__main__:Failed to initialize LLMRegistry: Unable to import langchain_openai. Please install with `pip install -U langchain-openai`
ERROR:__main__:Processing query 'What are the latest developments in large language models?' failed: Unable to import langchain_openai. Please install with `pip install -U langchain-openai`
CRITICAL:__main__:An unexpected error occurred during main execution: Unknown format code 'f' for object of type 'str'



--- Final State Summary ---
Main Query: What are the latest developments in large language models?
Total Errors: 1
Errors: ['[2025-09-07T06:40:38.737225+00:00] Workflow execution failed: Unable to import langchain_openai. Please install with `pip install -U langchain-openai`']
KB Results: 0 documents
Crawler Results: 0 documents
External API Results (Snippets): 0 documents
Final Cleaned Chunks: 0 documents
Remaining Crawl Queue: 0 URLs
Crawl Stats - Attempted: 0, Successful: 0, Failed: 0


In [ ]:
# agent_state.py

from typing import TypedDict, Dict, Any, List, Optional, Literal
from langchain_core.messages import BaseMessage
from KnowledgeState import KnowledgeState  # Import your refactored KnowledgeState
import operator
from typing_extensions import Annotated

# --- Define Pydantic models for specific tool outputs (or simple TypedDicts) ---
# Use Pydantic for more complex schemas
from pydantic import BaseModel, Field

class CodeExecutionResult(BaseModel):
    code: str
    output: Optional[str]
    error: Optional[str]

class CodeProjectState(BaseModel):
    project_goal: str
    language: Optional[str]
    framework: Optional[str]
    architecture: Optional[str]
    code_files: Dict[str, str] = Field(default_factory=dict)
    test_results: List[Dict[str, Any]] = Field(default_factory=list)
    reflection_log: List[str] = Field(default_factory=list)
    step_back_reasoning: List[str] = Field(default_factory=list)

class FileOperationResult(BaseModel):
    operation: str
    filename: str
    success: bool
    details: Optional[str]

class BashOperationResult(BaseModel):
    command: str
    output: Optional[str]
    error: Optional[str]

class FinancialTradeResult(BaseModel):
    action: str
    ticker: str
    amount: float
    status: str

class Plan(BaseModel):
    goal: str
    steps: List[str]

class Strategy(BaseModel):
    advice: str
    confidence: float
    source: str

class ReasoningOutput(BaseModel):
    process: str
    conclusion: str

class CorrectiveRAG(BaseModel):
    original_answer: str
    corrected_answer: str
    reasoning: str

class CodePlan(BaseModel):
    """A detailed plan for generating code from a research paper."""
    architecture_summary: str = Field(description="High-level summary of the system architecture.")
    implementation_plan: List[str] = Field(description="Step-by-step plan for implementation.")
    required_packages: List[str] = Field(description="List of required Python packages.")

class CodeResponse(BaseModel):
    preamble: str = Field(description="A preamble describing the code solution.")
    import_block: str = Field(description="The import statements for the code.")
    code: str = Field(description="The main body of the code.")


class AgentState(TypedDict):
    """
    The central state for the agent's graph, using Pydantic models for structured data.
    """
    messages: Annotated[List[BaseMessage], operator.add]
    # Nested KnowledgeState
    knowledge_state: KnowledgeState
    # KnowledgeState as a tool output
    knowledge_results: Optional[KnowledgeState]
    # Tool outputs (using Annotated for list merging)
    code_results: Annotated[List[CodeExecutionResult], operator.add]
    file_results: Annotated[List[FileOperationResult], operator.add]
    bash_results: Annotated[List[BashOperationResult], operator.add]
    financial_trades: Annotated[List[FinancialTradeResult], operator.add]
    coding_project: Optional[CodeProjectState]
    current_task: Optional[str]
    # Current plan, advice, etc. (these will be overwritten, so no operator.add)
    current_plan: Optional[Plan]
    advice: Optional[Strategy]
    reasoning: Optional[ReasoningOutput]
    rag_correction: Optional[CorrectiveRAG]
    # General agent status
    error: Optional[str]
    final_answer: Optional[str]

# Alias for LangGraph compatibility
class LangGraphAgentState(AgentState):
    # ... (existing fields) ...
    pass


In [ ]:
async def interpret_paper_node(state: LangGraphAgentState, llm: ChatOpenAI) -> dict:
    """
    Interprets a research paper and generates a high-level code plan.
    This uses the Nested KnowledgeState for context.
    """
    knowledge_state = state.get("knowledge_state")
    paper_content = " ".join([res.page_content for res in knowledge_state.crawler_results + knowledge_state.knowledge_base_results])

    prompt = PromptTemplate.from_template("""
    You are an expert software engineer. Given the following research paper content,
    provide a high-level architecture summary, a step-by-step implementation plan,
    and a list of required Python packages.

    Research Paper Content:
    {paper_content}
    """)

    chain = prompt | llm.with_structured_output(CodePlan)
    plan = await chain.ainvoke({"paper_content": paper_content})

    return {"code_generation_plan": plan.dict()}


async def generate_code_node(state: LangGraphAgentState, llm: ChatOpenAI) -> dict:
    """Generates code based on the plan and the research paper content."""
    knowledge_state = state.get("knowledge_state")
    plan = state.get("code_generation_plan")
    paper_content = " ".join([res.page_content for res in knowledge_state.crawler_results + knowledge_state.knowledge_base_results])

    prompt = PromptTemplate.from_template("""
    You are an expert Python developer. Given the following research paper content and
    implementation plan, generate the complete, production-ready, and executable code.

    Implementation Plan:
    {plan}

    Research Paper Content:
    {paper_content}
    """)

    chain = prompt | llm.with_structured_output(CodeResponse)
    code_response = await chain.ainvoke({"plan": plan, "paper_content": paper_content})

    return {"code_response": code_response.dict()}


async def validate_code_node(state: LangGraphAgentState, executor_tool) -> dict:
    """
    Validates the generated code by executing it.
    This node would call an external code execution tool.
    """
    code_response = state.get("code_response")
    full_code = f"{code_response.get('import_block')}\n\n{code_response.get('code')}"

    # Use the code_executor_tool (from your main agent's toolset)
    result = await executor_tool.invoke({"code": full_code})

    if result.get("error"):
        return {"code_validation_status": "failed", "code_execution_result": result}
    else:
        return {"code_validation_status": "success", "code_execution_result": result}

async def reflect_and_correct_node(state: LangGraphAgentState, llm: ChatOpenAI) -> dict:
    """Reflects on validation errors and generates a corrected version."""
    code_execution_result = state.get("code_execution_result")

    prompt = PromptTemplate.from_template("""
    The following code failed to execute. Analyze the error and provide a corrected version.

    Code:
    {code}

    Error:
    {error}
    """)

    chain = prompt | llm.with_structured_output(CodeResponse)
    corrected_code = await chain.ainvoke({"code": state.get("code_response"), "error": code_execution_result.get("error")})

    return {"code_response": corrected_code.dict()}

# --- Decision Nodes ---

def check_validation_status(state: LangGraphAgentState) -> str:
    """Conditional edge to check if code validation passed."""
    return "finalize" if state.get("code_validation_status") == "success" else "reflect_and_correct"

def check_reflection_limit(state: LangGraphAgentState) -> str:
    """Prevents infinite loops by limiting reflection attempts."""
    reflection_count = state.get("reflection_count", 0) + 1
    state["reflection_count"] = reflection_count
    return "validate" if reflection_count <= 3 else "fail"

def create_code_generation_subgraph(llm: ChatOpenAI, code_executor_tool: Any) -> StateGraph:
    """
    Creates the complete code generation subgraph.
    """
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("interpret", lambda state: interpret_paper_node(state, llm))
    graph.add_node("generate", lambda state: generate_code_node(state, llm))
    graph.add_node("validate", lambda state: validate_code_node(state, code_executor_tool))
    graph.add_node("reflect_and_correct", lambda state: reflect_and_correct_node(state, llm))

    graph.add_edge(START, "interpret")
    graph.add_edge("interpret", "generate")
    graph.add_edge("generate", "validate")
    graph.add_conditional_edges("validate", check_validation_status, {
        "finalize": END,
        "reflect_and_correct": "reflect_and_correct"
    })
    graph.add_edge("reflect_and_correct", "validate")

    return graph.compile()

@tool
def code_generator_tool(query: str) -> str:
    """Generates code from a research paper or a detailed description."""
    return f"CALL_SUBGRAPH:code_generation:{query}"


In [ ]:
def create_code_development_subgraph(llm: ChatOpenAI, shell_tool: ShellTool) -> StateGraph:
    """
    Sub-graph for planning, generating, and refining code.
    """
    graph = StateGraph(LangGraphAgentState)

    async def plan_and_research_node(state: LangGraphAgentState) -> dict:
        """Node for initial planning and research (using knowledge_acquisition_tool)."""
        prompt = PromptTemplate.from_template("""
        You are a seasoned full-stack developer. Given the user's request: "{request}",
        create a high-level development plan. Include the programming language,
        latest frameworks to use, a brief architecture, and any specific areas
        that require research to update your knowledge.

        User Request: {request}
        """)

        request = state['messages'][-1].content
        chain = prompt | llm.with_structured_output(JsonOutputParser)
        plan = await chain.ainvoke({"request": request})

        # Trigger knowledge acquisition for research topics
        state['current_task'] = "planning_and_research"
        return {"planning_node_output": plan}

    async def self_reflect_node(state: LangGraphAgentState) -> dict:
        """Node for self-reflection on the current code state."""
        prompt = PromptTemplate.from_template("""
        You have generated the following code:
        {code}

        Test results:
        {test_results}

        Reflect on the code and suggest improvements, adhering to best practices.
        """)

        # ... (implementation for reflection based on code and tests) ...
        return {"reflection_log": "Code reviewed, potential improvements identified."}

    async def step_back_node(state: LangGraphAgentState) -> dict:
        """Node for re-evaluating the approach using step-back reasoning."""
        prompt = PromptTemplate.from_template("""
        The current approach is not working. Re-evaluate the problem from a fundamental level.
        What is the core issue and how can we approach it differently?
        """)

        # ... (implementation for step-back reasoning) ...
        return {"step_back_reasoning": "Step-back reasoning complete, new approach considered."}

    async def execute_and_test_node(state: LangGraphAgentState, shell_tool: ShellTool) -> dict:
        """Node to execute and test the generated code."""
        # ... (implementation using shell_tool to run tests) ...
        # This will use the code_executor_subgraph logic
        return {"test_results": "Tests completed successfully."}

    graph.add_node("plan_and_research", lambda state: plan_and_research_node(state, llm))
    graph.add_node("generate_code", lambda state: generate_code_node(state, llm))
    graph.add_node("execute_and_test", lambda state: execute_and_test_node(state, shell_tool))
    graph.add_node("self_reflect", lambda state: self_reflect_node(state))
    graph.add_node("step_back", lambda state: step_back_node(state))

    graph.add_edge(START, "plan_and_research")
    graph.add_edge("plan_and_research", "generate_code")
    graph.add_edge("generate_code", "execute_and_test")
    graph.add_conditional_edges("execute_and_test", lambda state: "self_reflect" if "fail" in state["test_results"] else END)
    graph.add_edge("self_reflect", "generate_code")
    graph.add_edge("step_back", "plan_and_research") # Example of a step-back loop

    return graph.compile()

In [ ]:
# subgraphs.py

import logging
from langgraph.graph import StateGraph, END, START
from langchain_core.messages import HumanMessage, ToolMessage, BaseMessage
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain.chat_models import init_chat_model
from langchain.tools import ShellTool
from typing import Dict, Any, List

from agent_state import LangGraphAgentState, CodeProjectState

# Import the actual entry point for your knowledge acquisition graph
from workflow import create_knowledge_graph

logger = logging.getLogger(__name__)

# --- Simple Placeholder Nodes ---

async def _simple_passthrough_node(state: LangGraphAgentState) -> LangGraphAgentState:
    """A simple passthrough node for placeholder subgraphs."""
    # Simulate some work
    message_content = f"Executed placeholder node for {state['messages'][-1].content}"
    # Use operator.add behavior for messages
    new_messages = [HumanMessage(content=message_content)]
    state['messages'].extend(new_messages)
    return state

# --- Subgraph Creation Functions ---

def create_knowledge_subgraph(tools_and_configs: dict) -> StateGraph:
    """Creates the knowledge acquisition subgraph using your existing workflow."""
    # The actual implementation of create_knowledge_graph should be in workflow.py
    return create_knowledge_graph(tools_and_configs).compile()

def create_code_executor_subgraph() -> StateGraph:
    """Creates the code execution subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("code_node", _simple_passthrough_node)
    graph.add_edge("code_node", END)
    return graph.compile()

def create_bash_subgraph() -> StateGraph:
    """Creates the bash operation subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("bash_node", _simple_passthrough_node)
    graph.add_edge("bash_node", END)
    return graph.compile()

# --- Code Development Subgraph ---
def create_code_development_subgraph() -> StateGraph:
    """Creates the code development subgraph."""
    graph = StateGraph(LangGraphAgentState)

def create_file_operation_subgraph() -> StateGraph:
    """Creates the file operation subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("file_node", _simple_passthrough_node)
    graph.add_edge("file_node", END)
    return graph.compile()

def create_bash_subgraph() -> StateGraph:
    """Creates the bash operation subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("bash_node", _simple_passthrough_node)
    graph.add_edge("bash_node", END)
    return graph.compile()

def create_financial_trader_subgraph() -> StateGraph:
    """Creates the financial trader subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("trade", _simple_passthrough_node)
    graph.add_edge("trade", END)
    return graph.compile()

def create_planning_subgraph() -> StateGraph:
    """Creates the planning node subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("plan_node", _simple_passthrough_node)
    graph.add_edge("plan_node", END)
    return graph.compile()

def create_strategy_subgraph() -> StateGraph:
    """Creates the strategy advice subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("strategy_node", _simple_passthrough_node)
    graph.add_edge("strategy_node", END)
    return graph.compile()

def create_reasoning_subgraph() -> StateGraph:
    """Creates the reasoning node subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("reasoning_node", _simple_passthrough_node)
    graph.add_edge("reasoning_node", END)
    return graph.compile()

def create_corrective_rag_subgraph() -> StateGraph:
    """Creates the corrective RAG node subgraph."""
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("corrective_rag_node", _simple_passthrough_node)
    graph.add_edge("corrective_rag_node", END)
    return graph.compile()




In [ ]:
# subgraphs.py (additions)
# ... (existing imports and subgraphs) ...

import logging
from langgraph.graph import StateGraph, END, START
from langchain_core.messages import HumanMessage, BaseMessage
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from typing import Dict, Any, List

# --- Pydantic model for structured code output ---
from pydantic import BaseModel, Field

class CodePlan(BaseModel):
    """A detailed plan for generating code from a research paper."""
    architecture_summary: str = Field(description="High-level summary of the system architecture.")
    implementation_plan: List[str] = Field(description="Step-by-step plan for implementation.")
    required_packages: List[str] = Field(description="List of required Python packages.")

class CodeResponse(BaseModel):
    preamble: str = Field(description="A preamble describing the code solution.")
    import_block: str = Field(description="The import statements for the code.")
    code: str = Field(description="The main body of the code.")

# --- Code Generation Subgraph ---

async def interpret_paper_node(state: LangGraphAgentState, llm: ChatOpenAI) -> dict:
    """
    Interprets a research paper and generates a high-level code plan.
    This uses the Nested KnowledgeState for context.
    """
    knowledge_state = state.get("knowledge_state")
    paper_content = " ".join([res.page_content for res in knowledge_state.crawler_results + knowledge_state.knowledge_base_results])

    prompt = PromptTemplate.from_template("""
    You are an expert software engineer. Given the following research paper content,
    provide a high-level architecture summary, a step-by-step implementation plan,
    and a list of required Python packages.

    Research Paper Content:
    {paper_content}
    """)

    chain = prompt | llm.with_structured_output(CodePlan)
    plan = await chain.ainvoke({"paper_content": paper_content})

    return {"code_generation_plan": plan.dict()}

async def generate_code_node(state: LangGraphAgentState, llm: ChatOpenAI) -> dict:
    """Generates code based on the plan and the research paper content."""
    knowledge_state = state.get("knowledge_state")
    plan = state.get("code_generation_plan")
    paper_content = " ".join([res.page_content for res in knowledge_state.crawler_results + knowledge_state.knowledge_base_results])

    prompt = PromptTemplate.from_template("""
    You are an expert Python developer. Given the following research paper content and
    implementation plan, generate the complete, production-ready, and executable code.

    Implementation Plan:
    {plan}

    Research Paper Content:
    {paper_content}
    """)

    chain = prompt | llm.with_structured_output(CodeResponse)
    code_response = await chain.ainvoke({"plan": plan, "paper_content": paper_content})

    return {"code_response": code_response.dict()}

async def validate_code_node(state: LangGraphAgentState, executor_tool) -> dict:
    """
    Validates the generated code by executing it.
    This node would call an external code execution tool.
    """
    code_response = state.get("code_response")
    full_code = f"{code_response.get('import_block')}\n\n{code_response.get('code')}"

    # Use the code_executor_tool (from your main agent's toolset)
    result = await executor_tool.invoke({"code": full_code})

    if result.get("error"):
        return {"code_validation_status": "failed", "code_execution_result": result}
    else:
        return {"code_validation_status": "success", "code_execution_result": result}

async def reflect_and_correct_node(state: LangGraphAgentState, llm: ChatOpenAI) -> dict:
    """Reflects on validation errors and generates a corrected version."""
    code_execution_result = state.get("code_execution_result")

    prompt = PromptTemplate.from_template("""
    The following code failed to execute. Analyze the error and provide a corrected version.

    Code:
    {code}

    Error:
    {error}
    """)

    chain = prompt | llm.with_structured_output(CodeResponse)
    corrected_code = await chain.ainvoke({"code": state.get("code_response"), "error": code_execution_result.get("error")})

    return {"code_response": corrected_code.dict()}

# --- Decision Nodes ---

def check_validation_status(state: LangGraphAgentState) -> str:
    """Conditional edge to check if code validation passed."""
    return "finalize" if state.get("code_validation_status") == "success" else "reflect_and_correct"

def check_reflection_limit(state: LangGraphAgentState) -> str:
    """Prevents infinite loops by limiting reflection attempts."""
    reflection_count = state.get("reflection_count", 0) + 1
    state["reflection_count"] = reflection_count
    return "validate" if reflection_count <= 3 else "fail"

def create_code_generation_subgraph(llm: ChatOpenAI, code_executor_tool: Any) -> StateGraph:
    """
    Creates the complete code generation subgraph.
    """
    graph = StateGraph(LangGraphAgentState)
    graph.add_node("interpret", lambda state: interpret_paper_node(state, llm))
    graph.add_node("generate", lambda state: generate_code_node(state, llm))
    graph.add_node("validate", lambda state: validate_code_node(state, code_executor_tool))
    graph.add_node("reflect_and_correct", lambda state: reflect_and_correct_node(state, llm))

    graph.add_edge(START, "interpret")
    graph.add_edge("interpret", "generate")
    graph.add_edge("generate", "validate")
    graph.add_conditional_edges("validate", check_validation_status, {
        "finalize": END,
        "reflect_and_correct": "reflect_and_correct"
    })
    graph.add_edge("reflect_and_correct", "validate")

    return graph.compile()


In [ ]:
@tool
def code_generator_tool(query: str) -> str:
    """Generates code from a research paper or a detailed description."""
    return f"CALL_SUBGRAPH:code_generation:{query}"


In [ ]:
# agent_workflow.py
import operator
from typing import List, Dict, Any, Union
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, tool_executor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.agents import AgentAction, AgentFinish, AgentActionMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_community.tools import ShellTool # Use for code execution
from agent_state import LangGraphAgentState
from subgraphs import (
    create_knowledge_subgraph, create_code_executor_subgraph,
    create_file_operation_subgraph, create_bash_subgraph,
    create_financial_trader_subgraph, create_planning_subgraph,
    create_strategy_subgraph, create_reasoning_subgraph,
    create_corrective_rag_subgraph,
    create_code_development_subgraph
)
from config import GlobalConfig
shell_tool = ShellTool()

# --- Define Tools for the Main LLM Router ---

@tool
def knowledge_acquisition_tool(query: str) -> str:
    """Acquire comprehensive knowledge on a query."""
    return f"CALL_SUBGRAPH:knowledge_acquisition:{query}"

@tool
def code_executor_tool(code: str) -> str:
    """Execute code and get the result."""
    return f"CALL_SUBGRAPH:code_executor:{code}"

@tool
def file_operation_tool(operation: str, filename: str, content: str = "") -> str:
    """Perform file operations (read, write, delete)."""
    return f"CALL_SUBGRAPH:file_operation:{operation}:{filename}:{content}"

@tool
def bash_tool(command: str) -> str:
    """Execute a bash command."""
    return f"CALL_SUBGRAPH:bash:{command}"

@tool
def financial_trader_tool(query: str) -> str:
    """Execute a financial trading action based on a query."""
    return f"CALL_SUBGRAPH:financial_trader:{query}"

@tool
def planning_tool(query: str) -> str:
    """Create a step-by-step plan for a complex task."""
    return f"CALL_SUBGRAPH:planning:{query}"

@tool
def strategy_tool(query: str) -> str:
    """Provide strategic advice or analysis."""
    return f"CALL_SUBGRAPH:strategy:{query}"

@tool
def reasoning_tool(query: str) -> str:
    """Perform logical reasoning and inference."""
    return f"CALL_SUBGRAPH:reasoning:{query}"

@tool
def corrective_rag_tool(query: str) -> str:
    """Perform corrective retrieval-augmented generation."""
    return f"CALL_SUBGRAPH:corrective_rag:{query}"

@tool
def respond_tool(message: str) -> str:
    """Respond directly to the user."""
    return f"FINAL_RESPONSE:{message}"

@tool
def code_development_tool(query: str) -> str:
    """Generate, test, and debug code for a specific task."""
    return f"CALL_SUBGRAPH:code_development:{query}"

def create_unified_agent_workflow(llm: ChatOpenAI, all_tools: list, tools_and_configs: dict) -> StateGraph:
    """
    Creates the main agent graph with subgraphs as nodes.
    """
    # Compile the subgraphs
    subgraphs = {
        "knowledge_acquisition": create_knowledge_subgraph(tools_and_configs),
        "code_executor": create_code_executor_subgraph(),
        "file_operation": create_file_operation_subgraph(),
        "bash": create_bash_subgraph(),
        "financial_trader": create_financial_trader_subgraph(),
        "planning": create_planning_subgraph(),
        "strategy": create_strategy_subgraph(),
        "reasoning": create_reasoning_subgraph(),
        "corrective_rag": create_corrective_rag_subgraph(),
        "code_generation": create_code_generation_subgraph(llm, shell_tool),
        "code_development": create_code_development_subgraph(llm, shell_tool),
    }

    # Bind tools to the LLM
    llm_with_tools = llm.bind_tools(all_tools)

    workflow = StateGraph(LangGraphAgentState)

    async def route_agent_node(state: LangGraphAgentState) -> Dict[str, Any]:
        """The main agent reasoning node for routing."""
        # Use only the last message for routing to prevent context flooding
        messages = state['messages'][-1:]
        response = await llm_with_tools.ainvoke(messages)
        return {"messages": [response]}

    async def subgraph_executor_node(state: LangGraphAgentState) -> Dict[str, Any]:
        """Executes the subgraph selected by the router."""
        last_message = state['messages'][-1]

        # Extract subgraph call info from the dispatcher tool's output
        subgraph_call_string = last_message.tool_calls[0]['args']['query']

        if subgraph_call_string.startswith("CALL_SUBGRAPH:"):
            parts = subgraph_call_string.split(":", 2)
            subgraph_name = parts[1]
            # Assuming the query is the last part
            query = parts[2]

            subgraph = subgraphs.get(subgraph_name)
            if subgraph:
                # The subgraph will update the state directly
                return await subgraph.ainvoke(state)
            else:
                state['error'] = f"Unknown subgraph: {subgraph_name}"
                return state
        elif subgraph_call_string.startswith("FINAL_RESPONSE:"):
            response = subgraph_call_string.split(":", 1)[1]
            return {"final_answer": response}

        return state

    workflow.add_node("router", route_agent_node)
    workflow.add_node("executor", subgraph_executor_node)

    def route_to_executor(state: LangGraphAgentState) -> Union[str, END]:
        last_message = state['messages'][-1]
        if last_message.tool_calls:
            return "executor"
        else:
            return END

    workflow.add_conditional_edges(
        "router",
        route_to_executor,
        {"executor": "executor", END: END}
    )

    # Executor always returns to the router to check for follow-up actions
    workflow.add_edge("executor", "router")

    workflow.set_entry_point("router")

    return workflow.compile(checkpointer=MemorySaver())


In [ ]:
# cli.py
import asyncio
import logging
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

from agent_state import LangGraphAgentState, KnowledgeState
from agent_workflow import create_unified_agent_workflow
from config import GlobalConfig
from tools import (
    knowledge_acquisition_tool, code_executor_tool,
    file_operation_tool, bash_tool, financial_trader_tool,
    planning_tool, strategy_tool, reasoning_tool,
    corrective_rag_tool, respond_tool, code_development_tool
)


logging.basicConfig(level=logging.INFO)

async def main():
    """Main function for the agent CLI."""
    llm = ChatOpenAI(model="gpt-4o", temperature=0)

    # Configure tools and subgraphs
    global_config = GlobalConfig()

    tools = [
        knowledge_acquisition_tool, code_executor_tool,
        file_operation_tool, bash_tool, financial_trader_tool,
        planning_tool, strategy_tool, reasoning_tool,
        corrective_rag_tool, respond_tool, code_development_tool
    ]

    # Pass tools and configs to the workflow creator
    tools_and_configs = {"llm": llm, "config": global_config}
    agent_graph = create_unified_agent_workflow(llm, tools, tools_and_configs)

    while True:
        user_input = input("Enter your query (type 'exit' to quit): ")
        if user_input.lower() == 'exit':
            break

        # Create initial state
        initial_state = LangGraphAgentState(
            messages=[HumanMessage(content=user_input)],
            knowledge_state=KnowledgeState.(main_query=user_input),
            coding_project=None,
            code_results=[],
            file_results=[],
            bash_results=[],
            financial_trades=[],
            current_plan=None,
            advice=None,
            reasoning=None,
            rag_correction=None,
            error=None,
            final_answer=None
        )

        try:
            print("--- Invoking agent ---")
            final_state = await agent_graph.ainvoke(initial_state)
            print("--- Agent finished ---")
            print(f"Final response: {final_state.get('final_answer', 'No final answer generated.')}")

        except Exception as e:
            logging.error(f"An error occurred during agent execution: {e}")

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
import asyncio
import nest_asyncio
from unittest.mock import AsyncMock, MagicMock, patch
from langchain_core.documents import Document
import os
import shutil
# Assume KnowledgeState, VectorStoreConfig, VectorStoreNode, HuggingFaceEmbeddings are defined above
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
# Import the corrected VectorStoreNode (now only supports Chroma)
# from G2VgIWYSS1MF import VectorStoreNode # Assuming it's in G2VgIWYSS1MF
from chromadb import PersistentClient


nest_asyncio.apply()

# Define a temporary directory for ChromaDB persistence
TEST_CHROMA_DIR = "./test_vector_store_node_chroma_only_db"

# --- Cleanup function ---
def cleanup_test_db():
    if os.path.exists(TEST_CHROMA_DIR):
        shutil.rmtree(TEST_CHROMA_DIR)


# Cleanup before running the test
cleanup_test_db()

# 1. Create a mock KnowledgeState for the VectorStoreNode to interact with
mock_state = MagicMock(spec=KnowledgeState)
mock_state.add_error = MagicMock()
# Simulate cleaned_chunks being a list attribute
mock_state.cleaned_chunks = []

# 2. Create a VectorStoreConfig instance for ChromaDB test
config_chroma = VectorStoreConfig(
    vector_store_type="chroma", # Only Chroma supported now
    collection_name="test_collection_chroma_only", # Unique name
    chroma_persist_directory=TEST_CHROMA_DIR,
)
print("Vector Store Config (ChromaDB - Only):")
print(config_chroma.model_dump_json(indent=2))

# 3. Use the actual SentenceTransformerEmbeddings model
real_embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


# 4. Create a VectorStoreNode instance for ChromaDB test (using the updated class)
vector_store_node_chroma = VectorStoreNode(config=config_chroma, embedding_model=real_embedding_model)
print("\nVectorStoreNode (ChromaDB - Only) created.")

# 5. Prepare a list of simulated cleaned documents
documents_to_index = [
    Document(page_content="This is the first document about AI.", metadata={"url": "https://example.com/doc1"}),
    Document(page_content="Another document discussing machine learning.", metadata={"url": "https://example.com/doc2"}),
    Document(page_content="A third relevant piece of information.", metadata={"url": "https://example.com/doc3"}),
    # Add a duplicate document
    Document(page_content="This is the first document about AI.", metadata={"url": "https://example.com/doc1"}),
    # Document with different content but same source_id_key value as doc3 - Chroma's add_documents won't update by default, but useful for testing state
    Document(page_content="Updated information for the third piece.", metadata={"url": "https://example.com/doc3"}),

]

# Assign the documents to the mock state's cleaned_chunks for Chroma test
mock_state.cleaned_chunks = documents_to_index.copy() # Use a copy


print(f"\nIndexing {len(mock_state.cleaned_chunks)} documents into ChromaDB...")

# 6. Run the async index_documents method for ChromaDB
# Reset mock state calls before each test run
mock_state.reset_mock()
mock_state.add_error = MagicMock()
result_state_update_chroma = asyncio.run(vector_store_node_chroma.index_documents(mock_state))

print("\nIndexing complete for ChromaDB.")

# 7. Check the result state update (Chroma doesn't clear chunks in this implementation)
# The node's return value should indicate clearing the chunks
print("\nResult State Update (ChromaDB):")
print(result_state_update_chroma)
# Check the mock state's cleaned_chunks list directly
print(f"Mock state cleaned_chunks length after Chroma: {len(mock_state.cleaned_chunks)}")


# 8. Check mock state calls for ChromaDB
print("\nMock State Calls (ChromaDB):")
print(f"add_error called: {mock_state.add_error.call_count} times")


# 9. Check the vector store contents (for ChromaDB persistence)
if config_chroma.vector_store_type == "chroma":
    print("\nChecking ChromaDB contents...")
    try:
        client = PersistentClient(path=TEST_CHROMA_DIR)
        # Use the real embedding function for querying Chroma
        collection = client.get_collection(name=config_chroma.collection_name, embedding_function=real_embedding_model)

        # Retrieve some documents to verify
        retrieved_docs = await asyncio.to_thread(collection.get, include=['metadatas', 'documents'])
        print(f"Retrieved {len(retrieved_docs['ids'])} documents from ChromaDB.")
        # You can inspect retrieved_docs to verify content and metadata

    except Exception as e:
        print(f"Error checking ChromaDB: {e}")
        mock_state.add_error(f"Error checking ChromaDB: {e}")


# --- Test with unsupported type ---
print("\n--- Testing Unsupported Vector Store Type ---")
# Create config for an unsupported type
config_unsupported = VectorStoreConfig(
    vector_store_type="sqlite", # This should now raise an error
    collection_name="test_unsupported",
    chroma_persist_directory="./should_not_be_used"
)
print("Vector Store Config (Unsupported):")
print(config_unsupported.model_dump_json(indent=2))

# Reset mock state
mock_state.reset_mock()
mock_state.add_error = MagicMock()
mock_state.cleaned_chunks = [] # No docs needed for this test

print("\nAttempting to create VectorStoreNode with unsupported type...")
try:
    vector_store_node_unsupported = VectorStoreNode(config=config_unsupported, embedding_model=real_embedding_model)
    print("❌ Error: VectorStoreNode creation did NOT raise ValueError for unsupported type.")
except ValueError as e:
    print(f"✅ Success: VectorStoreNode creation correctly raised ValueError: {e}")
except Exception as e:
    print(f"❌ Error: VectorStoreNode creation raised unexpected error: {e}")


# Cleanup after the tests
cleanup_test_db()

# Task
Modify the `CrawlerEngine` class to implement a retry mechanism for failed URL crawls due to timeouts or connection errors. Add a `retry_count` field to the `ExtractedURL` model, configure a maximum number of retries in `CrawlerConfig`, and update the crawling logic to queue failed URLs for retry up to the maximum limit before discarding them. Update the test script to verify the retry mechanism.

## Update extractedurl model

### Subtask:
Add a `retry_count` field to the `ExtractedURL` model to track retry attempts.


**Reasoning**:
I need to add a `retry_count` field to the `ExtractedURL` model in the code cell that defines this class.



In [24]:
class ExtractedURL(BaseModel):
    """URL with extraction metadata."""
    query: str
    url: str = Field(..., description="The URL")
    domain: Optional[str] = Field(default=None, description="Domain extracted from URL")
    priority: float = Field(0.5, ge=0.0, le=1.0, description="Crawling priority")
    should_crawl: bool = Field(True, description="Whether this URL should be crawled")
    content_source: Optional[ContentSource] = Field(default=ContentSource.OTHER, description="Content source or citation")
    estimated_crawl_time: Optional[float] = Field(default=None, ge=1, description="Estimated crawl time in seconds")
    title: Optional[str] = Field(default=None, description="Page title if available")
    snippet: Optional[str] = Field(default=None, description="Content snippet")
    source_engine: Optional[str] = Field(default=None, description="Search engine that found this URL")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    retry_count: int = Field(default=0, description="Number of times crawling this URL has been retried.") # Added retry_count

    @model_validator(mode="after")
    def compute_domain(self) -> "ExtractedURL":
        """Ensure domain is auto-computed from URL if missing."""
        if not self.domain and self.url:
            try:
                # Robustly extract domain
                from urllib.parse import urlparse
                parsed_url = urlparse(self.url)
                if parsed_url.netloc:
                    object.__setattr__(self, "domain", parsed_url.netloc)
            except Exception:
                pass
        return self

## Update crawlerconfig

### Subtask:
Add a `max_retries` field to `CrawlerConfig`.


**Reasoning**:
The subtask is to add a `max_retries` field to the `CrawlerConfig` class. This involves modifying the existing class definition.



In [25]:
from pydantic import BaseModel, Field

class CrawlerConfig(BaseModel):
    """Configuration for the CrawlerEngine."""
    chunk_size: int = Field(default=1000, ge=100, description="Size of text chunks.")
    chunk_overlap: int = Field(default=200, ge=0, description="Overlap between text chunks.")
    max_concurrent: int = Field(default=5, ge=1, description="Maximum concurrent crawl tasks.")
    timeout: int = Field(default=15, ge=5, description="Timeout for each crawl request in seconds.")
    max_content_length: int = Field(default=150000, ge=10000, description="Max length of crawled content to prevent memory issues.")
    enable_embedding_content_scoring: bool = Field(default=True, description="Enable embedding-based scoring of crawled content.") # Changed to embedding scoring
    embedding_model_name: str = Field("sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use for content scoring.") # Added embedding model config
    embedding_relevance_threshold: float = Field(default=0.4, ge=0.0, le=1.0, description="Relevance score threshold for embedding-scored chunks.") # Changed threshold name
    max_retries: int = Field(default=2, ge=0, description="Maximum number of times to retry a failed URL crawl.") # Added max_retries

## Modify ` crawl single url`

### Subtask:
Modify the `_crawl_single_url` method in the `CrawlerEngine` to implement a retry mechanism for failed URL crawls due to timeouts or connection errors.


**Reasoning**:
Modify the `_crawl_single_url` method in the `CrawlerEngine` to implement the retry logic for specific exceptions.



In [26]:
class CrawlerEngine(AsyncCrawlerProtocol):
    """
    Enhanced, state-aware, async crawler for intelligent knowledge acquisition.
    Uses AsyncChromiumLoader for dynamic sites and falls back to WebBaseLoader,
    now including logic to prioritize recently published content and using embeddings
    for content relevance scoring.
    """
    def __init__(self, state: KnowledgeState, config: CrawlerConfig, embedding_model: Optional[Any] = None): # Added embedding_model
        self.state = state
        self.config = config
        self.embedding_model = embedding_model # Store embedding model
        self.semaphore = asyncio.Semaphore(self.config.max_concurrent)
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap
        )
        self.time_threshold = datetime.now(timezone.utc) - timedelta(days=3 * 365) # For 3 years

        if self.config.enable_embedding_content_scoring and not self.embedding_model:
             logger.warning("Embedding content scoring enabled but no embedding model provided. Disabling.")
             self.config.enable_embedding_content_scoring = False


    async def crawl_urls_batch(self, urls_to_crawl: List[ExtractedURL]) -> None:
        """
        Crawl a list of ExtractedURLs, process the content, and update the state.
        This method is the main entry point for crawling within the LangGraph node.
        """
        if not urls_to_crawl:
            return

        crawl_tasks = [self._crawl_single_url(url_info) for url_info in urls_to_crawl]
        results = await asyncio.gather(*crawl_tasks, return_exceptions=True)

        successful_crawls = 0
        failed_crawls = 0
        total_time = 0.0

        # Process results and handle retries
        newly_queued_for_retry = []
        processed_urls_count = 0

        for result in results:
            # Increment processed_urls_count regardless of success/failure
            processed_urls_count += 1

            if isinstance(result, Exception):
                # Error was handled within _crawl_single_url (either retried or discarded)
                failed_crawls += 1 # Still count as a failed attempt for stats
            elif result:
                # Successful crawl result (Document, crawl_time)
                document, crawl_time = result

                # Check publication date before further processing
                publication_date_str = document.metadata.get("publication_date")
                if publication_date_str:
                    try:
                        # Ensure date parsing is robust to different formats
                        pub_date = dateutil.parser.parse(publication_date_str)
                        # Ensure comparison is between timezone-aware datetimes
                        if pub_date.tzinfo is None:
                            # Assume UTC if no timezone info
                            pub_date = pub_date.replace(tzinfo=timezone.utc)

                        if pub_date < self.time_threshold:
                            logger.info(f"Skipping older document from {document.metadata.get('url')}")
                            continue # Skip processing this document
                    except (ValueError, TypeError) as e:
                        logger.warning(f"Could not parse or compare publication date '{publication_date_str}': {e}")

                successful_crawls += 1
                total_time += crawl_time

                # Split and add documents to state
                chunks = await self._split_document(document)

                if self.config.enable_embedding_content_scoring and self.embedding_model:
                    # Use embedding-based scoring
                    scored_chunks = await self._score_content_embedding(chunks, self.state.main_query)
                    relevant_chunks = [
                        c for c in scored_chunks
                        if c.metadata.get("embedding_relevance_score", 0) >= self.config.embedding_relevance_threshold
                    ]
                    if relevant_chunks:
                        self.state.add_results(relevant_chunks, source=DocumentSource.WEB_CRAWL)
                    else:
                         logger.info(f"No relevant chunks found after embedding scoring for {document.metadata.get('url')}")
                else:
                    # Add all chunks if embedding scoring is disabled or not possible
                    self.state.add_results(chunks, source=DocumentSource.WEB_CRAWL)

        # Update crawl stats based on the number of URLs attempted in this batch
        self.state.update_crawl_stats(
            attempted=processed_urls_count, # Use the count of URLs processed in this batch
            successful=successful_crawls,
            failed=failed_crawls,
            time=total_time
        )

        # Add URLs that were marked for retry back to the crawl queue
        # This assumes _crawl_single_url modifies url_info in place or returns it
        # A more robust approach might be to have _crawl_single_url return a status
        # and the url_info if it needs to be re-queued.
        # For now, we rely on _crawl_single_url modifying the list passed to it or state directly.
        # A better way is to collect urls needing retry in _crawl_single_url and return them.
        # Let's refactor _crawl_single_url to return (result, url_info_for_retry) or Exception
        # and update this loop accordingly.

        # REFACTOR NOTE: The current structure makes it hard for _crawl_single_url
        # to directly add back to state.crawl_queue and manage the list passed to it.
        # Let's keep the retry logic within _crawl_single_url for now and see if it works
        # by modifying the url_info object passed by reference (if possible with Pydantic/MagicMock).
        # If not, we'll need to adjust how results are returned and processed here.
        pass # The actual state update happens within _crawl_single_url now for retries.


    async def _crawl_single_url(self, url_info: ExtractedURL) -> Optional[tuple[Document, float]]:
        """Crawl a single URL safely and return the result or None if retried/discarded."""
        url = url_info.url
        start_time = time.perf_counter()

        try:
            async with self.semaphore:
                # Check if the URL has already been retried too many times
                if url_info.retry_count > self.config.max_retries:
                    logger.warning(f"Discarding URL {url} after {url_info.retry_count} retries.")
                    # Do not return anything for this URL, it's discarded
                    return None

                document = await self._load_and_extract(url)

                if not document or not document.page_content:
                    self.state.add_error(f"No content extracted from {url}")
                    return None

                end_time = time.perf_counter()
                crawl_time = end_time - start_time

                if len(document.page_content) > self.config.max_content_length:
                    document.page_content = document.page_content[:self.config.max_content_length] + "..."
                    document.metadata["truncated"] = True

                document.metadata.update({
                    "crawl_timestamp": datetime.now(timezone.utc).isoformat(),
                    "content_length": len(document.page_content),
                    "crawl_time": crawl_time,
                    "url": url,
                    "query": self.state.main_query
                })

                # Add publication and modification dates to metadata
                publication_date = await self._extract_publication_date(document.page_content)
                if publication_date:
                    document.metadata["publication_date"] = publication_date

                modification_date = await self._extract_modification_date(document.page_content)
                if modification_date:
                    document.metadata["modification_date"] = modification_date

                # Reset retry count on successful crawl
                url_info.retry_count = 0

                return (document, crawl_time)

        except (httpx.TimeoutException, httpx.ConnectError) as e:
            # Handle specific network errors for retry
            url_info.retry_count += 1
            if url_info.retry_count <= self.config.max_retries:
                logger.warning(f"Network error for {url} (attempt {url_info.retry_count}/{self.config.max_retries}). Re-queueing.")
                # Add the URL back to the crawl queue for retry
                self.state.crawl_queue.append(url_info)
                # Do not return anything for this URL yet, it's being retried
                return None
            else:
                logger.error(f"Network error for {url} after {url_info.retry_count} retries. Discarding URL. Error: {e}")
                self.state.add_error(f"Failed to crawl {url} after retries: {str(e)}")
                # Do not return anything, it's discarded
                return None

        except Exception as e:
            # Handle other exceptions (parsing errors, etc.) - do not retry for these
            logger.error(f"Failed to crawl {url} due to unexpected error: {e}")
            self.state.add_error(f"Failed to crawl {url}: {str(e)}")
            # For other errors, we don't retry and don't return content
            return None


    async def _load_and_extract(self, url: str) -> Optional[Document]:
        """
        Attempts to load a document using AsyncChromiumLoader, falling back to WebBaseLoader.
        Includes basic error handling for URL loading.
        """
        doc = None
        # Try AsyncChromiumLoader first for dynamic content
        try:
            loader = AsyncChromiumLoader([url])
            docs = await loader.aload()
            if docs and docs[0].page_content and docs[0].page_content.strip():
                docs[0].metadata["loader"] = "chromium"
                doc = docs[0]
                logger.info(f"Successfully loaded {url} with ChromiumLoader.")
            else:
                 logger.warning(f"Chromium loader got empty content for {url}.")
        except Exception as e:
            # Log the specific error but don't fail the crawl task yet
            logger.warning(f"Chromium loader failed for {url}: {e}")

        # Fallback to WebBaseLoader if Chromium fails or is not installed, or returned empty content
        if not doc:
            try:
                def blocking_load():
                     return WebBaseLoader(web_paths=(url,)).load()
                docs = await asyncio.to_thread(blocking_load)
                if docs and docs[0].page_content and docs[0].page_content.strip():
                    docs[0].metadata["loader"] = "webbase"
                    doc = docs[0]
                    logger.info(f"Successfully loaded {url} with WebBaseLoader.")
                else:
                    logger.warning(f"WebBaseLoader got empty content for {url}.")
            except Exception as e:
                logger.warning(f"WebBaseLoader failed for {url}: {e}")
                # If both loaders fail, return None
                return None

        return doc


    async def _extract_publication_date(self, html_content: str) -> Optional[str]:
        """
        Extracts the publication date from HTML content using htmldate.
        Uses asyncio.to_thread as htmldate is not natively async.
        Handles potential errors during date extraction.
        """
        try:
            # Using original_date=True to prioritize explicitly marked dates
            # Added timeout for robustness
            return await asyncio.to_thread(find_date, html_content, original_date=True, timeout=5)
        except Exception as e:
            # Log htmldate specific errors
            logger.warning(f"Failed to extract publication date with htmldate: {e}")
            return None

    async def _extract_modification_date(self, html_content: str) -> Optional[str]:
        """
        Extracts the modification date from HTML content using htmldate.
        Uses asyncio.to_thread as htmldate is not natively async.
        Handles potential errors during date extraction.
        Note: htmldate's ability to find modification dates might be limited or require specific arguments not available in all versions.
        Trying without specific arguments first, or exploring alternative libraries might be necessary.
        """
        try:
            # Calling find_date without the problematic 'lastmod=True'
            # It might still find a date, but less likely to be the modification date specifically
            # Added timeout for robustness
            return await asyncio.to_thread(find_date, html_content, timeout=5)
        except Exception as e:
            logger.warning(f"Failed to extract modification date with htmldate: {e}")
            return None


    async def _split_document(self, doc: Document) -> List[Document]:
        """Split a document into chunks asynchronously."""
        return await asyncio.to_thread(self.splitter.split_documents, [doc])

    async def _score_content_embedding(self, chunks: List[Document], query: str) -> List[Document]:
        """
        Use embeddings to score content chunks against the query based on semantic similarity.
        Returns the chunks with an added 'embedding_relevance_score' in their metadata.
        """
        if not self.embedding_model or not chunks:
            return chunks

        try:
            # Embed the query
            query_embedding = await asyncio.to_thread(self.embedding_model.embed_query, query)

            # Embed the documents/chunks
            # Embedding a list of documents can be done in a single call
            chunk_texts = [chunk.page_content for chunk in chunks]
            chunk_embeddings = await asyncio.to_thread(self.embedding_model.embed_documents, chunk_texts)

            # Calculate cosine similarity between query embedding and each chunk embedding
            # Cosine similarity ranges from -1 (opposite) to 1 (identical).
            # We want scores closer to 1.0.
            scores = []
            for chunk_embedding in chunk_embeddings:
                # Using dot product for cosine similarity if embeddings are normalized
                # If not normalized, need to normalize or use a different similarity metric
                # SentenceTransformer embeddings are usually normalized.
                score = sum(q * c for q, c in zip(query_embedding, chunk_embedding))
                # Ensure score is within [0, 1] range if needed, though cosine similarity is [-1, 1]
                # Map [-1, 1] to [0, 1] for thresholding: (score + 1) / 2
                normalized_score = (score + 1) / 2
                scores.append(normalized_score)


            # Add scores to chunk metadata
            for i, chunk in enumerate(chunks):
                chunk.metadata["embedding_relevance_score"] = scores[i]

            return chunks

        except Exception as e:
            logger.error(f"Embedding scoring failed: {e}")
            # If embedding fails, return original chunks without scores or with a default low score
            for chunk in chunks:
                 chunk.metadata["embedding_relevance_score"] = 0.0 # Assign a low score on failure
            return chunks


    async def _score_content_llm(self, chunks: List[Document], query: str) -> List[Document]:
        """
        Use an LLM to score content chunks against the query.
        Returns the chunks with an added 'llm_relevance_score' in their metadata.
        (Kept for reference if needed, but embedding is preferred as per user feedback)
        """
        if not self.llm or not chunks:
            return chunks

        scoring_prompt = """
        You are a quality assurance assistant for a web crawler. Your task is to score document chunks based on their relevance to a given query, and prioritize more recent content.
        Provide a single relevance score from 0.0 (not relevant) to 1.0 (highly relevant) for each chunk.
        Take into account both the topical relevance and the recency of the content.
        Query: {query}

        Example Output Format:
        [
          {{ "chunk_index": 0, "score": 0.85 }},
          {{ "chunk_index": 1, "score": 0.20 }},
          ...
        ]

        Now, score the following chunks:
        """

        batch_size = 5
        for i in range(0, len(chunks), batch_size):
            chunk_batch = chunks[i:i+batch_size]
            prompt = scoring_prompt.format(query=query)
            for j, chunk in enumerate(chunk_batch):
                # Include date in the prompt for LLM context
                date_info = f"Publication Date: {chunk.metadata.get('publication_date', 'N/A')}\nModification Date: {chunk.metadata.get('modification_date', 'N/A')}\n"
                prompt += f"\n--- Chunk {i+j} ---\n{date_info}{chunk.page_content[:500]}...\n"
            try:
                response = await self.llm.ainvoke(prompt)
                scores = self._parse_llm_response(response.content)

                for score_info in scores:
                    chunk_index = score_info.get("chunk_index")
                    score = score_info.get("score")
                    if chunk_index is not None and score is not None and 0 <= chunk_index < len(chunks):
                         chunks[chunk_index].metadata["llm_relevance_score"] = score
            except Exception as e:
                logger.error(f"LLM scoring failed for batch {i}: {e}")

        return chunks

    def _parse_llm_response(self, text: str) -> List[Dict[str, Union[int, float]]]:
        """Parses the LLM's response to extract scores."""
        try:
            json_match = re.search(r'\[\s*\{.*?\}\s*\]', text, re.DOTALL)
            if json_match:
                # Use json.loads for safer parsing
                import json
                return json.loads(json_match.group(0))
        except Exception as e:
            logger.error(f"Failed to parse LLM response: {e}")
            raise e
        return []

## Update crawl urls batch

### Subtask:
Implement logic to check the `retry_count` when selecting URLs from the queue and ensure URLs are processed based on priority and retry status.


**Reasoning**:
Modify the `crawl_urls_batch` method to properly select URLs from the queue, prioritize URLs for retry, and remove processed URLs from the queue regardless of success or re-queuing.



In [35]:
class CrawlerEngine(AsyncCrawlerProtocol):
    """
    Enhanced, state-aware, async crawler for intelligent knowledge acquisition.
    Uses AsyncChromiumLoader for dynamic sites and falls back to WebBaseLoader,
    now including logic to prioritize recently published content and using embeddings
    for content relevance scoring.
    """
    def __init__(self, state: KnowledgeState, config: CrawlerConfig, embedding_model: Optional[Any] = None): # Added embedding_model
        self.state = state
        self.config = config
        self.embedding_model = embedding_model # Store embedding model
        self.semaphore = asyncio.Semaphore(self.config.max_concurrent)
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap
        )
        self.time_threshold = datetime.now(timezone.utc) - timedelta(days=3 * 365) # For 3 years

        if self.config.enable_embedding_content_scoring and not self.embedding_model:
             logger.warning("Embedding content scoring enabled but no embedding model provided. Disabling.")
             self.config.enable_embedding_content_scoring = False


    async def crawl_urls_batch(self, urls_to_crawl: List[ExtractedURL]) -> None:
        """
        Crawl a list of ExtractedURLs, process the content, and update the state.
        This method is the main entry point for crawling within the LangGraph node.
        """
        if not urls_to_crawl:
            return

        # Sort URLs to prioritize those with fewer retries and higher priority
        # Simple approach: sort by retry_count (ascending) then by priority (descending)
        urls_to_process_in_batch = sorted(urls_to_crawl, key=lambda x: (x.retry_count, -x.priority))
        urls_to_process_in_batch = urls_to_process_in_batch[:self.config.max_concurrent] # Take only max_concurrent

        # Create a set of URLs being processed in this batch for easy removal later
        urls_being_processed_set = {url.url for url in urls_to_process_in_batch}


        crawl_tasks = [self._crawl_single_url(url_info) for url_info in urls_to_process_in_batch]
        results = await asyncio.gather(*crawl_tasks, return_exceptions=True)

        successful_crawls = 0
        failed_crawls = 0
        total_time = 0.0

        # Process results and handle retries
        processed_urls_count = len(urls_to_process_in_batch)

        # Remove the URLs processed in this batch from the state's crawl_queue
        # This needs to be done carefully to avoid issues with modifying a list while iterating
        # A robust way is to rebuild the queue or filter it.
        initial_queue = list(self.state.crawl_queue) # Get a snapshot
        self.state.crawl_queue.clear() # Clear the queue

        # Rebuild the queue with URLs NOT in the processed batch and any that were re-queued by _crawl_single_url
        # URLs re-queued by _crawl_single_url are already added back to state.crawl_queue in that method.
        # So, we just need to add back the URLs from the initial queue that were *not* processed in this batch.
        for url_info in initial_queue:
            if url_info.url not in urls_being_processed_set:
                self.state.crawl_queue.append(url_info)

        # Now process the results from the batch
        for result in results:
            if isinstance(result, Exception):
                # Error was handled within _crawl_single_url (either retried or discarded)
                failed_crawls += 1 # Still count as a failed attempt for stats
            elif result:
                # Successful crawl result (Document, crawl_time)
                document, crawl_time = result

                # Check publication date before further processing
                publication_date_str = document.metadata.get("publication_date")
                if publication_date_str:
                    try:
                        # Ensure date parsing is robust to different formats
                        pub_date = dateutil.parser.parse(publication_date_str)
                        # Ensure comparison is between timezone-aware datetimes
                        if pub_date.tzinfo is None:
                            # Assume UTC if no timezone info
                            pub_date = pub_date.replace(tzinfo=timezone.utc)

                        if pub_date < self.time_threshold:
                            logger.info(f"Skipping older document from {document.metadata.get('url')}")
                            continue # Skip processing this document
                    except (ValueError, TypeError) as e:
                        logger.warning(f"Could not parse or compare publication date '{publication_date_str}': {e}")

                successful_crawls += 1
                total_time += crawl_time

                # Split and add documents to state
                chunks = await self._split_document(document)

                if self.config.enable_embedding_content_scoring and self.embedding_model:
                    # Use embedding-based scoring
                    scored_chunks = await self._score_content_embedding(chunks, self.state.main_query)
                    relevant_chunks = [
                        c for c in scored_chunks
                        if c.metadata.get("embedding_relevance_score", 0) >= self.config.embedding_relevance_threshold
                    ]
                    if relevant_chunks:
                        self.state.add_results(relevant_chunks, source=DocumentSource.WEB_CRAWL)
                    else:
                         logger.info(f"No relevant chunks found after embedding scoring for {document.metadata.get('url')}")
                else:
                    # Add all chunks if embedding scoring is disabled or not possible
                    self.state.add_results(chunks, source=DocumentSource.WEB_CRAWL)

        self.state.update_crawl_stats(
            attempted=processed_urls_count, # Use the count of URLs processed in this batch
            successful=successful_crawls,
            failed=failed_crawls,
            time=total_time
        )

        # URLs that were successfully crawled or discarded are implicitly removed
        # because they are not re-added to the queue by _crawl_single_url.
        # Only URLs needing retry are added back by _crawl_single_url.
        pass # The queue manipulation is done above


    async def _crawl_single_url(self, url_info: ExtractedURL) -> Optional[tuple[Document, float]]:
        """Crawl a single URL safely and return the result or None if retried/discarded."""
        url = url_info.url
        start_time = time.perf_counter()

        try:
            async with self.semaphore:
                # Check if the URL has already been retried too many times
                if url_info.retry_count >= self.config.max_retries: # Use >= for correct check
                    logger.warning(f"Discarding URL {url} after {url_info.retry_count} retries.")
                    # Do not return anything for this URL, it's discarded
                    return None

                document = await self._load_and_extract(url)

                if not document or not document.page_content:
                    self.state.add_error(f"No content extracted from {url}")
                    return None

                end_time = time.perf_counter()
                crawl_time = end_time - start_time

                if len(document.page_content) > self.config.max_content_length:
                    document.page_content = document.page_content[:self.config.max_content_length] + "..."
                    document.metadata["truncated"] = True

                document.metadata.update({
                    "crawl_timestamp": datetime.now(timezone.utc).isoformat(),
                    "content_length": len(document.page_content),
                    "crawl_time": crawl_time,
                    "url": url,
                    "query": self.state.main_query
                })

                # Add publication and modification dates to metadata
                publication_date = await self._extract_publication_date(document.page_content)
                if publication_date:
                    document.metadata["publication_date"] = publication_date

                modification_date = await self._extract_modification_date(document.page_content)
                if modification_date:
                    document.metadata["modification_date"] = modification_date

                # Reset retry count on successful crawl
                url_info.retry_count = 0

                return (document, crawl_time)

        except (httpx.TimeoutException, httpx.ConnectError) as e:
            # Handle specific network errors for retry
            url_info.retry_count += 1
            if url_info.retry_count <= self.config.max_retries:
                logger.warning(f"Network error for {url} (attempt {url_info.retry_count}/{self.config.max_retries}). Re-queueing.")
                # Add the URL back to the crawl queue for retry
                self.state.crawl_queue.append(url_info)
                # Do not return anything for this URL yet, it's being retried
                return None
            else:
                logger.error(f"Network error for {url} after {url_info.retry_count} retries. Discarding URL. Error: {e}")
                self.state.add_error(f"Failed to crawl {url} after retries: {str(e)}")
                # Do not return anything, it's discarded
                return None

        except Exception as e:
            # Handle other exceptions (parsing errors, etc.) - do not retry for these
            logger.error(f"Failed to crawl {url} due to unexpected error: {e}")
            self.state.add_error(f"Failed to crawl {url}: {str(e)}")
            # For other errors, we don't retry and don't return content
            return None


    async def _load_and_extract(self, url: str) -> Optional[Document]:
        """
        Attempts to load a document using AsyncChromiumLoader, falling back to WebBaseLoader.
        Includes basic error handling for URL loading.
        """
        doc = None
        # Try AsyncChromiumLoader first for dynamic content
        try:
            loader = AsyncChromiumLoader([url])
            docs = await loader.aload()
            if docs and docs[0].page_content and docs[0].page_content.strip():
                docs[0].metadata["loader"] = "chromium"
                doc = docs[0]
                logger.info(f"Successfully loaded {url} with ChromiumLoader.")
            else:
                 logger.warning(f"Chromium loader got empty content for {url}.")
        except Exception as e:
            # Log the specific error but don't fail the crawl task yet
            logger.warning(f"Chromium loader failed for {url}: {e}")

        # Fallback to WebBaseLoader if Chromium fails or is not installed, or returned empty content
        if not doc:
            try:
                def blocking_load():
                     return WebBaseLoader(web_paths=(url,)).load()
                docs = await asyncio.to_thread(blocking_load)
                if docs and docs[0].page_content and docs[0].page_content.strip():
                    docs[0].metadata["loader"] = "webbase"
                    doc = docs[0]
                    logger.info(f"Successfully loaded {url} with WebBaseLoader.")
                else:
                    logger.warning(f"WebBaseLoader got empty content for {url}.")
            except Exception as e:
                logger.warning(f"WebBaseLoader failed for {url}: {e}")
                # If both loaders fail, return None
                return None

        return doc


    async def _extract_publication_date(self, html_content: str) -> Optional[str]:
        """
        Extracts the publication date from HTML content using htmldate.
        Uses asyncio.to_thread as htmldate is not natively async.
        Handles potential errors during date extraction.
        """
        try:
            # Using original_date=True to prioritize explicitly marked dates
            # Removed timeout argument as it seems unsupported or causing issues
            return await asyncio.to_thread(find_date, html_content, original_date=True)
        except Exception as e:
            # Log htmldate specific errors
            logger.warning(f"Failed to extract publication date with htmldate: {e}")
            return None

    async def _extract_modification_date(self, html_content: str) -> Optional[str]:
        """
        Extracts the modification date from HTML content using htmldate.
        Uses asyncio.to_thread as htmldate is not natively async.
        Handles potential errors during date extraction.
        Note: htmldate's ability to find modification dates might be limited or require specific arguments not available in all versions.
        Trying without specific arguments first, or exploring alternative libraries might be necessary.
        """
        try:
            # Calling find_date without the problematic 'lastmod=True' or 'timeout'
            # It might still find a date, but less likely to be the modification date specifically
            return await asyncio.to_thread(find_date, html_content)
        except Exception as e:
            logger.warning(f"Failed to extract modification date with htmldate: {e}")
            return None


    async def _split_document(self, doc: Document) -> List[Document]:
        """Split a document into chunks asynchronously."""
        return await asyncio.to_thread(self.splitter.split_documents, [doc])

    async def _score_content_embedding(self, chunks: List[Document], query: str) -> List[Document]:
        """
        Use embeddings to score content chunks against the query based on semantic similarity.
        Returns the chunks with an added 'embedding_relevance_score' in their metadata.
        """
        if not self.embedding_model or not chunks:
            return chunks

        try:
            # Embed the query
            query_embedding = await asyncio.to_thread(self.embedding_model.embed_query, query)

            # Embed the documents/chunks
            # Embedding a list of documents can be done in a single call
            chunk_texts = [chunk.page_content for chunk in chunks]
            # Corrected the call to embed_documents - it should be a method call, not subscripting
            chunk_embeddings = await asyncio.to_thread(self.embedding_model.embed_documents, chunk_texts)

            # Calculate cosine similarity between query embedding and each chunk embedding
            # Cosine similarity ranges from -1 (opposite) to 1 (identical).
            # We want scores closer to 1.0.
            scores = []
            for chunk_embedding in chunk_embeddings:
                # Using dot product for cosine similarity if embeddings are normalized
                # If not normalized, need to normalize or use a different similarity metric
                # SentenceTransformer embeddings are usually normalized.
                score = sum(q * c for q, c in zip(query_embedding, chunk_embedding))
                # Ensure score is within [0, 1] range if needed, though cosine similarity is [-1, 1]
                # Map [-1, 1] to [0, 1] for thresholding: (score + 1) / 2
                normalized_score = (score + 1) / 2
                scores.append(normalized_score)


            # Add scores to chunk metadata
            for i, chunk in enumerate(chunks):
                chunk.metadata["embedding_relevance_score"] = scores[i]

            return chunks

        except Exception as e:
            logger.error(f"Embedding scoring failed: {e}")
            # If embedding fails, return original chunks without scores or with a default low score
            for chunk in chunks:
                 chunk.metadata["embedding_relevance_score"] = 0.0 # Assign a low score on failure
            return chunks


    async def _score_content_llm(self, chunks: List[Document], query: str) -> List[Document]:
        """
        Use an LLM to score content chunks against the query.
        Returns the chunks with an added 'llm_relevance_score' in their metadata.
        (Kept for reference if needed, but embedding is preferred as per user feedback)
        """
        if not self.llm or not chunks:
            return chunks

        scoring_prompt = """
        You are a quality assurance assistant for a web crawler. Your task is to score document chunks based on their relevance to a given query, and prioritize more recent content.
        Provide a single relevance score from 0.0 to 1.0 for each chunk.
        Take into account both the topical relevance and the recency of the content.
        Query: {query}

        Example Output Format:
        [
          {{ "chunk_index": 0, "score": 0.85 }},
          {{ "chunk_index": 1, "score": 0.20 }},
          ...
        ]

        Now, score the following chunks:
        """

        batch_size = 5
        for i in range(0, len(chunks), batch_size):
            chunk_batch = chunks[i:i+batch_size]
            prompt = scoring_prompt.format(query=query)
            for j, chunk in enumerate(chunk_batch):
                # Include date in the prompt for LLM context
                date_info = f"Publication Date: {chunk.metadata.get('publication_date', 'N/A')}\nModification Date: {chunk.metadata.get('modification_date', 'N/A')}\n"
                prompt += f"\n--- Chunk {i+j} ---\n{date_info}{chunk.page_content[:500]}...\n" # Corrected typo here
            try:
                response = await self.llm.ainvoke(prompt)
                scores = self._parse_llm_response(response.content)

                for score_info in scores:
                    chunk_index = score_info.get("chunk_index")
                    score = score_info.get("score")
                    if chunk_index is not None and score is not None and 0 <= chunk_index < len(chunks):
                         chunks[chunk_index].metadata["llm_relevance_score"] = score
            except Exception as e:
                logger.error(f"LLM scoring failed for batch {i}: {e}")

        return chunks

    def _parse_llm_response(self, text: str) -> List[Dict[str, Union[int, float]]]:
        """Parses the LLM's response to extract scores."""
        try:
            json_match = re.search(r'\[\s*\{.*?\}\s*\]', text, re.DOTALL)
            if json_match:
                # Use json.loads for safer parsing
                import json
                return json.loads(json_match.group(0))
        except Exception as e:
            logger.error(f"Failed to parse LLM response: {e}")
            raise e
        return []

**Reasoning**:
Update the test script to simulate URLs needing retries and verify that the `crawl_urls_batch` method correctly selects, processes, and re-queues/discards URLs based on the new retry logic.



In [34]:
import asyncio
import nest_asyncio
from unittest.mock import MagicMock
from langchain_core.documents import Document
import time # Import time for simulating operations
import logging
from datetime import datetime, timedelta, timezone
from typing import Dict, Any, List, Optional, Protocol, Union
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader, AsyncChromiumLoader
import bs4
import re
import httpx
from pydantic import BaseModel, Field
from datetime import datetime, timezone
from htmldate import find_date # New import for date extraction
import dateutil.parser # Needed for flexible date parsing
from bs4 import BeautifulSoup # Import BeautifulSoup for parsing HTML metadata
from langchain_community.embeddings import SentenceTransformerEmbeddings # Import embeddings

nest_asyncio.apply()

# 1. Create a mock KnowledgeState for the CrawlerEngine to update
mock_state = MagicMock(spec=KnowledgeState)
mock_state.add_error = MagicMock()
mock_state.add_results = MagicMock()
mock_state.update_crawl_stats = MagicMock()
mock_state.main_query = "Latest advancements in AI"
mock_state.crawl_queue = [] # Simulate crawl_queue as a list attribute that can be modified

# 2. Create a CrawlerConfig instance, enabling embedding scoring for testing
config = CrawlerConfig(
    chunk_size=500,
    chunk_overlap=100,
    max_concurrent=3, # Test concurrency
    timeout=5, # Set a lower timeout to trigger simulated timeouts
    max_content_length=10000, # Test truncation
    enable_embedding_content_scoring=True, # Enable embedding scoring for testing
    embedding_model_name="sentence-transformers/all-MiniLM-L6-v2", # Specify embedding model
    embedding_relevance_threshold=0.4, # Set a threshold for filtering
    max_retries=2 # Set max retries for testing
)
print("Crawler Config:")
print(config.model_dump_json(indent=2))

# 3. Initialize a real Embedding Model for scoring
try:
    real_embedding_model = SentenceTransformerEmbeddings(model_name=config.embedding_model_name)
    print("\nInitialized real Embedding Model for scoring.")
except Exception as e:
    print(f"\nFailed to initialize real Embedding Model: {e}. Embedding scoring will not run.")
    real_embedding_model = None
    config.enable_embedding_content_scoring = False # Disable if embedding model fails

# 4. Create a CrawlerEngine instance, passing the real Embedding Model
crawler_engine = CrawlerEngine(state=mock_state, config=config, embedding_model=real_embedding_model)
print("\nCrawler Engine created.")

# 5. Prepare a list of simulated ExtractedURLs to crawl, including edge cases
# Simulate URLs that will succeed, fail and be retried, and fail permanently
urls_to_crawl_initial = [
    # Real URLs that should succeed
    ExtractedURL(query=mock_state.main_query, url="https://microsoft.ai/news/two-new-in-house-models/", priority=0.9, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://deepmind.google/models/", priority=0.85, should_crawl=True),
    ExtractedURL(query=mock_state.main_query, url="https://huggingface.co/", priority=0.7, should_crawl=True),

    # Simulate URLs that will timeout and need retries
    # Use httpbin.org/delay/ to simulate delays longer than the timeout
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/6", priority=0.6, should_crawl=True), # Will timeout, needs retry 1
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/7", priority=0.6, should_crawl=True), # Will timeout, needs retry 1

    # Simulate a URL that will fail consistently and be discarded after max_retries
    # Start it with retry_count = max_retries - 1 so it fails one more time and is discarded
    ExtractedURL(query=mock_state.main_query, url="https://httpbin.org/delay/8", priority=0.5, should_crawl=True, retry_count=config.max_retries - 1),

    # Simulate a URL that should NOT be crawled based on should_crawl=False
    ExtractedURL(query=mock_state.main_query, url="https://github.com/", priority=0.2, should_crawl=False),
]

# Add the initial URLs to the mock state's crawl_queue
mock_state.crawl_queue.extend([url for url in urls_to_crawl_initial if url.should_crawl])

print(f"\nInitial Crawl Queue ({len(mock_state.crawl_queue)} URLs):")
for url_info in mock_state.crawl_queue:
     print(f"- URL: {url_info.url}, Priority: {url_info.priority}, Retry Count: {url_info.retry_count}")


print(f"\nStarting crawling process with max_concurrent={config.max_concurrent}, max_retries={config.max_retries}...")

# 6. Run the async crawl_urls_batch method in multiple iterations to simulate retries
num_iterations = 5 # Run multiple iterations to allow retries to happen

for iteration in range(num_iterations):
    print(f"\n--- Running Crawl Iteration {iteration + 1} ---")
    if not mock_state.crawl_queue:
        print("Crawl queue is empty. Stopping.")
        break

    # Pass a copy of the current crawl queue to crawl_urls_batch
    # The method will select its batch internally and modify state.crawl_queue directly
    current_queue_snapshot = list(mock_state.crawl_queue)
    asyncio.run(crawler_engine.crawl_urls_batch(current_queue_snapshot))

    print(f"Crawl Queue after iteration {iteration + 1} ({len(mock_state.crawl_queue)} URLs):")
    for url_info in mock_state.crawl_queue:
         print(f"- URL: {url_info.url}, Priority: {url_info.priority}, Retry Count: {url_info.retry_count}")


print("\nCrawling process finished.")

# 7. Check the final state and mock calls
print("\nMock State Calls (Final):")
print(f"add_error called: {mock_state.add_error.call_count} times")
print(f"add_results called: {mock_state.add_results.call_count} times")
print(f"update_crawl_stats called: {mock_state.update_crawl_stats.call_count} times (should match num_iterations or fewer if queue emptied early)")

# Print final crawl stats summary
if mock_state.update_crawl_stats.call_count > 0:
    print("\nFinal Crawl Stats Summary:")
    # Sum up stats from all update_crawl_stats calls
    total_attempted = sum(call_kwargs.get('attempted', 0) for call_args, call_kwargs in mock_state.update_crawl_stats.call_args_list)
    total_successful = sum(call_kwargs.get('successful', 0) for call_args, call_kwargs in mock_state.update_crawl_stats.call_args_list)
    total_failed = sum(call_kwargs.get('failed', 0) for call_args, call_kwargs in mock_state.update_crawl_stats.call_args_list)
    total_time = sum(call_kwargs.get('time', 0.0) for call_args, call_kwargs in mock_state.update_crawl_stats.call_args_list)

    print(f"Total Attempted: {total_attempted}")
    print(f"Total Successful: {total_successful}")
    print(f"Total Failed (attempts): {total_failed}")
    print(f"Total Time: {total_time:.2f}")


# Verify that URLs that timed out max_retries times are no longer in the queue
discarded_url = "https://httpbin.org/delay/8"
is_discarded_url_in_queue = any(url_info.url == discarded_url for url_info in mock_state.crawl_queue)
print(f"\nIs discarded URL ({discarded_url}) still in queue? {is_discarded_url_in_queue}")

# Verify that URLs that timed out but still have retries are in the queue with updated retry_count
retry_url_1 = "https://httpbin.org/delay/6"
retry_url_2 = "https://httpbin.org/delay/7"
retry_url_1_in_queue = next((url_info for url_info in mock_state.crawl_queue if url_info.url == retry_url_1), None)
retry_url_2_in_queue = next((url_info for url_info in mock_state.crawl_queue if url_info.url == retry_url_2), None)

print(f"\nIs retry URL 1 ({retry_url_1}) in queue? {retry_url_1_in_queue is not None}")
if retry_url_1_in_queue:
    print(f"  Retry URL 1 final retry_count: {retry_url_1_in_queue.retry_count}")

print(f"Is retry URL 2 ({retry_url_2}) in queue? {retry_url_2_in_queue is not None}")
if retry_url_2_in_queue:
    print(f"  Retry URL 2 final retry_count: {retry_url_2_in_queue.retry_count}")

# Verify that successfully crawled URLs are NOT in the queue
successful_urls = ["https://microsoft.ai/news/two-new-in-house-models/", "https://deepmind.google/models/", "https://huggingface.co/"]
for url in successful_urls:
    is_successful_url_in_queue = any(url_info.url == url for url_info in mock_state.crawl_queue)
    print(f"\nIs successful URL ({url}) still in queue? {is_successful_url_in_queue}")

# Remove the line that checks LLM call count as it's no longer relevant
# if config.enable_ml_content_scoring and real_llm:
#     print(f"\nLLM ainvoke called: {real_llm.ainvoke.call_count} times (for content scoring batches)")

Crawler Config:
{
  "chunk_size": 500,
  "chunk_overlap": 100,
  "max_concurrent": 3,
  "timeout": 5,
  "max_content_length": 10000,
  "enable_embedding_content_scoring": true,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_relevance_threshold": 0.4,
  "max_retries": 2
}



Initialized real Embedding Model for scoring.

Crawler Engine created.

Initial Crawl Queue (6 URLs):
- URL: https://microsoft.ai/news/two-new-in-house-models/, Priority: 0.9, Retry Count: 0
- URL: https://deepmind.google/models/, Priority: 0.85, Retry Count: 0
- URL: https://huggingface.co/, Priority: 0.7, Retry Count: 0
- URL: https://httpbin.org/delay/6, Priority: 0.6, Retry Count: 0
- URL: https://httpbin.org/delay/7, Priority: 0.6, Retry Count: 0
- URL: https://httpbin.org/delay/8, Priority: 0.5, Retry Count: 1

Starting crawling process with max_concurrent=3, max_retries=2...

--- Running Crawl Iteration 1 ---


INFO:__main__:Successfully loaded https://microsoft.ai/news/two-new-in-house-models/ with ChromiumLoader.
INFO:__main__:Successfully loaded https://deepmind.google/models/ with ChromiumLoader.
INFO:__main__:Successfully loaded https://huggingface.co/ with ChromiumLoader.


Crawl Queue after iteration 1 (3 URLs):
- URL: https://httpbin.org/delay/6, Priority: 0.6, Retry Count: 0
- URL: https://httpbin.org/delay/7, Priority: 0.6, Retry Count: 0
- URL: https://httpbin.org/delay/8, Priority: 0.5, Retry Count: 1

--- Running Crawl Iteration 2 ---


INFO:__main__:Successfully loaded https://httpbin.org/delay/6 with ChromiumLoader.
INFO:__main__:Successfully loaded https://httpbin.org/delay/7 with ChromiumLoader.
INFO:__main__:Successfully loaded https://httpbin.org/delay/8 with ChromiumLoader.


Crawl Queue after iteration 2 (0 URLs):

--- Running Crawl Iteration 3 ---
Crawl queue is empty. Stopping.

Crawling process finished.

Mock State Calls (Final):
add_error called: 0 times
add_results called: 6 times
update_crawl_stats called: 2 times (should match num_iterations or fewer if queue emptied early)

Final Crawl Stats Summary:
Total Attempted: 6
Total Successful: 6
Total Failed (attempts): 0
Total Time: 65.60

Is discarded URL (https://httpbin.org/delay/8) still in queue? False

Is retry URL 1 (https://httpbin.org/delay/6) in queue? False
Is retry URL 2 (https://httpbin.org/delay/7) in queue? False

Is successful URL (https://microsoft.ai/news/two-new-in-house-models/) still in queue? False

Is successful URL (https://deepmind.google/models/) still in queue? False

Is successful URL (https://huggingface.co/) still in queue? False


# Task
Integrate the enhanced `CrawlerEngine` into the existing workflow, ensuring correct handling of the crawl queue, retries, and state updates within the graph. Update the workflow definition, node implementation, and test script for readability and functionality.

## Review workflow definition

### Subtask:
Examine the existing workflow in `workflow.py` to understand how the `web_crawling_node` is currently integrated.


**Reasoning**:
The subtask is to examine the existing workflow definition in `workflow.py` to understand the integration of the `web_crawling_node`. This requires reading the Python file.



In [36]:
# workflow.py

import asyncio
import logging
from typing import Dict, Any
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

from nodes import (
    kb_lookup_node,
    generate_subqueries_node,
    external_search_node,
    extract_and_rank_urls_node,
    web_crawling_node,
    assess_content_quality_node,
    subquery_search_node,
    process_documents_node,
    finalize_results_node,
    error_handler_node
)
from KnowledgeState import KnowledgeState # Use the Pydantic model
from decisions import (
    should_search_external_node,
    should_crawl_urls_node,
    should_do_subquery_search_node
)
from config import GlobalConfig

logger = logging.getLogger(__name__)

def create_knowledge_graph(tools_and_configs: Dict[str, Any]) -> StateGraph:
    """
    Creates and compiles the LangGraph for knowledge acquisition.
    Uses the Pydantic KnowledgeState model.
    """
    try:
        workflow = StateGraph(KnowledgeState) # Use Pydantic model directly

        config: GlobalConfig = tools_and_configs["config"]
        llm = tools_and_configs['llm'] # Assuming LLM is passed in tools_and_configs
        vectorstore_node = tools_and_configs['vectorstore_node']
        url_analyzer = tools_and_configs['url_analyzer']
        search_engine = tools_and_configs['search_engine']
        crawler_engine = tools_and_configs['crawler_engine']
        cleaner_engine = tools_and_configs['cleaner_engine']


        # Add all nodes, passing required tools/configs
        workflow.add_node("kb_lookup", lambda state: kb_lookup_node(state, vectorstore_node))
        workflow.add_node("generate_subqueries", lambda state: generate_subqueries_node(state, llm))
        workflow.add_node("external_search", lambda state: external_search_node(state, search_engine))
        workflow.add_node("extract_urls", lambda state: extract_and_rank_urls_node(state, url_analyzer))
        workflow.add_node("web_crawling", lambda state: web_crawling_node(state, crawler_engine))
        # Note: assess_content_quality_node and finalize_results_node don't take external tools, just state
        workflow.add_node("assess_quality", assess_content_quality_node)
        workflow.add_node("subquery_search", lambda state: subquery_search_node(state, search_engine))
        workflow.add_node("process_documents", lambda state: process_documents_node(state, cleaner_engine))
        workflow.add_node("finalize", finalize_results_node)
        workflow.add_node("error_handler", error_handler_node)


        # Define the graph flow
        workflow.set_entry_point("kb_lookup")

        # Decision point after KB lookup
        workflow.add_conditional_edges(
            "kb_lookup",
            lambda state: should_search_external_node(state, config),
            {
                "external_search": "generate_subqueries", # Go to generate subqueries after external search is decided
                "finalize": "finalize"
            }
        )

        # Subquery generation and external search
        workflow.add_edge("generate_subqueries", "external_search")
        workflow.add_edge("external_search", "extract_urls")

        # Crawling decision after URL extraction
        workflow.add_conditional_edges(
            "extract_urls",
            lambda state: should_crawl_urls_node(state, config),
            {
                "web_crawling": "web_crawling",
                "assess_quality": "assess_quality"
            }
        )

        workflow.add_edge("web_crawling", "assess_quality")

        # Quality-based decisions
        workflow.add_conditional_edges(
            "assess_quality",
            lambda state: should_do_subquery_search_node(state, config),
            {
                "subquery_search": "subquery_search",
                "process_documents": "process_documents"
            }
        )

        # Subquery search loops back to URL extraction
        workflow.add_edge("subquery_search", "extract_urls")

        # Final processing chain
        workflow.add_edge("process_documents", "finalize")
        workflow.add_edge("finalize", END)

        # Add error transitions (fallback to the error handler)
        # Use a simple error transition from any node that might fail
        # A more complex graph might have specific error handling per node type
        workflow.add_edge("kb_lookup", "error_handler") # Already has conditional, this is fallback
        workflow.add_edge("generate_subqueries", "error_handler")
        workflow.add_edge("external_search", "error_handler")
        workflow.add_edge("extract_urls", "error_handler")
        workflow.add_edge("web_crawling", "error_handler")
        workflow.add_edge("assess_quality", "error_handler")
        workflow.add_edge("subquery_search", "error_handler")
        workflow.add_edge("process_documents", "error_handler")
        workflow.add_edge("finalize", "error_handler") # Already goes to END, this is just a fallback for errors
        workflow.add_edge("error_handler", END) # Error handler always ends

        # Explicitly remove edges to error_handler that are implicitly handled by conditional edges
        # Or ensure the conditional logic correctly handles exceptions and transitions to error
        # For simplicity, the current conditional edges are assumed to return a valid next step or raise.
        # If they raise, the LangGraph runtime handles the error and transitions to nodes with error edges defined.
        # The current definition is a bit redundant but safe.

        return workflow

    except Exception as e:
        logger.error(f"Graph creation failed: {e}")
        raise

ModuleNotFoundError: No module named 'nodes'

**Reasoning**:
The previous attempt to read `workflow.py` failed because the notebook environment does not have the necessary files (`nodes.py`, `KnowledgeState.py`, `decisions.py`, `config.py`) structured as modules. The subtask is to examine the existing workflow definition, which is present in a previous cell output. I need to read the previous cell output to understand the workflow. I will copy the relevant code from the previous output into a new cell and examine it there.



In [59]:
# workflow.py

import asyncio
import logging
from typing import Dict, Any
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# Assume the following are defined in previous cells:
# from nodes import (
#     kb_lookup_node,
#     generate_subqueries_node,
#     external_search_node,
#     extract_and_rank_urls_node,
#     web_crawling_node,
#     assess_content_quality_node,
#     subquery_search_node,
#     process_documents_node,
#     finalize_results_node,
#     error_handler_node
# )
# from KnowledgeState import KnowledgeState, CrawlStatus # Import CrawlStatus
# from decisions import (
#     should_search_external_node,
#     should_crawl_urls_node, # This decision logic might need adjustment
#     should_do_subquery_search_node
# )
# from config import GlobalConfig


logger = logging.getLogger(__name__)

def create_knowledge_graph(tools_and_configs: Dict[str, Any]) -> StateGraph:
    """
    Creates and compiles the LangGraph for knowledge acquisition.
    Uses the Pydantic KnowledgeState model.
    """
    try:
        # Assuming KnowledgeState and CrawlStatus are defined in a previous cell
        workflow = StateGraph(KnowledgeState) # Use Pydantic model directly

        # Assuming config, llm, vectorstore_node, etc., are passed in tools_and_configs
        config: GlobalConfig = tools_and_configs["config"]
        llm = tools_and_configs['llm']
        vectorstore_node = tools_and_configs['vectorstore_node']
        url_analyzer = tools_and_configs['url_analyzer']
        search_engine = tools_and_configs['search_engine']
        crawler_engine = tools_and_configs['crawler_engine']
        cleaner_engine = tools_and_configs['cleaner_engine']


        # Add all nodes, passing required tools/configs
        # Assuming node functions are defined in previous cells
        workflow.add_node("kb_lookup", lambda state: asyncio.run(kb_lookup_node(state, vectorstore_node)))
        workflow.add_node("generate_subqueries", lambda state: asyncio.run(generate_subqueries_node(state, llm)))
        workflow.add_node("external_search", lambda state: asyncio.run(external_search_node(state, search_engine)))
        workflow.add_node("extract_urls", lambda state: asyncio.run(extract_and_rank_urls_node(state, url_analyzer)))
        workflow.add_node("web_crawling", lambda state: asyncio.run(web_crawling_node(state, crawler_engine)))
        # Note: assess_content_quality_node and finalize_results_node don't take external tools, just state
        workflow.add_node("assess_quality", lambda state: asyncio.run(assess_content_quality_node(state)))
        workflow.add_node("subquery_search", lambda state: asyncio.run(subquery_search_node(state, search_engine)))
        workflow.add_node("process_documents", lambda state: asyncio.run(process_documents_node(state, cleaner_engine)))
        workflow.add_node("finalize", lambda state: asyncio.run(finalize_results_node(state)))
        workflow.add_node("error_handler", lambda state: asyncio.run(error_handler_node(state)))


        # Define the graph flow
        workflow.set_entry_point("kb_lookup")

        # Decision point after KB lookup
        workflow.add_conditional_edges(
            "kb_lookup",
            lambda state: should_search_external_node(state, config),
            {
                "external_search": "generate_subqueries", # Go to generate subqueries after external search is decided
                "finalize": "finalize",
                "error_handler": "error_handler" # Add error transition
            }
        )

        # Subquery generation and external search
        workflow.add_edge("generate_subqueries", "external_search")
        workflow.add_edge("external_search", "extract_urls")


        # Crawling decision after URL extraction
        # This decision needs to check if there are URLs in the crawl queue
        def should_crawl_urls(state: KnowledgeState, config: GlobalConfig) -> str:
            """Decision logic for web crawling based on queue and config."""
            # Assuming should_crawl_urls_node logic is available and updated or simplified
            # For now, let's check if there are URLs in the queue
            if state.crawl_queue:
                return "web_crawling"
            else:
                # No URLs to crawl, move to quality assessment
                return "assess_quality"

        workflow.add_conditional_edges(
            "extract_urls",
            lambda state: should_crawl_urls(state, config),
            {
                "web_crawling": "web_crawling",
                "assess_quality": "assess_quality",
                "error_handler": "error_handler" # Add error transition
            }
        )

        # After web crawling, decide based on crawl status and remaining queue
        def continue_crawling_decision(state: KnowledgeState) -> str:
            """Decision after web crawling batch."""
            # If there are still URLs in the queue (including retries), loop back to crawling
            if state.crawl_queue:
                logger.info(f"Crawl queue not empty ({len(state.crawl_queue)} URLs). Continuing crawling.")
                return "web_crawling"
            else:
                # Queue is empty, all URLs processed or discarded. Move to quality assessment.
                logger.info("Crawl queue empty. Moving to assess quality.")
                return "assess_quality"

        workflow.add_conditional_edges(
             "web_crawling",
             continue_crawling_decision,
             {
                 "web_crawling": "web_crawling", # Loop back for next batch/retries
                 "assess_quality": "assess_quality",
                 "error_handler": "error_handler" # Add error transition
             }
        )


        # Quality-based decisions
        # Assuming should_do_subquery_search_node logic is available and used
        workflow.add_conditional_edges(
            "assess_quality",
            lambda state: should_do_subquery_search_node(state, config),
            {
                "subquery_search": "subquery_search",
                "process_documents": "process_documents",
                "error_handler": "error_handler" # Add error transition
            }
        )

        # Subquery search loops back to URL extraction (to process new search results)
        workflow.add_edge("subquery_search", "extract_urls")

        # Final processing chain
        workflow.add_edge("process_documents", "finalize")

        # Finalize node can transition to END or error_handler
        workflow.add_conditional_edges(
            "finalize",
            lambda state: END if not state.errors else "error_handler",
            {
                END: END,
                "error_handler": "error_handler"
            }
        )


        # Add error transitions (fallback to the error handler)
        # These are fallback edges if a node raises an unhandled exception
        workflow.add_edge("generate_subqueries", "error_handler")
        workflow.add_edge("external_search", "error_handler")
        workflow.add_edge("subquery_search", "error_handler") # Already has conditional, this is fallback
        workflow.add_edge("process_documents", "error_handler")


        # Error handler always ends
        workflow.add_edge("error_handler", END)

        return workflow

    except Exception as e:
        logger.error(f"Graph creation failed: {e}")
        raise

## Update `web crawling node`

### Subtask:
Modify the `web_crawling_node` in `nodes.py` to correctly utilize the enhanced `CrawlerEngine`, specifically how it handles the `crawl_queue` (selecting a batch, processing results, and re-queuing failed URLs).


**Reasoning**:
Modify the `web_crawling_node` to use the enhanced `CrawlerEngine` by removing the manual batch selection and calling `crawl_urls_batch` with the full `state.crawl_queue`. Update the return value to reflect the state changes made by the `CrawlerEngine`.



In [58]:
# nodes.py

import logging
import asyncio
from typing import Dict, Any, List
import time
import hashlib
from langchain_core.documents import Document
from pydantic import ValidationError

# Assume KnowledgeState, CrawlStatus, SubQuery, convert_langchain_doc_to_document_result, DocumentSource, ContentSource are defined
# Assume SearchEngine, CrawlerEngine, CleanerEngine, VectorStoreNode are defined
# Assume GlobalConfig is defined
from datetime import datetime, timezone

logger = logging.getLogger(__name__)

# --- Helper function for LLM calls with retry logic ---

async def _call_llm(llm: Any, prompt: str) -> str:
    """Helper to call the LLM and extract the response with retry logic."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Assuming an async-capable LLM client
            response = await llm.ainvoke(prompt)
            if hasattr(response, 'content'):
                return response.content
            return str(response)
        except Exception as e:
            logger.warning(f"LLM call attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                raise Exception(f"All LLM call attempts failed: {e}")
            await asyncio.sleep(1)

# ---- Node functions for LangGraph ----

async def kb_lookup_node(state: KnowledgeState, vectorstore_node: VectorStoreNode) -> KnowledgeState:
    """LangGraph node for KB lookup."""
    try:
        results_with_scores = await asyncio.to_thread(
            vectorstore_node._vectorstore.similarity_search_with_score,
            query=state.main_query,
            k=6
        )

        state.knowledge_base_results.clear()

        for doc, score in results_with_scores:
            doc_result = convert_langchain_doc_to_document_result(
                doc,
                source_type=DocumentSource.KNOWLEDGE_BASE,
                relevance_score=1.0 - score
            )
            state.knowledge_base_results.append(doc_result)

        if results_with_scores:
            avg_score = sum(score for _, score in results_with_scores) / len(results_with_scores)
            confidence = max(0.0, min(1.0, 1.0 - (avg_score / 2.0)))
            state.state_metrics["kb_confidence"] = confidence
        else:
            state.state_metrics["kb_confidence"] = 0.0

        logger.info(f"KB lookup complete. Found {len(results_with_scores)} docs, confidence: {state.state_metrics.get('kb_confidence', 0.0):.2f}")

    except Exception as e:
        logger.error(f"KB lookup failed: {e}")
        state.add_error(f"KB lookup failed: {str(e)}")
        state.state_metrics["kb_confidence"] = 0.0

    return state


async def generate_subqueries_node(state: KnowledgeState, llm: Any) -> KnowledgeState:
    """LangGraph node to generate subqueries."""
    try:
        prompt = f"""
        Given the main query: "{state.main_query}"
        Generate 3-4 focused sub-queries that would help find comprehensive knowledge.
        Consider different aspects like:
        - Technical details and implementation
        - Recent developments and trends
        - Practical applications and examples
        - Comparative analysis and alternatives
        Format your response as a list of queries, one per line:
        - [subquery 1]
        - [subquery 2]
        - [subquery 3]
        - [subquery 4]
        """
        response_content = await _call_llm(llm, prompt)

        lines = [line.strip() for line in response_content.split('\n') if line.strip().startswith('-')]
        subqueries_texts = [line[1:].strip() for line in lines if len(line) > 1]

        subqueries = []
        for i, sq in enumerate(subqueries_texts):
            if sq:
                priority = 0.9 if i == 0 else (0.8 if i == 1 else 0.7)
                deep = i < 2
                query_type = ("technical" if "technical" in sq.lower() or "implementation" in sq.lower() else
                              "trends" if "recent" in sq.lower() or "developments" in sq.lower() else "general")
                subqueries.append(SubQuery(query=sq.strip(), priority=priority, deep=deep, query_type=query_type))

        if subqueries:
            state.add_queries(subqueries)
            return state

        logger.info("Generated no new subqueries.")

    except Exception as e:
        logger.error(f"Failed to generate subqueries: {e}")
        state.add_error(f"Subquery generation failed: {str(e)}")

    return state


async def external_search_node(state: KnowledgeState, search_engine: SearchEngine) -> KnowledgeState:
    """LangGraph node for external search."""
    try:
        await search_engine.run_searches(queries=[state.main_query], max_results_per_query=8)
        logger.info("External search tool invoked successfully.")
    except Exception as e:
        logger.error(f"External search failed: {e}")
        state.add_error(f"External search failed: {str(e)}")
    return state


async def extract_and_rank_urls_node(state: KnowledgeState, url_analyzer: URLAnalyzer) -> KnowledgeState:
    """LangGraph node to extract and rank URLs."""
    try:
        search_results_dicts = [
            {'url': r.metadata.get('url', ''), 'snippet': r.page_content}
            for r in state.external_api_results
        ]

        await url_analyzer.analyze_batch(search_results=search_results_dicts, query=state.main_query)
        logger.info("URL analyzer tool invoked successfully.")

    except Exception as e:
        logger.error(f"URL extraction failed: {e}")
        state.add_error(f"URL extraction failed: {str(e)}")

    return state


async def web_crawling_node(state: KnowledgeState, crawler_engine: CrawlerEngine) -> KnowledgeState:
    """LangGraph node for web crawling."""
    try:
        if not state.crawl_queue:
            logger.info("No URLs in crawl queue. Skipping crawling.")
            state.crawl_status = CrawlStatus.NEUTRAL
            return state

        state.crawl_status = CrawlStatus.IN_PROGRESS

        # The crawler_engine.crawl_urls_batch method now handles batching and queue management internally.
        # Pass the entire current crawl_queue to it.
        # The method will select max_concurrent URLs, crawl them, and update state.crawl_queue
        # by removing processed URLs and re-adding those needing retries.
        await crawler_engine.crawl_urls_batch(state.crawl_queue)

        # After the batch is processed by the engine, check the state of the queue
        if not state.crawl_queue:
            # If the queue is empty after processing, all relevant URLs were either successful or discarded
            state.crawl_status = CrawlStatus.SUCCESS
        else:
            # If there are still URLs in the queue, it means some need retries
            state.crawl_status = CrawlStatus.PARTIAL # Indicate partial success/ongoing process

        logger.info(f"Crawling complete for batch. Status: {state.crawl_status}")

    except Exception as e:
        logger.error(f"Web crawling failed: {e}")
        state.add_error(f"Web crawling failed: {str(e)}")
        state.crawl_status = CrawlStatus.FAILED

    return state


async def assess_content_quality_node(state: KnowledgeState) -> KnowledgeState:
    """LangGraph node for assessing content quality."""
    try:
        quality_score = state.calculate_weighted_quality_score()
        state.state_metrics["quality_score"] = quality_score

        gaps = []
        if len(state.knowledge_base_results) + len(state.crawler_results) < 2:
            gaps.append("Insufficient knowledge base or crawled results")
        if state.state_metrics.get("kb_confidence", 0.0) < 0.5:
            gaps.append("Low knowledge base confidence")
        if quality_score < 0.3:
            gaps.append("Overall low content quality")

        state.gaps = gaps

        logger.info(f"Content quality assessment: score={quality_score:.2f}, gaps={len(gaps)}")

    except Exception as e:
        logger.error(f"Content quality assessment failed: {e}")
        state.add_error(f"Content quality assessment failed: {str(e)}")

    return state


async def subquery_search_node(state: KnowledgeState, search_engine: SearchEngine) -> KnowledgeState:
    """LangGraph node for targeted search using subqueries."""
    try:
        # Select a batch of subqueries to process
        subqueries_to_search = [sq.query for sq in state.sub_queries_to_generate][:3]

        if not subqueries_to_search:
            logger.info("No subqueries available for search. Skipping.")
            return state

        await search_engine.run_searches(queries=subqueries_to_search, max_results_per_query=5)

        # Move processed subqueries from `to_generate` to `processed`
        processed_subqueries = [sq.query for sq in state.sub_queries_to_generate[:3]]
        state.processed_queries.extend(processed_subqueries)
        state.sub_queries_to_generate = state.sub_queries_to_generate[3:]

        logger.info(f"Subquery search invoked for: {processed_subqueries}")

    except Exception as e:
        logger.error(f"Subquery search failed: {e}")
        state.add_error(f"Subquery search failed: {str(e)}")

    return state


async def process_documents_node(state: KnowledgeState, cleaner_engine: CleanerEngine) -> KnowledgeState:
    """LangGraph node for cleaning and chunking documents."""
    try:
        all_docs = []
        all_docs.extend(state.knowledge_base_results)
        all_docs.extend(state.crawler_results)
        all_docs.extend(state.external_api_results)

        if not all_docs:
            logger.info("No documents to process. Skipping.")
            return state

        # Call the refactored cleaner tool
        await cleaner_engine.clean_and_chunk_documents(all_docs)

        logger.info(f"Document processing complete. Chunks ready.")

    except Exception as e:
        logger.error(f"Document processing failed: {e}")
        state.add_error(f"Document processing failed: {str(e)}")

    return state


async def finalize_results_node(state: KnowledgeState) -> KnowledgeState:
    """LangGraph node for finalizing and sorting results."""
    try:
        all_results = state.knowledge_base_results + state.cleaned_chunks

        if not all_results:
            logger.info("No documents to finalize.")
            return state

        seen_hashes = set()
        final_docs = []
        for doc_result in all_results:
            if doc_result.page_content:
                content_hash = hashlib.sha256(doc_result.page_content.encode()).hexdigest()
                if content_hash not in seen_hashes:
                    seen_hashes.add(content_hash)
                    final_docs.append(doc_result)

        def sort_key(doc_result):
            source_priority = {
                DocumentSource.KNOWLEDGE_BASE: 4,
                DocumentSource.EXTERNAL_API: 3,
                DocumentSource.WEB_CRAWL: 2,
                DocumentSource.SEARCH_SNIPPET: 1
            }.get(doc_result.source_type, 1)

            content_boost = {
                ContentSource.ACADEMIC: 1.3,
                ContentSource.GOVERNMENT: 1.2,
                ContentSource.WIKI: 1.1,
                ContentSource.OTHER: 1.0
            }.get(doc.content_source, 1.0) if doc.content_source else 1.0

            relevance = doc_result.relevance_score or 0.5
            length_factor = min(1.0, (doc_result.content_length or 0) / 1000)

            # Combine factors - adjust weights as needed
            # Example: Prioritize KB, then recent/relevant crawled content, then search snippets
            # Incorporate source priority, content boost, and relevance
            # A simple weighted sum, or a more complex ranking function could be used
            # For now, let's use a combination of source priority and relevance,
            # and potentially content boost.
            # Higher score is better.
            combined_score = (source_priority * 0.4) + (relevance * 0.4) + (content_boost * 0.2)
            return combined_score

        final_docs.sort(key=sort_key, reverse=True)

        max_final_docs = state.state_metrics.get('max_final_documents', 20)
        state.cleaned_chunks = final_docs[:max_final_docs]
        state.metadata.total_iterations += 1

        final_quality = state.calculate_weighted_quality_score()
        state.state_metrics["final_quality_score"] = final_quality

        state.metadata.final = True

        logger.info(f"Finalization complete. {len(state.cleaned_chunks)} final documents.")

    except Exception as e:
        logger.error(f"Finalization failed: {e}")
        state.add_error(f"Finalization failed: {str(e)}")

    return state


async def error_handler_node(state: KnowledgeState) -> KnowledgeState:
    """
    Handles errors recorded in the state.
    """
    if state.errors:
        logger.error(f"Graph encountered errors: {state.errors}")
    return state

**Reasoning**:
The previous code block failed because `VectorStoreNode` was not defined. This is because the definitions for the tools were commented out in the code cell. I need to uncomment the imports for the tools, including `VectorStoreNode`, `SearchEngine`, `CrawlerEngine`, and `CleanerEngine`.



In [61]:








class CrawlerConfig(BaseModel):
    """Configuration for the CrawlerEngine."""
    chunk_size: int = Field(default=1000, ge=100, description="Size of text chunks.")
    chunk_overlap: int = Field(default=200, ge=0, description="Overlap between text chunks.")
    max_concurrent: int = Field(default=5, ge=1, description="Maximum concurrent crawl tasks.")
    timeout: int = Field(default=15, ge=5, description="Timeout for each crawl request in seconds.")
    max_content_length: int = Field(default=150000, ge=10000, description="Max length of crawled content to prevent memory issues.")
    enable_embedding_content_scoring: bool = Field(default=True, description="Enable embedding-based scoring of crawled content.") # Changed from enable_ml_content_scoring
    embedding_model_name: str = Field("sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use for content scoring.") # Added embedding model config
    embedding_relevance_threshold: float = Field(default=0.4, ge=0.0, le=1.0, description="Relevance score threshold for embedding-scored chunks.") # Changed threshold name
    max_retries: int = Field(default=2, ge=0, description="Maximum number of times to retry a failed URL crawl.") # Added max_retries


class CleanerConfig(BaseModel):
    """Configuration for the CleanerEngine."""
    chunk_size: int = Field(default=1200, ge=200, description="Size of text chunks.")
    chunk_overlap: int = Field(default=160, ge=0, description="Overlap between text chunks.")
    use_semantic_chunking: bool = Field(default=False, description="Use embedding-based semantic chunking.")
    embedding_model_name: str = Field("sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use.")
    model_config = ConfigDict(extra='forbid') # Ensure no extra fields are allowed


class VectorStoreConfig(BaseModel):
    """Configuration for the VectorStoreNode."""
    vector_store_type: str = Field(
        default="chroma",
        description="Type of vector store to use: 'chroma', 'postgres', or 'sqlite'."
    )
    collection_name: str = Field(default="default-collection", description="The name of the vector collection/table.")
    chroma_persist_directory: str = Field(default="./chroma_db", description="Directory for Chroma persistence.")
    # Add embedding model config here, or keep it separate if preferred
    embedding_model_name: str = Field(default="sentence-transformers/all-MiniLM-L6-v2", description="Name of the embedding model to use.")
    embedding_model_provider: str = Field(default="huggingface", description="Provider for the embedding model: 'huggingface' or 'sentence-transformer'.")
    # Add embedding model itself as a field, allowing Any or specific protocol
    embedding_model: Optional[Any] = Field(default=None, description="Initialized Embedding Model instance.") # Added field for embedding model

    model_config = ConfigDict(extra='forbid', arbitrary_types_allowed=True)


class LLMConfig(BaseModel):
    """Configuration for Language Models."""
    chat_model_name: str = Field(default="gpt-4o", description="Name of the main chat model.")
    tool_model_name: str = Field(default="gpt-4o", description="Name of the tool-calling model.")
    embedding_model_name: str = Field(default="text-embedding-ada-002", description="Name of the embedding model.") # This one might be redundant if SentenceTransformer is used directly
    temperature: float = Field(default=0.0, ge=0.0, le=2.0, description="Temperature for creative tasks.")
    # Add other LLM-specific configs (e.g., API keys, max tokens)

class DecisionsConfig(BaseModel):
    """Configuration for Graph Decision Logic."""
    confidence_threshold: float = Field(default=0.7, ge=0.0, le=1.0, description="KB confidence threshold to skip external search.")
    low_success_rate_threshold: float = Field(default=0.3, ge=0.0, le=1.0, description="Crawl success rate below which crawling is stopped.")
    quality_threshold: float = Field(default=0.6, ge=0.0, le=1.0, description="Content quality score threshold to stop searching.")
    max_iterations: int = Field(default=5, ge=1, description="Maximum number of graph iterations.")
    min_results: int = Field(default=2, ge=0, description="Minimum number of results needed for quality assessment.")
    max_final_documents: int = Field(default=20, ge=1, description="Maximum number of documents in the final result set.")


class GlobalConfig(BaseModel):
    """
    Comprehensive configuration for the Knowledge Acquisition workflow.
    """
    search: "SearchConfig" = Field(default_factory=lambda: SearchConfig())
    url_analyzer: "URLAnalysisConfig" = Field(default_factory=lambda: URLAnalysisConfig())
    crawler: "CrawlerConfig" = Field(default_factory=lambda: CrawlerConfig())
    vector_store: "VectorStoreConfig" = Field(default_factory=lambda: VectorStoreConfig())
    llm: LLMConfig = Field(default_factory=LLMConfig)
    decisions: "DecisionsConfig" = Field(default_factory=lambda: DecisionsConfig())

    model_config = ConfigDict(extra="forbid")

# --- Global LLM Registry ---
# Assuming init_chat_model is available
from langchain.chat_models import init_chat_model
import uuid # Import uuid

class LLMRegistry:
    _models: Dict[str, Any] = {}

    @classmethod
    def init_from_config(cls, config: LLMConfig):
        if "chat" not in cls._models:
            cls._models["chat"] = init_chat_model(
                config.chat_model_name,
                temperature=config.temperature,
            )
        if "tool" not in cls._models:
            cls._models["tool"] = init_chat_model(
                config.tool_model_name,
                temperature=config.temperature,
            )
        # Embedding model from LLMConfig might be different from the one in Crawler/VectorStoreConfig
        # Depending on your architecture, you might initialize it here or keep it separate
        # For now, we'll rely on the embedding model passed directly to the tools
        return cls._models

    @classmethod
    def get(cls, role: str = "chat"):
        if role not in cls._models:
            raise ValueError(f"LLM '{role}' not initialized. Call init_from_config first.")
        return cls._models[role]

### Run the Integrated Knowledge Acquisition Workflow

Now that the tool definitions, state/config models, and workflow graph are defined and updated, we can run the `main` function to execute the integrated knowledge acquisition workflow. This will simulate the end-to-end process, including searching, URL extraction, crawling with retries and embedding scoring, cleaning, and finalization.